# CODE_V4 — Force-GB (prep+hyperparameter individual), Forecast 3 & 4 Bulan, Rolling vs Langsung

Notebook ini mereplikasi persis Tahap 1 (preprocessing) → Tahap 2 (24 kombinasi) → Tahap 2b (drill-down 12 kombinasi)
dari `code_v3.ipynb`, dijalankan untuk **2 horizon forecast (4 bulan & 3 bulan)**, lalu menambahkan 2 analisis baru:

1. **Force-GB**: untuk unit/kriteria yang combo terbaiknya di Tahap 2b BUKAN Gradient Boosting, dicari ulang
   kombinasi (data-prep × hyperparameter GB) yang menghasilkan MAPE terendah KHUSUS untuk GB (fit ulang penuh,
   bukan ambil dari data lama). Hasilnya dibandingkan ke MAPE terbaik v2b (selisih dihitung, per unit & kriteria).
2. **Rolling vs Langsung**: untuk kombinasi GB final tiap unit/kriteria, dibandingkan forecast dengan cara
   *langsung* (lag pakai nilai aktual — cara yang sudah dipakai dari awal) vs *rolling* (lag bulan ke-2/3/4
   pakai hasil prediksi sendiri, digeser maju).

**Catatan / asumsi yang dipakai** (tolong dikoreksi kalau salah sebelum dijalankan):
- Horizon 4 bulan: TRAIN Jan 2023-Agu 2025, TEST Sep-Des 2025. Horizon 3 bulan: TRAIN Jan 2023-Sep 2025, TEST Okt-Des 2025.
- Grid search Tahap 2 & drill-down Tahap 2b diulang PENUH dari nol untuk horizon 3 bulan (bukan reuse parameter horizon 4 bulan).
- Force-GB hanya diterapkan ke unit/kriteria yang combo terbaiknya bukan GB; yang sudah menang GB di Tahap 2b dibawa apa adanya (tidak di-refit ulang).
- Rolling-vs-Langsung hanya untuk model Gradient Boosting (ARIMAX sudah native multi-step, LSTM di pipeline lama sudah rolling).
- Sel print sanity-check murni (dtypes/nunique, shape antar-langkah) dari `code_v3.ipynb` tidak dibawa; plot/EDA (cek outlier volume container) dan semua sel yang menghasilkan file (completeness, kriteria-unit-HM mapping) tetap dibawa.\n- `code_v3.ipynb` sendiri TIDAK punya cell skewness/outlier-IQR khusus UTILITAS (itu ada di versi lebih lama, sudah digantikan pendekatan 4 varian data-prep) — jadi tidak direplikasi di sini. Kasih tahu kalau tetap mau ditambahkan.


## 1. Setup Path & File Sumber (sama seperti v3)

In [1]:
import pandas as pd
import numpy as np
import warnings
import json
import re
import os
import gc
import itertools
from pathlib import Path
from difflib import get_close_matches

warnings.filterwarnings('ignore')

UPLOAD_DIR = Path('.')
OUT_DIR = Path('./checkpoints_v4')          # folder dasar v4 (dipakai utk preprocessing bersama)
MAP_DIR = Path('./anonymization_maps')      # sama seperti v2/v3 -- deterministik
OUT_DIR.mkdir(exist_ok=True)
MAP_DIR.mkdir(exist_ok=True)

BBM_FILES = {
    2023: UPLOAD_DIR / 'BBM AAB 2023.xlsx',
    2024: UPLOAD_DIR / 'BBM AAB 2024.xlsx',
    2025: UPLOAD_DIR / 'BBM AAB 2025.xlsx',
}
CONTAINER_FILES = {
    2023: UPLOAD_DIR / 'Container 2023.xlsx',
    2024: UPLOAD_DIR / 'Container 2024.xlsx',
    2025: UPLOAD_DIR / 'Container 2025.xlsx',
}
COST_FILE = UPLOAD_DIR / 'cost & bbm 2022 sd 2025 HP & Type.xlsx'
RAINFALL_FILE = UPLOAD_DIR / 'Curah Hujan 2023-2025.xlsx'


## 2. Preprocessing (identik dengan `code_v3.ipynb`, sel diagnostik dibuang)

In [2]:
def scan_bbm_unit_names(bbm_files):
    units = set()
    for yr, path in bbm_files.items():
        xls = pd.ExcelFile(path)
        for sh in xls.sheet_names:
            df = pd.read_excel(path, sheet_name=sh, header=[0, 1, 2], nrows=1)
            if df.empty:
                continue
            level0 = df.columns.get_level_values(0)
            for l0 in set(level0):
                if l0 == 'EQUIP NAME':
                    continue
                subcols = set(df.columns[level0 == l0].get_level_values(2))
                if 'HM' in subcols and 'LITER' in subcols:
                    units.add(l0)
    return units

def scan_container_branch_codes(container_files):
    branches = set()
    for yr, path in container_files.items():
        df = pd.read_excel(path, sheet_name=0, header=None)
        col0 = df.iloc[:, 0]
        starts = col0[col0 == 'CURRSTATE'].index.tolist()
        for i, s in enumerate(starts):
            end = starts[i + 1] if i + 1 < len(starts) else len(df)
            data_start = s + 9
            block_col0 = df.iloc[data_start:end, 0]
            gt_idx = block_col0[block_col0 == 'Grand Total'].index
            data_end = gt_idx[0] if len(gt_idx) > 0 else end
            branches.update(df.iloc[data_start:data_end, 0].dropna().unique().tolist())
    return branches

def scan_cost_file_names_and_lokasi(cost_file):
    df = pd.read_excel(cost_file, sheet_name='Sheet2', header=1)
    names = set(df['NAMA ALAT BERAT'].dropna().unique().tolist())
    lokasi = set(df['DES 2025'].dropna().unique().tolist())
    return names, lokasi

print('Scanning BBM AAB unit names...')
bbm_units = scan_bbm_unit_names(BBM_FILES)
print('Scanning Container branch codes...')
branch_codes = scan_container_branch_codes(CONTAINER_FILES)
print('Scanning cost&bbm file...')
cost_names, cost_lokasi = scan_cost_file_names_and_lokasi(COST_FILE)

all_unit_names = sorted(bbm_units | cost_names)
unit_map = {name: f'UNIT_{i+1:04d}' for i, name in enumerate(all_unit_names)}

branch_map = {name: f'BRANCH_{i+1:03d}' for i, name in enumerate(sorted(branch_codes))}
lokasi_map = {name: f'LOC_{i+1:03d}' for i, name in enumerate(sorted(cost_lokasi))}

print(f'Total unit: {len(unit_map)} | branch cabang: {len(branch_map)} | lokasi: {len(lokasi_map)}')

with open(MAP_DIR / 'unit_map.json', 'w') as f:
    json.dump(unit_map, f, indent=2, ensure_ascii=False)
with open(MAP_DIR / 'branch_map.json', 'w') as f:
    json.dump(branch_map, f, indent=2, ensure_ascii=False)
with open(MAP_DIR / 'lokasi_map.json', 'w') as f:
    json.dump(lokasi_map, f, indent=2, ensure_ascii=False)


Scanning BBM AAB unit names...
Scanning Container branch codes...
Scanning cost&bbm file...
Total unit: 355 | branch cabang: 49 | lokasi: 35


In [3]:
def process_bbm_sheet(path, sheet, unit_map):
    df = pd.read_excel(path, sheet_name=sheet, header=[0, 1, 2])
    if df.empty or df.shape[1] == 0:
        return None

    cols = df.columns
    level0 = cols.get_level_values(0)

    tgl_col = next((c for c in cols if c[2] == 'TANGGAL'), None)
    if tgl_col is None:
        return None

    unit_cols = []
    for l0 in set(level0):
        if l0 == 'EQUIP NAME':
            continue
        subcols = set(cols[level0 == l0].get_level_values(2))
        if 'HM' in subcols and 'LITER' in subcols:
            for c in cols[level0 == l0]:
                if c[2] in ('HM', 'LITER'):
                    unit_cols.append(c)

    if not unit_cols:
        return None

    sub = df[[tgl_col] + unit_cols].copy()
    sub.columns = pd.MultiIndex.from_tuples([('__TANGGAL__', '', '')] + list(sub.columns[1:]))

    tgl_raw = sub[('__TANGGAL__', '', '')].astype(str)
    sub = sub[~tgl_raw.isin(['AVG', 'TOTAL LTR/UTL', 'nan'])]

    sub = sub.set_index(('__TANGGAL__', '', ''))
    sub.index.name = 'TANGGAL'
    sub.columns = pd.MultiIndex.from_tuples([(c[0], c[2]) for c in sub.columns])

    long_df = sub.stack(level=0, future_stack=True).reset_index()
    long_df = long_df.rename(columns={'level_1': 'EQUIP NAME'})
    long_df['EQUIP NAME'] = long_df['EQUIP NAME'].map(unit_map)
    return long_df


frames = []
for yr, path in BBM_FILES.items():
    xls = pd.ExcelFile(path)
    for sh in xls.sheet_names:
        r = process_bbm_sheet(path, sh, unit_map)
        if r is not None:
            frames.append(r)

raw_bbm = pd.concat(frames, ignore_index=True)
print('raw_bbm shape:', raw_bbm.shape)


raw_bbm shape: (333643, 4)


In [4]:
raw_bbm['TANGGAL'] = pd.to_datetime(raw_bbm['TANGGAL'], format='%d-%m-%Y', errors='coerce')
raw_bbm = raw_bbm.dropna(subset=['TANGGAL'])

raw_bbm['HM_Clean'] = pd.to_numeric(raw_bbm['HM'], errors='coerce')
raw_bbm['HM_Clean'] = raw_bbm['HM_Clean'].replace(0, np.nan)
raw_bbm['HM_Clean'] = raw_bbm.groupby('EQUIP NAME')['HM_Clean'].ffill()
raw_bbm['HM_Clean'] = raw_bbm['HM_Clean'].fillna(0)

raw_bbm['DELTA_HM'] = raw_bbm.groupby('EQUIP NAME')['HM_Clean'].diff().fillna(0)
raw_bbm.loc[raw_bbm['DELTA_HM'] < 0, 'DELTA_HM'] = 0
raw_bbm.loc[raw_bbm['DELTA_HM'] > 24, 'DELTA_HM'] = 24

raw_bbm.to_csv(OUT_DIR / '01_daily_delta_hm.csv', index=False)

raw_bbm['TAHUN_BULAN'] = raw_bbm['TANGGAL'].dt.to_period('M')
monthly_hm = (raw_bbm.groupby(['EQUIP NAME', 'TAHUN_BULAN'])['DELTA_HM']
              .sum().round(1).reset_index().rename(columns={'DELTA_HM': 'UTILITAS'}))

monthly_hm.to_csv(OUT_DIR / '02_monthly_operating_hours.csv', index=False)
print(monthly_hm.shape, '| range:', monthly_hm['TAHUN_BULAN'].min(), '-', monthly_hm['TAHUN_BULAN'].max())


(10959, 3) | range: 2023-01 - 2025-12


In [5]:
MONTH_MAP = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
             'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}

TARGET_CURRSTATE = {'FXD', 'FAC', 'STR', 'MAS', 'MTB', 'FOB', 'MXD', 'MOB'}

PREFERRED_PREV = {
    'FXD': 'All', 'FAC': 'All', 'STR': 'All', 'MAS': 'All',
    'MTB': 'All', 'FOB': 'All', 'MXD': 'All', 'MOB': 'MTL',
}

VALID_CONTAINER_TYPES = {
    '10 CO', '10 DC', '20 CO', '20 DC', '20 FT', '20 HC', '20 IT', '20 OT', '20 RF', '20 RH', '20 RM',
    '21 DC', '40 CO', '40 DC', '40 FT', '40 HC', '40 IT', '40 OT', '40 RF', '40 RH', '40 RM', '41 DC', '45 HC',
}

def parse_container_file(path, branch_map, target_currstate=TARGET_CURRSTATE, preferred_prev=PREFERRED_PREV):
    df = pd.read_excel(path, sheet_name=0, header=None)
    col0 = df.iloc[:, 0]
    starts = col0[col0 == 'CURRSTATE'].index.tolist()

    monthly_totals = {m: 0.0 for m in range(1, 13)}
    branch_rows_log = []
    blocks_used = []

    for i, s in enumerate(starts):
        curr = df.iloc[s, 1]
        prev = df.iloc[s + 1, 1]
        if curr not in target_currstate or prev != preferred_prev.get(curr):
            continue
        blocks_used.append((curr, prev))

        end = starts[i + 1] if i + 1 < len(starts) else len(df)
        coc_row = df.iloc[s + 6, :].copy()
        month_row = df.iloc[s + 7, :].copy()
        type_row = df.iloc[s + 8, :]
        coc_row.iloc[1:] = coc_row.iloc[1:].ffill()
        month_row.iloc[1:] = month_row.iloc[1:].ffill()

        data_start = s + 9
        block_col0 = df.iloc[data_start:end, 0]
        gt_idx = block_col0[block_col0 == 'Grand Total'].index
        data_end = gt_idx[0] if len(gt_idx) > 0 else end
        branch_rows = df.iloc[data_start:data_end]

        for col_idx in range(1, df.shape[1]):
            mon = month_row.iloc[col_idx]
            typ = type_row.iloc[col_idx]
            if pd.isna(mon) or mon not in MONTH_MAP:
                continue
            if typ not in VALID_CONTAINER_TYPES:
                continue
            vals = pd.to_numeric(branch_rows.iloc[:, col_idx], errors='coerce').fillna(0)
            monthly_totals[MONTH_MAP[mon]] += vals.sum()

            for r_idx in range(len(branch_rows)):
                branch_raw = branch_rows.iloc[r_idx, 0]
                if pd.isna(branch_raw):
                    continue
                v = vals.iloc[r_idx]
                if v != 0:
                    branch_rows_log.append({
                        'CABANG': branch_map.get(branch_raw, branch_raw),
                        'CURRSTATE': curr, 'BULAN': MONTH_MAP[mon],
                        'TIPE_CONTAINER': typ, 'VOLUME': v,
                    })

    return monthly_totals, pd.DataFrame(branch_rows_log), blocks_used

container_monthly_frames = []
container_branch_checkpoints = []
for yr, path in CONTAINER_FILES.items():
    totals, branch_df, used = parse_container_file(path, branch_map)
    print(yr, 'blok dipakai:', used, '| total tahun:', sum(totals.values()))
    for m, v in totals.items():
        container_monthly_frames.append({'TAHUN_BULAN': pd.Period(f'{yr}-{m:02d}', freq='M'), 'VOLUME_CONTAINER': v})
    branch_df['TAHUN'] = yr
    container_branch_checkpoints.append(branch_df)

monthly_container = pd.DataFrame(container_monthly_frames).sort_values('TAHUN_BULAN').reset_index(drop=True)
container_branch_detail = pd.concat(container_branch_checkpoints, ignore_index=True)

monthly_container.to_csv(OUT_DIR / '03a_monthly_container.csv', index=False)
container_branch_detail.to_csv(OUT_DIR / '03a_container_per_cabang_detail.csv', index=False)


2023 blok dipakai: [('FXD', 'All'), ('FAC', 'All'), ('STR', 'All'), ('MAS', 'All'), ('MTB', 'All'), ('FOB', 'All'), ('MXD', 'All'), ('MOB', 'MTL')] | total tahun: 4330703.0
2024 blok dipakai: [('FXD', 'All'), ('FAC', 'All'), ('STR', 'All'), ('MAS', 'All'), ('MTB', 'All'), ('FOB', 'All'), ('MXD', 'All'), ('MOB', 'MTL')] | total tahun: 4323469.0
2025 blok dipakai: [('FXD', 'All'), ('FAC', 'All'), ('STR', 'All'), ('MAS', 'All'), ('MTB', 'All'), ('FOB', 'All'), ('MXD', 'All'), ('MOB', 'MTL')] | total tahun: 4012779.0


In [6]:
import matplotlib
matplotlib.use('Agg')  # aman dijalankan headless/server (tidak butuh GUI)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly_container['TAHUN_BULAN'].astype(str), monthly_container['VOLUME_CONTAINER'], marker='o')
ax.set_title('Volume container per bulan (sum semua CURRSTATE) -- cek outlier!')
ax.set_ylim(bottom=0)
plt.xticks(rotation=60)
plt.tight_layout()
plt.savefig(OUT_DIR / '03b_container_volume_outlier_check.png', dpi=150)
plt.close(fig)

print(monthly_container.describe())
print(f'Plot outlier volume container tersimpan: {OUT_DIR / "03b_container_volume_outlier_check.png"}')


       VOLUME_CONTAINER
count         36.000000
mean      351859.750000
std        25862.723744
min       271398.000000
25%       341021.000000
50%       355495.000000
75%       368425.750000
max       396862.000000
Plot outlier volume container tersimpan: checkpoints_v4\03b_container_volume_outlier_check.png


In [7]:
rain = pd.read_excel(RAINFALL_FILE, header=3)
rain = rain.rename(columns={'precipitation_sum (mm)': 'precipitation_sum'})
rain['time'] = pd.to_datetime(rain['time'])
rain = rain.set_index('time')

monthly_rain = rain['precipitation_sum'].resample('M').sum().round(1).reset_index()
monthly_rain['TAHUN_BULAN'] = monthly_rain['time'].dt.to_period('M')
monthly_rain = monthly_rain[['TAHUN_BULAN', 'precipitation_sum']].rename(columns={'precipitation_sum': 'CURAH_HUJAN'})

monthly_full = monthly_hm.merge(monthly_rain, on='TAHUN_BULAN', how='left')
monthly_full = monthly_full.merge(monthly_container, on='TAHUN_BULAN', how='left')

# ── Populasi unit resmi = ADA di cost & bbm HP&Type SAJA (identik v3) ──
cost_df = pd.read_excel(COST_FILE, sheet_name='Sheet2', header=1)
kriteria_raw = cost_df[['NAMA ALAT BERAT', 'DES 2025', 'ALAT BERAT', 'TYPE/MERK', 'CAP', 'HP']].copy()
kriteria_raw = kriteria_raw.rename(columns={'DES 2025': 'LOKASI_RAW', 'ALAT BERAT': 'JENIS'})

MANUAL_OVERRIDE = {
    'TOP LOADER KALMAR 35 T/TOBATI': 'TOBATI (EX.FL KALMAR 32T)',
    'L 9902 UR / S75': 'L 8477 UUC (EX.L 9902 UR)',
    'TOP LOADER BOSS': 'WIND RIVER (EX.TL BOSS 42T)',
}

def extract_after_ex(name):
    m = re.search(r'EX\.\s*([^)]+)\)', str(name), re.IGNORECASE)
    return m.group(1).strip() if m else None

def extract_before_paren(name):
    name = str(name)
    return name.split('(')[0].strip() if '(' in name else None

unit_ids_with_data = set(monthly_hm['EQUIP NAME'].unique())

alias_after_ex, alias_before_paren = {}, {}
for full_name, uid in unit_map.items():
    if uid not in unit_ids_with_data:
        continue
    after = extract_after_ex(full_name)
    if after:
        alias_after_ex.setdefault(after.upper(), full_name)
    before = extract_before_paren(full_name)
    if before:
        alias_before_paren.setdefault(before.upper(), full_name)

def resolve_to_data_bearing_unit(nama_cost):
    nama_cost = str(nama_cost).strip()
    uid = unit_map.get(nama_cost)
    if uid and uid in unit_ids_with_data:
        return nama_cost, 'exact'
    if nama_cost in MANUAL_OVERRIDE and unit_map.get(MANUAL_OVERRIDE[nama_cost]) in unit_ids_with_data:
        return MANUAL_OVERRIDE[nama_cost], 'manual_override'
    upper = nama_cost.upper()
    if upper in alias_after_ex:
        return alias_after_ex[upper], 'match_after_EX'
    if upper in alias_before_paren:
        return alias_before_paren[upper], 'match_before_paren'
    return None, 'unmatched'

resolved = kriteria_raw['NAMA ALAT BERAT'].apply(
    lambda x: pd.Series(resolve_to_data_bearing_unit(x), index=['NAMA_RESOLVED', 'METODE_MATCH'])
)
kriteria_raw = pd.concat([kriteria_raw, resolved], axis=1)
kriteria_raw['EQUIP NAME'] = kriteria_raw['NAMA_RESOLVED'].map(unit_map)
kriteria_raw['LOKASI'] = kriteria_raw['LOKASI_RAW'].map(lokasi_map)

print('Ringkasan metode matching cost&bbm -> unit BBM AAB:')
print(kriteria_raw['METODE_MATCH'].value_counts())

kriteria_raw = kriteria_raw.drop(columns=['NAMA ALAT BERAT', 'LOKASI_RAW', 'NAMA_RESOLVED'])
kriteria_raw['KRITERIA_ID'] = kriteria_raw[['JENIS', 'TYPE/MERK', 'CAP', 'HP']].astype(str).agg('|'.join, axis=1)
kriteria = (kriteria_raw.dropna(subset=['EQUIP NAME'])
            .drop_duplicates(subset=['EQUIP NAME'])
            .reset_index(drop=True))
kriteria.to_csv(OUT_DIR / '08_kriteria_unit.csv', index=False)

unit_resmi = set(kriteria['EQUIP NAME'])
semua_unit_bbm = set(monthly_full['EQUIP NAME'].unique())
unit_dikecualikan = semua_unit_bbm - unit_resmi

inv_unit_map = {v: k for k, v in unit_map.items()}
excluded_df0 = pd.DataFrame([
    {'EQUIP NAME': u, 'NAMA_ASLI': inv_unit_map.get(u, u)} for u in unit_dikecualikan
]).sort_values('NAMA_ASLI')
excluded_df0.to_csv(OUT_DIR / '00b_unit_tidak_ada_di_costbbm_dikecualikan.csv', index=False)
        
print(f'Unit di BBM AAB: {len(semua_unit_bbm)} | match cost&bbm (dipakai): {len(unit_resmi)} | dikecualikan: {len(excluded_df0)}')

monthly_full = monthly_full[monthly_full['EQUIP NAME'].isin(unit_resmi)].reset_index(drop=True)
monthly_full.to_csv(OUT_DIR / '04_monthly_with_exog.csv', index=False)
print('Jumlah unit final di monthly_full:', monthly_full['EQUIP NAME'].nunique())


Ringkasan metode matching cost&bbm -> unit BBM AAB:
METODE_MATCH
exact                 196
match_after_EX         39
unmatched               5
manual_override         3
match_before_paren      2
Name: count, dtype: int64
Unit di BBM AAB: 306 | match cost&bbm (dipakai): 240 | dikecualikan: 66
Jumlah unit final di monthly_full: 240


In [8]:
# ── Exclude unit non-operasional / alat pendukung sejak tahap awal ETL ──
EXCLUDED_OLD_CODES = {
    'UNIT_0306', 'UNIT_0336', 'UNIT_0088', 'UNIT_0134', 'UNIT_0159',
    'UNIT_0166', 'UNIT_0095', 'UNIT_0037', 'UNIT_0330', 'UNIT_0096',
}

_old_codes_sorted = sorted(set(unit_map.values()), key=lambda c: int(c.split('_')[1]))
_remaining_old_codes = [c for c in _old_codes_sorted if c not in EXCLUDED_OLD_CODES]
REMAP_OLD_TO_NEW = {old: f'UNIT_{i+1:04d}' for i, old in enumerate(_remaining_old_codes)}

monthly_full = monthly_full[~monthly_full['EQUIP NAME'].isin(EXCLUDED_OLD_CODES)].copy()
monthly_full['EQUIP NAME'] = monthly_full['EQUIP NAME'].map(REMAP_OLD_TO_NEW)

kriteria = kriteria[~kriteria['EQUIP NAME'].isin(EXCLUDED_OLD_CODES)].copy()
kriteria['EQUIP NAME'] = kriteria['EQUIP NAME'].map(REMAP_OLD_TO_NEW)

# Eliminasi 15 unit non-operasional tambahan dari populasi armada awal
EXCLUDED_ADDITIONAL_15 = {
    'UNIT_0006', 'UNIT_0142', 'UNIT_0220', 'UNIT_0187', 'UNIT_0080',
    'UNIT_0021', 'UNIT_0344', 'UNIT_0091', 'UNIT_0197', 'UNIT_0185',
    'UNIT_0081', 'UNIT_0339', 'UNIT_0027', 'UNIT_0301', 'UNIT_0024',
}
monthly_full = monthly_full[~monthly_full['EQUIP NAME'].isin(EXCLUDED_ADDITIONAL_15)].copy()
kriteria = kriteria[~kriteria['EQUIP NAME'].isin(EXCLUDED_ADDITIONAL_15)].copy()

unit_map = {name: REMAP_OLD_TO_NEW[code] for name, code in unit_map.items() if code in REMAP_OLD_TO_NEW and REMAP_OLD_TO_NEW[code] not in EXCLUDED_ADDITIONAL_15}
with open(OUT_DIR / 'unit_map_v4.json', 'w') as f:
    json.dump(unit_map, f, indent=2)

kriteria.to_csv(OUT_DIR / '08_kriteria_unit.csv', index=False)
monthly_full.to_csv(OUT_DIR / '04_monthly_with_exog.csv', index=False)

print(f'Jumlah unit armada operasional resmi awal: {monthly_full["EQUIP NAME"].nunique()}')


Unit dikecualikan: 10 | Unit tersisa & diremap ulang: 345
Jumlah unit final setelah exclude+remap: 230


In [9]:
completeness = (monthly_full.groupby('EQUIP NAME')['TAHUN_BULAN']
                 .agg(['min', 'max', 'count'])
                 .rename(columns={'min': 'BULAN_AWAL', 'max': 'BULAN_AKHIR', 'count': 'JUMLAH_BULAN'}))
completeness['KONDISI'] = np.where(completeness['JUMLAH_BULAN'] >= 36, 'DATA_PENUH', 'DATA_PARSIAL')
completeness.to_csv(OUT_DIR / '05_completeness_per_unit.csv')
print(completeness['KONDISI'].value_counts())
print(f'Completeness per unit tersimpan: {OUT_DIR / "05_completeness_per_unit.csv"}')


KONDISI
DATA_PENUH    230
Name: count, dtype: int64
Completeness per unit tersimpan: checkpoints_v4\05_completeness_per_unit.csv


In [10]:
# -- Excel: pemetaan KRITERIA -> unit anggota + data HM lengkap (biar gak perlu buka raw pas sidang) --
kriteria_hm = monthly_full.merge(kriteria[['EQUIP NAME', 'KRITERIA_ID', 'JENIS', 'TYPE/MERK', 'CAP', 'HP']],
                                  on='EQUIP NAME', how='left')

ringkasan_kriteria = (kriteria_hm.groupby('KRITERIA_ID')
                       .agg(JUMLAH_UNIT=('EQUIP NAME', 'nunique'),
                            DAFTAR_UNIT=('EQUIP NAME', lambda x: ', '.join(sorted(x.unique()))),
                            TOTAL_HM=('UTILITAS', 'sum'),
                            RATA2_HM_PER_BULAN=('UTILITAS', 'mean'),
                            JUMLAH_BARIS_BULAN=('UTILITAS', 'count'))
                       .reset_index()
                       .sort_values('JUMLAH_UNIT', ascending=False))

detail_hm = (kriteria_hm[['KRITERIA_ID', 'EQUIP NAME', 'TAHUN_BULAN', 'UTILITAS']]
             .sort_values(['KRITERIA_ID', 'EQUIP NAME', 'TAHUN_BULAN']).copy())
detail_hm['TAHUN_BULAN'] = detail_hm['TAHUN_BULAN'].astype(str)

pivot_hm = (kriteria_hm.assign(TAHUN_BULAN=kriteria_hm['TAHUN_BULAN'].astype(str))
            .pivot_table(index='TAHUN_BULAN', columns=['KRITERIA_ID', 'EQUIP NAME'],
                         values='UTILITAS', aggfunc='sum')
            .sort_index())

total_hm_per_kriteria_bulan = (kriteria_hm.assign(TAHUN_BULAN=kriteria_hm['TAHUN_BULAN'].astype(str))
                                .pivot_table(index='TAHUN_BULAN', columns='KRITERIA_ID',
                                             values='UTILITAS', aggfunc='sum')
                                .sort_index())

total_hm_per_kriteria_long = (kriteria_hm.assign(TAHUN_BULAN=kriteria_hm['TAHUN_BULAN'].astype(str))
                               .groupby(['KRITERIA_ID', 'TAHUN_BULAN'])['UTILITAS']
                               .sum().reset_index()
                               .rename(columns={'UTILITAS': 'TOTAL_HM_KRITERIA'})
                               .sort_values(['KRITERIA_ID', 'TAHUN_BULAN']))

KRITERIA_MAPPING_EXCEL = OUT_DIR / 'v4_kriteria_unit_mapping.xlsx'
with pd.ExcelWriter(KRITERIA_MAPPING_EXCEL, engine='openpyxl') as writer:
    ringkasan_kriteria.to_excel(writer, sheet_name='Ringkasan_Kriteria', index=False)
    total_hm_per_kriteria_long.to_excel(writer, sheet_name='Total_HM_per_Kriteria_per_Bulan', index=False)
    total_hm_per_kriteria_bulan.to_excel(writer, sheet_name='Pivot_Total_HM_per_Kriteria')
    detail_hm.to_excel(writer, sheet_name='Detail_HM_per_Unit_Kriteria', index=False)
    pivot_hm.to_excel(writer, sheet_name='Pivot_HM_per_Kriteria')

print(f'Kriteria-unit-HM mapping tersimpan: {KRITERIA_MAPPING_EXCEL}')
print(f'{ringkasan_kriteria.shape[0]} kriteria, {kriteria_hm["EQUIP NAME"].nunique()} unit, '
      f'periode {detail_hm["TAHUN_BULAN"].min()} s/d {detail_hm["TAHUN_BULAN"].max()}')


Kriteria-unit-HM mapping tersimpan: checkpoints_v4\v4_kriteria_unit_mapping.xlsx
85 kriteria, 230 unit, periode 2023-01 s/d 2025-12


## 2b. EDA (Analisa Data) -- Trendline, Deskriptif, Distribusi, Outlier, Korelasi

Dipakai: `monthly_full` (222 unit final, SEBELUM split train/test per horizon).
- UTILITAS: trendline & korelasi pakai **agregat bulanan** (total & mean semua unit, sejalan dgn CURAH_HUJAN/VOLUME_CONTAINER yg memang sudah 1 series/bulan); mean/median/skewness/distribusi/outlier pakai **pooled semua baris unit-bulan**.
- CURAH_HUJAN & VOLUME_CONTAINER: semua EDA pakai series bulanan apa adanya (sudah 1 angka/bulan).
Outlier IQR: `eda_08_outlier_iqr.csv` (ringkasan) + `eda_08b_outlier_detail.csv` (titik per unit/bulan yg jadi outlier, lengkap dgn label), boxplot juga dikasih label 3 titik paling ekstrem.
Semua output disimpan di `./checkpoints_v4/eda/` (CSV + PNG), tidak mengubah variabel apa pun yg dipakai sel sesudahnya.


In [11]:
EDA_DIR = OUT_DIR / 'eda'
EDA_DIR.mkdir(exist_ok=True)

from scipy.stats import skew

# -- Agregat bulanan (dipakai utk trendline & korelasi) --
col_target = 'OPERATING_HOURS' if 'OPERATING_HOURS' in monthly_full.columns else 'UTILITAS'
eda_monthly = (monthly_full.groupby('TAHUN_BULAN')
               .agg(OPERATING_HOURS_TOTAL=(col_target, 'sum'),
                    OPERATING_HOURS_MEAN=(col_target, 'mean'),
                    CURAH_HUJAN=('CURAH_HUJAN', 'first'),
                    VOLUME_CONTAINER=('VOLUME_CONTAINER', 'first'))
               .sort_index())
eda_monthly.to_csv(EDA_DIR / 'eda_00_agregat_bulanan.csv')
print(f'[EDA] Agregat bulanan: {eda_monthly.shape} | tersimpan: {EDA_DIR / "eda_00_agregat_bulanan.csv"}')
eda_monthly.head()


[EDA] Agregat bulanan: (36, 4) | tersimpan: checkpoints_v4\eda\eda_00_agregat_bulanan.csv


,UTILITAS_TOTAL,UTILITAS_MEAN,CURAH_HUJAN,VOLUME_CONTAINER
TAHUN_BULAN,,,,
2023-01,52551.6,228.485217,254.7,368115.0
2023-02,50054.4,217.627826,362.1,355535.0
2023-03,56599.7,246.085652,273.7,350609.0
2023-04,40667.6,176.815652,243.4,296614.0
2023-05,49033.3,213.188261,76.1,365756.0


In [12]:
def _plot_trend(series, title, fname, ylabel):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(series.index.astype(str), series.values, marker='o')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_ylim(bottom=0)
    plt.xticks(rotation=60)
    plt.tight_layout()
    plt.savefig(EDA_DIR / fname, dpi=150)
    plt.close(fig)
    print(f'[EDA] Trendline tersimpan: {EDA_DIR / fname}')


_plot_trend(eda_monthly['OPERATING_HOURS_TOTAL'], 'Trendline Operating Hours (total semua unit per bulan)',
            'eda_01_trend_operating_hours_total.png', 'Operating Hours (jam)')
_plot_trend(eda_monthly['OPERATING_HOURS_MEAN'], 'Trendline Operating Hours (rata-rata semua unit per bulan)',
            'eda_02_trend_operating_hours_mean.png', 'Operating Hours (jam)')
_plot_trend(eda_monthly['CURAH_HUJAN'], 'Trendline CURAH_HUJAN (Surabaya) per bulan',
            'eda_03_trend_curah_hujan.png', 'mm')
_plot_trend(eda_monthly['VOLUME_CONTAINER'], 'Trendline VOLUME_CONTAINER per bulan',
            'eda_04_trend_volume_container.png', 'TEUs')


[EDA] Trendline tersimpan: checkpoints_v4\eda\eda_01_trend_utilitas_total.png
[EDA] Trendline tersimpan: checkpoints_v4\eda\eda_01b_trend_utilitas_mean.png
[EDA] Trendline tersimpan: checkpoints_v4\eda\eda_02_trend_curah_hujan.png
[EDA] Trendline tersimpan: checkpoints_v4\eda\eda_03_trend_volume_container.png


In [13]:
col_target = 'OPERATING_HOURS' if 'OPERATING_HOURS' in monthly_full.columns else 'UTILITAS'
eda_vars = {
    'Operating Hours (pooled unit-bulan)': monthly_full[col_target].dropna(),
    'CURAH_HUJAN (bulanan)': eda_monthly['CURAH_HUJAN'].dropna(),
    'VOLUME_CONTAINER (bulanan)': eda_monthly['VOLUME_CONTAINER'].dropna(),
}

desc_rows = []
for name, s in eda_vars.items():
    s = s.astype(float)
    desc_rows.append({
        'VARIABEL': name, 'N': len(s), 'MEAN': s.mean(), 'MEDIAN': s.median(),
        'STD': s.std(), 'MIN': s.min(), 'MAX': s.max(), 'SKEWNESS': skew(s),
    })
desc_df = pd.DataFrame(desc_rows)
desc_df.to_csv(EDA_DIR / 'eda_04_deskriptif_skewness.csv', index=False)
print(desc_df.to_string(index=False))
desc_df


                    VARIABEL    N          MEAN   MEDIAN          STD      MIN      MAX  SKEWNESS
UTILITAS (pooled unit-bulan) 8280    224.071135    200.0   178.257323      0.0    744.0  0.541273
       CURAH_HUJAN (bulanan)   36    157.975000    121.4   144.147394      0.1    426.0  0.334191
  VOLUME_CONTAINER (bulanan)   36 351859.750000 355495.0 25862.723744 271398.0 396862.0 -1.058057


,VARIABEL,N,MEAN,MEDIAN,STD,MIN,MAX,SKEWNESS
0,UTILITAS (pooled unit-bulan),8280,224.071135,200.0,178.257323,0.0,744.0,0.541273
1,CURAH_HUJAN (bulanan),36,157.975000,121.4,144.147394,0.1,426.0,0.334191
2,VOLUME_CONTAINER (bulanan),36,351859.750000,355495.0,25862.723744,271398.0,396862.0,-1.058057


In [14]:
def _plot_hist(s, title, fname, xlabel, bins=30):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(s.astype(float), bins=bins, edgecolor='black')
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Frekuensi')
    ax.set_ylim(bottom=0)
    plt.tight_layout()
    plt.savefig(EDA_DIR / fname, dpi=150)
    plt.close(fig)
    print(f'[EDA] Distribusi tersimpan: {EDA_DIR / fname}')


_plot_hist(eda_vars['Operating Hours (pooled unit-bulan)'], 'Distribusi Operating Hours (pooled seluruh unit-bulan)',
           'eda_05_dist_operating_hours.png', 'Operating Hours (jam)', bins=30)
_plot_hist(eda_vars['CURAH_HUJAN (bulanan)'], 'Distribusi CURAH_HUJAN (bulanan)',
           'eda_06_dist_curah_hujan.png', 'mm', bins=12)
_plot_hist(eda_vars['VOLUME_CONTAINER (bulanan)'], 'Distribusi VOLUME_CONTAINER (bulanan)',
           'eda_07_dist_volume_container.png', 'TEUs', bins=12)


[EDA] Distribusi tersimpan: checkpoints_v4\eda\eda_05_dist_utilitas.png
[EDA] Distribusi tersimpan: checkpoints_v4\eda\eda_06_dist_curah_hujan.png
[EDA] Distribusi tersimpan: checkpoints_v4\eda\eda_07_dist_volume_container.png


In [15]:
def _label_frame(name):
    """Kembalikan DataFrame NILAI + LABEL (unit/bulan atau bulan saja) utk identifikasi outlier."""
    if name.startswith('Operating Hours') or name.startswith('UTILITAS') or name.startswith('Operating Hours'):
        df = (monthly_full[['EQUIP NAME', 'TAHUN_BULAN', 'UTILITAS']]
              .dropna(subset=['UTILITAS'])
              .rename(columns={'UTILITAS': 'NILAI'}))
        df['LABEL'] = df['EQUIP NAME'] + ' / ' + df['TAHUN_BULAN'].astype(str)
    else:
        col = 'CURAH_HUJAN' if 'CURAH_HUJAN' in name else 'VOLUME_CONTAINER'
        df = eda_monthly.reset_index()[['TAHUN_BULAN', col]].dropna().rename(columns={col: 'NILAI'})
        df['LABEL'] = df['TAHUN_BULAN'].astype(str)
    return df


def _iqr_outlier_count(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((s < low) | (s > high)).sum())
    return n_out, low, high


outlier_rows = []
detail_frames = []
for name, s in eda_vars.items():
    n_out, low, high = _iqr_outlier_count(s)
    outlier_rows.append({'VARIABEL': name, 'N': len(s), 'N_OUTLIER_IQR': n_out,
                          'PCT_OUTLIER': round(100 * n_out / len(s), 2),
                          'BATAS_BAWAH_IQR': low, 'BATAS_ATAS_IQR': high})

    # -- Identifikasi: titik mana (unit/bulan, atau bulan) yg jadi outlier --
    lf = _label_frame(name)
    out_rows = lf[(lf['NILAI'] < low) | (lf['NILAI'] > high)].copy()
    out_rows['VARIABEL'] = name
    out_rows['BATAS_BAWAH_IQR'] = low
    out_rows['BATAS_ATAS_IQR'] = high
    out_rows['SELISIH_DARI_BATAS'] = np.where(out_rows['NILAI'] > high,
                                               out_rows['NILAI'] - high,
                                               low - out_rows['NILAI'])
    detail_frames.append(out_rows.sort_values('SELISIH_DARI_BATAS', ascending=False))

outlier_df = pd.DataFrame(outlier_rows)
outlier_df.to_csv(EDA_DIR / 'eda_08_outlier_iqr.csv', index=False)
print(outlier_df.to_string(index=False))

outlier_detail_df = pd.concat(detail_frames, ignore_index=True) if detail_frames else pd.DataFrame()
outlier_detail_df.to_csv(EDA_DIR / 'eda_08b_outlier_detail.csv', index=False)
print(f'\n[EDA] Detail titik outlier (semua, bukan cuma 3 teratas) tersimpan: {EDA_DIR / "eda_08b_outlier_detail.csv"}')
for name in eda_vars:
    sub = outlier_detail_df[outlier_detail_df['VARIABEL'] == name].head(10)
    if not sub.empty:
        print(f"\n-- {name}: {len(outlier_detail_df[outlier_detail_df['VARIABEL'] == name])} outlier, 10 terekstrem --")
        print(sub[['LABEL', 'NILAI', 'SELISIH_DARI_BATAS']].to_string(index=False))


def _plot_box(s, title, fname, label_df, var_name, ylabel):
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.boxplot(s.astype(float).values, vert=True)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_ylim(bottom=0)
    ax.set_xticks([1])
    ax.set_xticklabels([var_name])
    # -- Beri label 3 titik paling ekstrem di atas & bawah, biar kebaca bulan/unit-nya --
    top = label_df.sort_values('SELISIH_DARI_BATAS', ascending=False).head(3)
    for _, r in top.iterrows():
        ax.annotate(r['LABEL'], xy=(1, r['NILAI']), xytext=(1.08, r['NILAI']),
                    fontsize=7, va='center')
    plt.tight_layout()
    plt.savefig(EDA_DIR / fname, dpi=150)
    plt.close(fig)
    print(f'[EDA] Boxplot tersimpan: {EDA_DIR / fname}')


_plot_box(eda_vars['Operating Hours (pooled unit-bulan)'], 'Outlier Operating Hours (pooled unit-bulan)',
          'eda_09_box_operating_hours.png',
          outlier_detail_df[outlier_detail_df['VARIABEL'] == 'Operating Hours (pooled unit-bulan)'],
          'Operating Hours', 'Operating Hours (jam)')
_plot_box(eda_vars['CURAH_HUJAN (bulanan)'], 'Outlier CURAH_HUJAN (bulanan)',
          'eda_10_box_curah_hujan.png',
          outlier_detail_df[outlier_detail_df['VARIABEL'] == 'CURAH_HUJAN (bulanan)'],
          'CURAH_HUJAN', 'mm')
_plot_box(eda_vars['VOLUME_CONTAINER (bulanan)'], 'Outlier VOLUME_CONTAINER (bulanan)',
          'eda_11_box_volume_container.png',
          outlier_detail_df[outlier_detail_df['VARIABEL'] == 'VOLUME_CONTAINER (bulanan)'],
          'VOLUME_CONTAINER', 'TEUs')
outlier_df


                    VARIABEL    N  N_OUTLIER_IQR  PCT_OUTLIER  BATAS_BAWAH_IQR  BATAS_ATAS_IQR
UTILITAS (pooled unit-bulan) 8280              0         0.00        -388.0625        808.8375
       CURAH_HUJAN (bulanan)   36              0         0.00        -367.0500        658.3500
  VOLUME_CONTAINER (bulanan)   36              2         5.56      299913.8750     409532.8750

[EDA] Detail titik outlier (semua, bukan cuma 3 teratas) tersimpan: checkpoints_v4\eda\eda_08b_outlier_detail.csv

-- VOLUME_CONTAINER (bulanan): 2 outlier, 10 terekstrem --
  LABEL    NILAI  SELISIH_DARI_BATAS
2025-04 271398.0           28515.875
2023-04 296614.0            3299.875
[EDA] Boxplot tersimpan: checkpoints_v4\eda\eda_09_box_utilitas.png
[EDA] Boxplot tersimpan: checkpoints_v4\eda\eda_10_box_curah_hujan.png
[EDA] Boxplot tersimpan: checkpoints_v4\eda\eda_11_box_volume_container.png


,VARIABEL,N,N_OUTLIER_IQR,PCT_OUTLIER,BATAS_BAWAH_IQR,BATAS_ATAS_IQR
0,UTILITAS (pooled unit-bulan),8280,0,0.00,-388.0625,808.8375
1,CURAH_HUJAN (bulanan),36,0,0.00,-367.0500,658.3500
2,VOLUME_CONTAINER (bulanan),36,2,5.56,299913.8750,409532.8750


In [16]:
corr_df = eda_monthly[['OPERATING_HOURS_MEAN', 'CURAH_HUJAN', 'VOLUME_CONTAINER']].corr(method='pearson')
corr_df.to_csv(EDA_DIR / 'eda_12_korelasi_pearson.csv')
print(corr_df.round(3).to_string())

fig, ax = plt.subplots(figsize=(6, 4.8))
im = ax.imshow(corr_df.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_df.columns)))
ax.set_xticklabels(corr_df.columns, rotation=35, ha='right')
ax.set_yticks(range(len(corr_df.columns)))
ax.set_yticklabels(corr_df.columns)
for i in range(len(corr_df)):
    for j in range(len(corr_df)):
        ax.text(j, i, f'{corr_df.values[i, j]:.2f}', ha='center', va='center')
plt.colorbar(im)
ax.set_title('Korelasi antar variabel (agregat bulanan)')
plt.tight_layout()
plt.savefig(EDA_DIR / 'eda_13_korelasi_heatmap.png', dpi=150)
plt.close(fig)
print(f'[EDA] Heatmap korelasi tersimpan: {EDA_DIR / "eda_13_korelasi_heatmap.png"}')
corr_df


                      OPERATING_HOURS_MEAN  CURAH_HUJAN  VOLUME_CONTAINER
OPERATING_HOURS_MEAN                 1.000       -0.313             0.697
CURAH_HUJAN                         -0.313        1.000            -0.249
VOLUME_CONTAINER                     0.697       -0.249             1.000
[EDA] Heatmap korelasi tersimpan: checkpoints_v4\eda\eda_13_korelasi_heatmap.png


,OPERATING_HOURS_MEAN,CURAH_HUJAN,VOLUME_CONTAINER
OPERATING_HOURS_MEAN,1.000000,-0.312757,0.696991
CURAH_HUJAN,-0.312757,1.000000,-0.248796
VOLUME_CONTAINER,0.696991,-0.248796,1.000000


In [17]:
# ── EDA Skewness & Outlier IQR per Unit (230 unit pada monthly_full) ──
# Catatan: Jika kernel baru dibuka dan belum menjalankan bagian 1 & 2, blok di bawah otomatis
# memuat checkpoint './checkpoints_v4/04_monthly_with_exog.csv' agar sel ini bisa langsung di-run sendiri.
if 'monthly_full' not in globals():
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    from pathlib import Path
    from scipy.stats import skew
    OUT_DIR = Path('./checkpoints_v4')
    EDA_DIR = OUT_DIR / 'eda'
    EDA_DIR.mkdir(parents=True, exist_ok=True)
    monthly_full = pd.read_csv(OUT_DIR / '04_monthly_with_exog.csv')
    if 'OPERATING_HOURS' not in monthly_full.columns and 'UTILITAS' in monthly_full.columns:
        monthly_full['OPERATING_HOURS'] = monthly_full['UTILITAS']

if '_iqr_outlier_count' not in globals():
    def _iqr_outlier_count(s):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = int(((s < low) | (s > high)).sum())
        return n_out, low, high


def _kategori_skew(sk):
    if sk < -1.0:
        return 'Left-Skewed Ekstrem (s < -1.0)'
    elif sk < -0.5:
        return 'Left-Skewed Moderat (-1.0 <= s < -0.5)'
    elif sk <= 0.5:
        return 'Simetris / Normal (-0.5 <= s <= 0.5)'
    elif sk <= 1.0:
        return 'Right-Skewed Moderat (0.5 < s <= 1.0)'
    else:
        return 'Right-Skewed Ekstrem (s > 1.0)'


unit_skew_rows = []
for unit, grp in monthly_full.groupby('EQUIP NAME'):
    col_target = 'OPERATING_HOURS' if 'OPERATING_HOURS' in grp.columns else 'UTILITAS'
    s = grp[col_target].dropna().astype(float)
    sk = float(skew(s)) if s.std() > 0 else 0.0
    n_out, low, high = _iqr_outlier_count(s)
    unit_skew_rows.append({
        'EQUIP NAME': unit,
        'N': len(s),
        'MEAN': s.mean(),
        'MEDIAN': s.median(),
        'STD': s.std(),
        'MIN': s.min(),
        'MAX': s.max(),
        'SKEWNESS': sk,
        'KATEGORI_SKEWNESS': _kategori_skew(sk),
        'N_OUTLIER_IQR': n_out,
        'PCT_OUTLIER_IQR': round(100 * n_out / len(s), 2),
    })

unit_skew_df = pd.DataFrame(unit_skew_rows).sort_values('EQUIP NAME').reset_index(drop=True)
unit_skew_df.to_csv(EDA_DIR / 'eda_14_skewness_per_unit_detail.csv', index=False)

n_total = len(unit_skew_df)
kat_order = [
    'Left-Skewed Ekstrem (s < -1.0)',
    'Left-Skewed Moderat (-1.0 <= s < -0.5)',
    'Simetris / Normal (-0.5 <= s <= 0.5)',
    'Right-Skewed Moderat (0.5 < s <= 1.0)',
    'Right-Skewed Ekstrem (s > 1.0)',
]

summary_rows = []
for kat in kat_order:
    sub = unit_skew_df[unit_skew_df['KATEGORI_SKEWNESS'] == kat]
    cnt = len(sub)
    summary_rows.append({
        'KATEGORI_SKEWNESS': kat,
        'JUMLAH_UNIT': cnt,
        'PERSENTASE_UNIT (%)': round(100 * cnt / n_total, 2),
        'MIN_SKEWNESS': round(sub['SKEWNESS'].min(), 4) if cnt > 0 else np.nan,
        'MAX_SKEWNESS': round(sub['SKEWNESS'].max(), 4) if cnt > 0 else np.nan,
        'UNIT_DENGAN_OUTLIER_IQR': int((sub['N_OUTLIER_IQR'] > 0).sum()),
    })

sub_skewed = unit_skew_df[unit_skew_df['SKEWNESS'].abs() > 0.5]
sub_extreme = unit_skew_df[unit_skew_df['SKEWNESS'].abs() > 1.0]
summary_rows.append({
    'KATEGORI_SKEWNESS': 'TOTAL UNIT MIRING (|s| > 0.5)',
    'JUMLAH_UNIT': len(sub_skewed),
    'PERSENTASE_UNIT (%)': round(100 * len(sub_skewed) / n_total, 2),
    'MIN_SKEWNESS': round(sub_skewed['SKEWNESS'].min(), 4),
    'MAX_SKEWNESS': round(sub_skewed['SKEWNESS'].max(), 4),
    'UNIT_DENGAN_OUTLIER_IQR': int((sub_skewed['N_OUTLIER_IQR'] > 0).sum()),
})
summary_rows.append({
    'KATEGORI_SKEWNESS': 'TOTAL UNIT MIRING EKSTREM (|s| > 1.0)',
    'JUMLAH_UNIT': len(sub_extreme),
    'PERSENTASE_UNIT (%)': round(100 * len(sub_extreme) / n_total, 2),
    'MIN_SKEWNESS': round(sub_extreme['SKEWNESS'].min(), 4),
    'MAX_SKEWNESS': round(sub_extreme['SKEWNESS'].max(), 4),
    'UNIT_DENGAN_OUTLIER_IQR': int((sub_extreme['N_OUTLIER_IQR'] > 0).sum()),
})
summary_rows.append({
    'KATEGORI_SKEWNESS': 'TOTAL SELURUH UNIT',
    'JUMLAH_UNIT': n_total,
    'PERSENTASE_UNIT (%)': 100.0,
    'MIN_SKEWNESS': round(unit_skew_df['SKEWNESS'].min(), 4),
    'MAX_SKEWNESS': round(unit_skew_df['SKEWNESS'].max(), 4),
    'UNIT_DENGAN_OUTLIER_IQR': int((unit_skew_df['N_OUTLIER_IQR'] > 0).sum()),
})

skew_summary_df = pd.DataFrame(summary_rows)
skew_summary_df.to_csv(EDA_DIR / 'eda_15_skewness_per_unit_ringkasan.csv', index=False)

print('=== STATISTIK DESKRIPTIF SKEWNESS PER UNIT (N=230) ===')
print(unit_skew_df['SKEWNESS'].describe().round(4).to_string())
print('\n=== TABEL RINGKASAN KATEGORI SKEWNESS & OUTLIER PER UNIT ===')
print(skew_summary_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(unit_skew_df['SKEWNESS'], bins=25, edgecolor='black', alpha=0.85)
ax.axvline(-0.5, color='darkorange', linestyle='--', linewidth=1.5, label='Batas Simetris (s = ±0.5)')
ax.axvline(0.5, color='darkorange', linestyle='--', linewidth=1.5)
ax.axvline(-1.0, color='crimson', linestyle=':', linewidth=1.5, label='Batas Ekstrem (s = ±1.0)')
ax.axvline(1.0, color='crimson', linestyle=':', linewidth=1.5)
ax.set_title('Distribusi Nilai Skewness Operating Hours per Unit (230 Unit)')
ax.set_xlabel('Skewness Operating Hours per Unit')
ax.set_ylabel('Jumlah Unit')
ax.set_ylim(bottom=0)
ax.legend()
plt.tight_layout()
plt.savefig(EDA_DIR / 'eda_16_dist_skewness_per_unit.png', dpi=150)
plt.close(fig)
print(f'\n[EDA] Detail skewness per unit tersimpan: {EDA_DIR / "eda_14_skewness_per_unit_detail.csv"}')
print(f'[EDA] Ringkasan kategori skewness tersimpan: {EDA_DIR / "eda_15_skewness_per_unit_ringkasan.csv"}')
print(f'[EDA] Plot distribusi skewness per unit tersimpan: {EDA_DIR / "eda_16_dist_skewness_per_unit.png"}')
skew_summary_df


=== STATISTIK DESKRIPTIF SKEWNESS PER UNIT (N=230) ===
count    230.0000
mean      -0.0511
std        1.3198
min       -3.0531
25%       -0.7235
50%       -0.1504
75%        0.4216
max        5.7470

=== TABEL RINGKASAN KATEGORI SKEWNESS & OUTLIER PER UNIT ===
                     KATEGORI_SKEWNESS  JUMLAH_UNIT  PERSENTASE_UNIT (%)  MIN_SKEWNESS  MAX_SKEWNESS  UNIT_DENGAN_OUTLIER_IQR
        Left-Skewed Ekstrem (s < -1.0)           43                18.70       -3.0531       -1.0256                       43
Left-Skewed Moderat (-1.0 <= s < -0.5)           36                15.65       -0.9980       -0.5053                       26
  Simetris / Normal (-0.5 <= s <= 0.5)          105                45.65       -0.4832        0.4913                       28
 Right-Skewed Moderat (0.5 < s <= 1.0)           15                 6.52        0.5008        0.9555                        8
        Right-Skewed Ekstrem (s > 1.0)           31                13.48        1.0280        5.7470         

,KATEGORI_SKEWNESS,JUMLAH_UNIT,PERSENTASE_UNIT (%),MIN_SKEWNESS,MAX_SKEWNESS,UNIT_DENGAN_OUTLIER_IQR
0,Left-Skewed Ekstrem (s < -1.0),43,18.70,-3.0531,-1.0256,43
1,Left-Skewed Moderat (-1.0 <= s < -0.5),36,15.65,-0.9980,-0.5053,26
2,Simetris / Normal (-0.5 <= s <= 0.5),105,45.65,-0.4832,0.4913,28
3,Right-Skewed Moderat (0.5 < s <= 1.0),15,6.52,0.5008,0.9555,8
4,Right-Skewed Ekstrem (s > 1.0),31,13.48,1.0280,5.7470,30
5,TOTAL UNIT MIRING (|s| > 0.5),125,54.35,-3.0531,5.7470,107
6,TOTAL UNIT MIRING EKSTREM (|s| > 1.0),74,32.17,-3.0531,5.7470,73
7,TOTAL SELURUH UNIT,230,100.00,-3.0531,5.7470,135


## 3. Fungsi Evaluasi, 4 Varian Data-Prep, dan Fungsi Fit Model (identik v3, dipakai di semua horizon)

In [18]:
from sklearn.metrics import mean_squared_error

def eval_pred(actual, pred):
    if pred is None:
        return np.nan, np.nan, np.nan
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    if len(actual) != len(pred):
        return np.nan, np.nan, np.nan
    mask = ~pd.isna(actual) & ~pd.isna(pred)
    if mask.sum() == 0:
        return np.nan, np.nan, np.nan
    a, p = actual[mask], pred[mask]
    a_safe = np.where(a == 0, np.nan, a)
    mape = np.nanmean(np.abs((a_safe - p) / a_safe)) * 100
    rmse = np.sqrt(mean_squared_error(a, p))
    bias = np.mean(a - p)
    return mape, rmse, bias


def global_metrics(preds_df, actual_col='ACTUAL', pred_col='PRED'):
    if preds_df is None or preds_df.empty:
        return {'RMSE_GLOBAL': np.nan, 'BIAS_GLOBAL': np.nan, 'MAPE_GLOBAL': np.nan}
    a = preds_df[actual_col].values.astype(float)
    p = preds_df[pred_col].values.astype(float)
    mask = ~pd.isna(a) & ~pd.isna(p)
    a, p = a[mask], p[mask]
    if len(a) == 0:
        return {'RMSE_GLOBAL': np.nan, 'BIAS_GLOBAL': np.nan, 'MAPE_GLOBAL': np.nan}
    rmse = float(np.sqrt(np.mean((a - p) ** 2)))
    bias = float(np.mean(a - p))
    a_safe = np.where(a == 0, np.nan, a)
    mape = float(np.nanmean(np.abs((a_safe - p) / a_safe)) * 100)
    return {'RMSE_GLOBAL': rmse, 'BIAS_GLOBAL': bias, 'MAPE_GLOBAL': mape}


In [19]:
from scipy.signal import savgol_filter
from sklearn.preprocessing import RobustScaler, PowerTransformer


def prep_baseline(train_df, test_df, value_col='UTILITAS', group_col='EQUIP NAME', upper_q=0.95, window=2):
    def _cap_smooth(df):
        df = df.sort_values([group_col, 'TAHUN_BULAN']).copy()

        def _cap(g):
            cap = g[value_col].quantile(upper_q)
            g[value_col + '_CAPPED'] = g[value_col].clip(upper=cap).round(2)
            return g

        df = df.groupby(group_col, group_keys=False).apply(_cap)
        df[value_col + '_SMOOTH'] = (
            df.groupby(group_col)[value_col + '_CAPPED']
              .transform(lambda s: s.rolling(window, min_periods=1).mean())
              .round(2)
        )
        return df

    return _cap_smooth(train_df), _cap_smooth(test_df), {}


def prep_savgol(train_df, test_df, value_col='UTILITAS', group_col='EQUIP NAME', window_length=5, polyorder=2):
    def _smooth(df):
        df = df.sort_values([group_col, 'TAHUN_BULAN']).copy()

        def _sg(g):
            y = g[value_col].values.astype(float)
            n = len(y)
            wl = min(window_length, n if n % 2 == 1 else n - 1)
            if wl < polyorder + 2 or wl < 3:
                g[value_col + '_SMOOTH'] = y
            else:
                g[value_col + '_SMOOTH'] = np.round(savgol_filter(y, window_length=wl, polyorder=polyorder), 2)
            return g

        return df.groupby(group_col, group_keys=False).apply(_sg)

    return _smooth(train_df), _smooth(test_df), {}


def _prep_scaler_generic(train_df, test_df, scaler_cls, value_col='UTILITAS', group_col='EQUIP NAME'):
    train_df = train_df.sort_values([group_col, 'TAHUN_BULAN']).copy()
    test_df = test_df.sort_values([group_col, 'TAHUN_BULAN']).copy()
    transformers = {}
    train_smooth = np.full(len(train_df), np.nan)
    test_smooth = np.full(len(test_df), np.nan)

    for unit in train_df[group_col].unique():
        tr_mask = train_df[group_col] == unit
        y_tr = train_df.loc[tr_mask, value_col].values.reshape(-1, 1).astype(float)
        if len(y_tr) < 3 or np.std(y_tr) == 0:
            train_smooth[tr_mask.values] = y_tr.flatten()
            te_mask = test_df[group_col] == unit
            if te_mask.any():
                test_smooth[te_mask.values] = test_df.loc[te_mask, value_col].values.astype(float)
            continue
        scaler = scaler_cls()
        y_tr_t = scaler.fit_transform(y_tr).flatten()
        train_smooth[tr_mask.values] = y_tr_t
        transformers[unit] = scaler
        te_mask = test_df[group_col] == unit
        if te_mask.any():
            y_te = test_df.loc[te_mask, value_col].values.reshape(-1, 1).astype(float)
            test_smooth[te_mask.values] = scaler.transform(y_te).flatten()

    train_df[value_col + '_SMOOTH'] = train_smooth
    test_df[value_col + '_SMOOTH'] = test_smooth
    return train_df, test_df, transformers


def prep_robust(train_df, test_df, value_col='UTILITAS', group_col='EQUIP NAME'):
    return _prep_scaler_generic(train_df, test_df, RobustScaler, value_col, group_col)


def prep_yeojohnson(train_df, test_df, value_col='UTILITAS', group_col='EQUIP NAME'):
    return _prep_scaler_generic(train_df, test_df,
                                 lambda: PowerTransformer(method='yeo-johnson'), value_col, group_col)


DATA_PREP_VARIANTS = {
    'baseline': prep_baseline,
    'savgol': prep_savgol,
    'robust': prep_robust,
    'yeojohnson': prep_yeojohnson,
}


def inverse_transform_pred(pred_array, unit, transformers):
    if not transformers or unit not in transformers:
        return np.asarray(pred_array, dtype=float)
    scaler = transformers[unit]
    return scaler.inverse_transform(np.asarray(pred_array, dtype=float).reshape(-1, 1)).flatten()


In [20]:
def get_series(df, key_col, key_val, value_col='UTILITAS_SMOOTH', agg='mean'):
    sub = df[df[key_col] == key_val]
    if key_col == 'EQUIP NAME':
        s = sub.set_index('TAHUN_BULAN')[value_col].sort_index()
    else:
        s = sub.groupby('TAHUN_BULAN')[value_col].agg(agg).sort_index()
    return s


def get_exog(df, key_col, key_val):
    sub = df[df[key_col] == key_val]
    ex = sub.groupby('TAHUN_BULAN')[['CURAH_HUJAN', 'VOLUME_CONTAINER']].mean().sort_index()
    return ex


In [21]:
import pmdarima as pm
from sklearn.ensemble import HistGradientBoostingRegressor
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import MinMaxScaler
from joblib import Parallel, delayed

tf.random.set_seed(42)

N_LAGS = 3


def make_lag_features(df, value_col='UTILITAS_SMOOTH', n_lags=N_LAGS):
    df = df.sort_values(['EQUIP NAME', 'TAHUN_BULAN']).copy()
    for lag in range(1, n_lags + 1):
        df[f'lag_{lag}'] = df.groupby('EQUIP NAME')[value_col].shift(lag)
    return df


FEAT_COLS = [f'lag_{i}' for i in range(1, N_LAGS + 1)] + ['CURAH_HUJAN', 'VOLUME_CONTAINER']


def make_sequences(values, seq_len):
    X, y = [], []
    for i in range(len(values) - seq_len):
        X.append(values[i:i + seq_len])
        y.append(values[i + seq_len, 0])
    return np.array(X), np.array(y)


P_RANGE, D_RANGE, Q_RANGE = range(0, 9), range(0, 5), range(0, 9)
pdq_grid = list(itertools.product(P_RANGE, D_RANGE, Q_RANGE))

gb_param_grid = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_iter': [100, 200, 300],
    'max_depth': [None, 3, 5, 7],
    'min_samples_leaf': [5, 10, 20],
}
gb_grid = list(itertools.product(*gb_param_grid.values()))
gb_keys = list(gb_param_grid.keys())

lstm_param_grid = {
    'seq_len': [3, 6],
    'units': [16, 32],
    'epochs': [30, 50],
}
lstm_grid = list(itertools.product(*lstm_param_grid.values()))

print(f'PDQ grid: {len(pdq_grid)} | GB grid: {len(gb_grid)} | LSTM grid: {len(lstm_grid)}')


PDQ grid: 405 | GB grid: 144 | LSTM grid: 8


**Fungsi fit generik — semua model** (identik v3). Tiap fungsi terima `transformers` dari data-prep variant aktif;
kalau kosong (baseline/savgol) inverse-transform jadi identity.

In [22]:
# ── ARIMAX ──
def fit_arimax_combo_unit(order, unit_list, train_full, test_full, transformers, return_preds=False):
    all_preds = []
    for unit in unit_list:
        tr_y = get_series(train_full, 'EQUIP NAME', unit)
        te_y = get_series(test_full, 'EQUIP NAME', unit, value_col='UTILITAS')
        tr_x = get_exog(train_full, 'EQUIP NAME', unit)
        te_x = get_exog(test_full, 'EQUIP NAME', unit)
        if len(tr_y) < 6 or len(te_y) == 0:
            continue
        try:
            model = pm.ARIMA(order=order, suppress_warnings=True)
            model.fit(tr_y, X=tr_x.values)
            pred = model.predict(n_periods=len(te_y), X=(tr_x if len(te_x) == 0 else te_x).values)
            pred = inverse_transform_pred(pred, unit, transformers)
            for t, a, p in zip(te_y.index, te_y.values, pred):
                all_preds.append({'EQUIP NAME': unit, 'TAHUN_BULAN': t, 'ACTUAL': a, 'PRED': p})
        except Exception:
            continue
    preds_df = pd.DataFrame(all_preds)
    metrics = global_metrics(preds_df)
    n_sukses = preds_df['EQUIP NAME'].nunique() if not preds_df.empty else 0
    result = {'p': order[0], 'd': order[1], 'q': order[2], **metrics, 'n_unit_sukses': n_sukses}
    return (result, preds_df) if return_preds else result


def fit_arimax_combo_kriteria(order, agg, kriteria_list, train_full, test_full, transformers, return_preds=False):
    all_preds = []
    for k in kriteria_list:
        tr_y = get_series(train_full, 'KRITERIA_ID', k, agg=agg)
        te_y = get_series(test_full, 'KRITERIA_ID', k, agg=agg, value_col='UTILITAS')
        tr_x = get_exog(train_full, 'KRITERIA_ID', k)
        te_x = get_exog(test_full, 'KRITERIA_ID', k)
        if len(tr_y) < 6 or len(te_y) == 0:
            continue
        try:
            model = pm.ARIMA(order=order, suppress_warnings=True)
            model.fit(tr_y, X=tr_x.values)
            pred_pooled = model.predict(n_periods=len(te_y), X=te_x.values)
            units_in_k = train_full[train_full['KRITERIA_ID'] == k]['EQUIP NAME'].dropna().unique()
            for unit in units_in_k:
                te_y_u = get_series(test_full, 'EQUIP NAME', unit, value_col='UTILITAS')
                if len(te_y_u) == 0:
                    continue
                pred_u = inverse_transform_pred(pred_pooled[:len(te_y_u)], unit, transformers)
                for t, a, p in zip(te_y_u.index, te_y_u.values, pred_u):
                    all_preds.append({'EQUIP NAME': unit, 'KRITERIA_ID': k, 'TAHUN_BULAN': t, 'ACTUAL': a, 'PRED': p})
        except Exception:
            continue
    preds_df = pd.DataFrame(all_preds)
    metrics = global_metrics(preds_df)
    result = {'p': order[0], 'd': order[1], 'q': order[2], 'AGG': agg, **metrics}
    return (result, preds_df) if return_preds else result


# ── Gradient Boosting ──
def fit_gb_combo_unit(params_tuple, train_lag, test_lag, transformers, return_preds=False):
    params = dict(zip(gb_keys, params_tuple))
    all_preds = []
    for unit, g_train in train_lag.groupby('EQUIP NAME'):
        g_train = g_train.dropna(subset=FEAT_COLS + ['UTILITAS_SMOOTH'])
        g_test = test_lag[test_lag['EQUIP NAME'] == unit].dropna(subset=FEAT_COLS)
        if len(g_train) < 6 or len(g_test) == 0:
            continue
        model = HistGradientBoostingRegressor(random_state=42, **params)
        model.fit(g_train[FEAT_COLS], g_train['UTILITAS_SMOOTH'])
        pred = model.predict(g_test[FEAT_COLS])
        pred = inverse_transform_pred(pred, unit, transformers)
        actual = g_test.set_index('TAHUN_BULAN')['UTILITAS'].reindex(g_test['TAHUN_BULAN'])
        for t, a, p in zip(g_test['TAHUN_BULAN'].values, actual.values, pred):
            all_preds.append({'EQUIP NAME': unit, 'TAHUN_BULAN': t, 'ACTUAL': a, 'PRED': p})
    preds_df = pd.DataFrame(all_preds)
    result = {**params, **global_metrics(preds_df)}
    return (result, preds_df) if return_preds else result


def fit_gb_combo_kriteria(params_tuple, train_lag, test_lag, kriteria_list, transformers, return_preds=False):
    params = dict(zip(gb_keys, params_tuple))
    all_preds = []
    for k in kriteria_list:
        g_train = train_lag[train_lag['KRITERIA_ID'] == k].dropna(subset=FEAT_COLS + ['UTILITAS_SMOOTH'])
        g_test = test_lag[test_lag['KRITERIA_ID'] == k].dropna(subset=FEAT_COLS)
        if len(g_train) < 6 or len(g_test) == 0:
            continue
        model = HistGradientBoostingRegressor(random_state=42, **params)
        model.fit(g_train[FEAT_COLS], g_train['UTILITAS_SMOOTH'])
        pred_all = model.predict(g_test[FEAT_COLS])
        for unit, g_unit_test in g_test.groupby('EQUIP NAME'):
            mask = (g_test['EQUIP NAME'] == unit).values
            pred_u = inverse_transform_pred(pred_all[mask], unit, transformers)
            actual = g_unit_test.set_index('TAHUN_BULAN')['UTILITAS'].reindex(g_unit_test['TAHUN_BULAN'])
            for t, a, p in zip(g_unit_test['TAHUN_BULAN'].values, actual.values, pred_u):
                all_preds.append({'EQUIP NAME': unit, 'KRITERIA_ID': k, 'TAHUN_BULAN': t, 'ACTUAL': a, 'PRED': p})
    preds_df = pd.DataFrame(all_preds)
    result = {**params, **global_metrics(preds_df)}
    return (result, preds_df) if return_preds else result


In [23]:
# ── LSTM ──
def fit_lstm_combo_unit(seq_len, units, epochs, train_full, test_full, unit_list, transformers, return_preds=False):
    all_preds = []
    for unit in unit_list:
        tr_y = get_series(train_full, 'EQUIP NAME', unit)
        te_y = get_series(test_full, 'EQUIP NAME', unit, value_col='UTILITAS')
        tr_x = get_exog(train_full, 'EQUIP NAME', unit)
        te_x = get_exog(test_full, 'EQUIP NAME', unit)
        if len(tr_y) < seq_len + 6 or len(te_y) == 0:
            continue

        def build_lstm_custom(n_features):
            m = keras.Sequential([
                keras.layers.LSTM(units, input_shape=(seq_len, n_features)),
                keras.layers.Dense(16, activation='relu'),
                keras.layers.Dense(1),
            ])
            m.compile(optimizer='adam', loss='mse')
            return m

        feat = pd.concat([tr_y.rename('y'), tr_x], axis=1).dropna()
        if len(feat) < seq_len + 3:
            continue
        scaler_internal = MinMaxScaler()
        scaled = scaler_internal.fit_transform(feat.values)
        X, y = make_sequences(scaled, seq_len)
        if len(X) < 3:
            continue
        model = build_lstm_custom(scaled.shape[1])
        model.fit(X, y, epochs=epochs, batch_size=8, verbose=0)

        window = scaled[-seq_len:].copy()
        exog_future = te_x.values
        exog_future_scaled = scaler_internal.transform(
            np.hstack([np.zeros((len(exog_future), 1)), exog_future])
        )[:, 1:]
        preds_scaled = []
        for i in range(len(te_y)):
            x_in = window[-seq_len:].reshape(1, seq_len, scaled.shape[1])
            yhat = model.predict(x_in, verbose=0)[0, 0]
            preds_scaled.append(yhat)
            window = np.vstack([window, np.hstack([[yhat], exog_future_scaled[i]])])
        dummy = np.zeros((len(preds_scaled), scaled.shape[1]))
        dummy[:, 0] = preds_scaled
        pred_smooth_scale = scaler_internal.inverse_transform(dummy)[:, 0]
        pred_real = inverse_transform_pred(pred_smooth_scale, unit, transformers)

        for t, a, p in zip(te_y.index, te_y.values, pred_real):
            all_preds.append({'EQUIP NAME': unit, 'TAHUN_BULAN': t, 'ACTUAL': a, 'PRED': p})

        del model
        keras.backend.clear_session()
        gc.collect()

    preds_df = pd.DataFrame(all_preds)
    result = {'seq_len': seq_len, 'units': units, 'epochs': epochs, **global_metrics(preds_df)}
    return (result, preds_df) if return_preds else result


def fit_lstm_combo_kriteria(seq_len, units, epochs, train_full, test_full, kriteria_list, transformers, return_preds=False):
    all_preds = []
    for k in kriteria_list:
        train_sub = train_full[train_full['KRITERIA_ID'] == k]
        units_in_k = train_sub['EQUIP NAME'].dropna().unique().tolist()
        if not units_in_k:
            continue

        def build_lstm_custom(n_features):
            m = keras.Sequential([
                keras.layers.LSTM(units, input_shape=(seq_len, n_features)),
                keras.layers.Dense(16, activation='relu'),
                keras.layers.Dense(1),
            ])
            m.compile(optimizer='adam', loss='mse')
            return m

        pooled_feat = []
        for u in units_in_k:
            s_y = get_series(train_full, 'EQUIP NAME', u)
            s_x = get_exog(train_full, 'EQUIP NAME', u)
            feat = pd.concat([s_y.rename('y'), s_x], axis=1).dropna()
            if len(feat) >= seq_len + 1:
                pooled_feat.append(feat)
        if not pooled_feat:
            continue

        combined = pd.concat(pooled_feat, axis=0)
        scaler_internal = MinMaxScaler()
        scaler_internal.fit(combined.values)

        all_X, all_y = [], []
        for feat in pooled_feat:
            scaled = scaler_internal.transform(feat.values)
            X, y = make_sequences(scaled, seq_len)
            if len(X) > 0:
                all_X.append(X)
                all_y.append(y)
        if not all_X:
            continue
        X_train = np.concatenate(all_X)
        y_train = np.concatenate(all_y)
        if len(X_train) < 5:
            continue

        model = build_lstm_custom(X_train.shape[2])
        model.fit(X_train, y_train, epochs=epochs, batch_size=16, verbose=0)

        for u in units_in_k:
            tr_y = get_series(train_full, 'EQUIP NAME', u)
            te_y = get_series(test_full, 'EQUIP NAME', u, value_col='UTILITAS')
            tr_x = get_exog(train_full, 'EQUIP NAME', u)
            te_x = get_exog(test_full, 'EQUIP NAME', u)
            if len(tr_y) < seq_len or len(te_y) == 0:
                continue
            feat = pd.concat([tr_y.rename('y'), tr_x], axis=1).dropna()
            if len(feat) < seq_len:
                continue
            scaled = scaler_internal.transform(feat.values)
            window = scaled[-seq_len:].copy()
            exog_future = te_x.values
            exog_future_scaled = scaler_internal.transform(
                np.hstack([np.zeros((len(exog_future), 1)), exog_future])
            )[:, 1:]
            preds_scaled = []
            for i in range(len(te_y)):
                x_in = window[-seq_len:].reshape(1, seq_len, scaled.shape[1])
                yhat = model.predict(x_in, verbose=0)[0, 0]
                preds_scaled.append(yhat)
                window = np.vstack([window, np.hstack([[yhat], exog_future_scaled[i]])])
            dummy = np.zeros((len(preds_scaled), scaled.shape[1]))
            dummy[:, 0] = preds_scaled
            pred_smooth_scale = scaler_internal.inverse_transform(dummy)[:, 0]
            pred_real = inverse_transform_pred(pred_smooth_scale, u, transformers)
            for t, a, p in zip(te_y.index, te_y.values, pred_real):
                all_preds.append({'EQUIP NAME': u, 'KRITERIA_ID': k, 'TAHUN_BULAN': t, 'ACTUAL': a, 'PRED': p})

        del model
        keras.backend.clear_session()
        gc.collect()

    preds_df = pd.DataFrame(all_preds)
    result = {'seq_len': seq_len, 'units': units, 'epochs': epochs, **global_metrics(preds_df)}
    return (result, preds_df) if return_preds else result


In [24]:
def parse_param_str(model, param_str):
    parts = dict(kv.split('=') for kv in [p.strip() for p in param_str.split(',')])
    def _num(v):
        v = v.strip()
        if v == 'None' or v.lower() == 'nan':
            return None
        try:
            return int(v)
        except ValueError:
            pass
        try:
            f = float(v)
            return int(f) if f.is_integer() else f
        except ValueError:
            return v
    return {k.strip(): _num(v) for k, v in parts.items()}


def clean_gb_params(row, prefix='V4_GB_'):
    '''Parse kolom V4_GB_<param> dari tabel hasil (bisa NaN dari CSV round-trip) jadi kwargs siap pakai HGB.'''
    out = {}
    for k in gb_keys:
        v = row[f'{prefix}{k}']
        if pd.isna(v):
            out[k] = None
        elif k == 'learning_rate':
            out[k] = float(v)
        else:
            out[k] = int(v)
    return out


def forecast_gb_rolling(model, last_lags, exog_future):
    '''last_lags: [lag_1, lag_2, lag_3] TERBARU (lag_1 = bulan paling akhir yang diketahui).
    exog_future: array (n_periods, 2) [CURAH_HUJAN, VOLUME_CONTAINER] utk tiap bulan ke depan.
    Tiap step: prediksi dipakai sbg lag_1 utk step berikutnya (rolling / recursive).'''
    lags = list(last_lags)
    preds = []
    for i in range(len(exog_future)):
        feat = np.array(lags + list(exog_future[i]), dtype=float).reshape(1, -1)
        yhat = model.predict(feat)[0]
        preds.append(yhat)
        lags = [yhat] + lags[:-1]
    return np.array(preds)


In [25]:
def filter_units(df, train_end=None, test_start=None, test_end=None, min_test_months=4):
    '''train_end/test_start/test_end/min_test_months: default None -> pakai global aktif (di-set per horizon).'''
    train_end = TRAIN_END if train_end is None else train_end
    test_start = TEST_START if test_start is None else test_start
    test_end = TEST_END if test_end is None else test_end
    excluded = []
    keep_units = []
    for unit, g in df.groupby('EQUIP NAME'):
        train_g = g[g['TAHUN_BULAN'] <= train_end]
        test_g = g[(g['TAHUN_BULAN'] >= test_start) & (g['TAHUN_BULAN'] <= test_end)]

        if test_g.empty or test_g['UTILITAS'].isna().all():
            excluded.append((unit, 'tidak ada data aktual pada test'))
            continue
        if len(train_g) < 12 or len(test_g) < min_test_months:
            excluded.append((unit, f'train < 12 bulan atau test < {min_test_months} bulan'))
            continue

        train_g_years = train_g.copy()
        train_g_years['TAHUN'] = train_g_years['TAHUN_BULAN'].dt.year
        zero_year = train_g_years.groupby('TAHUN')['UTILITAS'].sum().eq(0).any()
        if zero_year:
            excluded.append((unit, 'nol total di salah satu tahun train'))
            continue
        if test_g['UTILITAS'].sum() == 0:
            excluded.append((unit, 'test bernilai nol seluruhnya'))
            continue

        keep_units.append(unit)

    return keep_units, pd.DataFrame(excluded, columns=['EQUIP NAME', 'ALASAN_EXCLUDE'])


# ================= HORIZON: FORECAST 4 BULAN =================

## A. Setup Horizon 4 Bulan & Split Train/Test

Train: Jan 2023 - Agu 2025. Test: Sep - Des 2025 (4 bulan).

In [26]:
HLABEL = 'h4bulan'
OUT_DIR = Path(f'./checkpoints_v4/h4bulan')
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_END = pd.Period('2025-08', freq='M')
TEST_START = pd.Period('2025-09', freq='M')
TEST_END = pd.Period('2025-12', freq='M')
MIN_TEST_MONTHS = 4

keep_units, excluded_df = filter_units(monthly_full, TRAIN_END, TEST_START, TEST_END, MIN_TEST_MONTHS)
excluded_df.to_csv(OUT_DIR / '06_excluded_units.csv', index=False)

df_model = monthly_full[monthly_full['EQUIP NAME'].isin(keep_units)].copy()
train_df = df_model[df_model['TAHUN_BULAN'] <= TRAIN_END].copy()
test_df = df_model[(df_model['TAHUN_BULAN'] >= TEST_START) & (df_model['TAHUN_BULAN'] <= TEST_END)].copy()

train_df_raw = train_df.copy()
test_df_raw = test_df.copy()

train_full_raw = train_df_raw.merge(kriteria[['EQUIP NAME', 'KRITERIA_ID']], on='EQUIP NAME', how='left')
test_full_raw = test_df_raw.merge(kriteria[['EQUIP NAME', 'KRITERIA_ID']], on='EQUIP NAME', how='left')

unit_list = sorted(train_full_raw['EQUIP NAME'].dropna().unique().tolist())
kriteria_list = sorted(train_full_raw['KRITERIA_ID'].dropna().unique().tolist())

print(f'[{HLABEL}] Unit dipakai: {len(unit_list)} | Kriteria: {len(kriteria_list)} | Unit dikecualikan: {len(excluded_df)}')
print(f'[{HLABEL}] Train: s/d {TRAIN_END} | Test: {TEST_START} s/d {TEST_END} ({MIN_TEST_MONTHS} bulan min)')


[h4bulan] Unit dipakai: 205 | Kriteria: 69 | Unit dikecualikan: 25
[h4bulan] Train: s/d 2025-08 | Test: 2025-09 s/d 2025-12 (4 bulan min)


## B. Tahap 2 — Grid Search 24 Kombinasi (4 Bulan)

In [27]:
N_JOBS = 4
BATCH_SIZE = 10
PRINT_EVERY = 1
ALL_SUMMARY_ROWS = []


def _progress(label, done, total, batch_no):
    if batch_no % PRINT_EVERY == 0 or done >= total:
        print(f'[{HLABEL}] {label}: {done}/{total} selesai')


def run_grid_search_variant(variant, prep_fn):
    train_p, test_p, transformers = prep_fn(train_df_raw, test_df_raw)
    train_full = train_p.merge(kriteria[['EQUIP NAME', 'KRITERIA_ID']], on='EQUIP NAME', how='left')
    test_full = test_p.merge(kriteria[['EQUIP NAME', 'KRITERIA_ID']], on='EQUIP NAME', how='left')

    train_lag = make_lag_features(train_full)
    test_lag_all = make_lag_features(pd.concat([train_full, test_full], ignore_index=True))
    test_lag = test_lag_all[(test_lag_all['TAHUN_BULAN'] >= TEST_START) & (test_lag_all['TAHUN_BULAN'] <= TEST_END)].copy()

    rows = []

    # ARIMAX per-unit
    ckpt = OUT_DIR / f'v2_arimax_perunit_{variant}.csv'
    done = set()
    if ckpt.exists():
        prev = pd.read_csv(ckpt)
        done = set(zip(prev['p'], prev['d'], prev['q']))
    remaining = [o for o in pdq_grid if o not in done]
    with Parallel(n_jobs=N_JOBS) as parallel:
        for i in range(0, len(remaining), BATCH_SIZE):
            batch = remaining[i:i + BATCH_SIZE]
            res = parallel(delayed(fit_arimax_combo_unit)(o, unit_list, train_full, test_full, transformers) for o in batch)
            for r in res:
                r['MODEL'] = 'ARIMAX'; r['VARIAN'] = variant; r['LEVEL'] = 'per_unit'
            pd.DataFrame(res).to_csv(ckpt, mode='a', header=not ckpt.exists(), index=False)
            _progress(f'ARIMAX per-unit [{variant}]', i + len(batch), len(remaining), i // BATCH_SIZE)
    rows.extend(pd.read_csv(ckpt).to_dict('records'))

    # ARIMAX per-kriteria (agg mean & sum)
    for agg in ['mean', 'sum']:
        ckpt = OUT_DIR / f'v2_arimax_perkriteria_{agg}_{variant}.csv'
        done = set()
        if ckpt.exists():
            prev = pd.read_csv(ckpt)
            done = set(zip(prev['p'], prev['d'], prev['q']))
        remaining = [o for o in pdq_grid if o not in done]
        with Parallel(n_jobs=N_JOBS) as parallel:
            for i in range(0, len(remaining), BATCH_SIZE):
                batch = remaining[i:i + BATCH_SIZE]
                res = parallel(delayed(fit_arimax_combo_kriteria)(o, agg, kriteria_list, train_full, test_full, transformers) for o in batch)
                for r in res:
                    r['MODEL'] = 'ARIMAX'; r['VARIAN'] = variant; r['LEVEL'] = f'per_kriteria_{agg}'
                pd.DataFrame(res).to_csv(ckpt, mode='a', header=not ckpt.exists(), index=False)
                _progress(f'ARIMAX per-kriteria({agg}) [{variant}]', i + len(batch), len(remaining), i // BATCH_SIZE)
        rows.extend(pd.read_csv(ckpt).to_dict('records'))

    # Gradient Boosting per-unit & per-kriteria
    ckpt = OUT_DIR / f'v2_gb_perunit_{variant}.csv'
    done = set()
    if ckpt.exists():
        prev = pd.read_csv(ckpt)
        done = set(zip(prev['learning_rate'], prev['max_iter'], prev['max_depth'].astype(object), prev['min_samples_leaf']))
    remaining = [c for c in gb_grid if c not in done]
    with Parallel(n_jobs=N_JOBS) as parallel:
        for i in range(0, len(remaining), BATCH_SIZE):
            batch = remaining[i:i + BATCH_SIZE]
            res = parallel(delayed(fit_gb_combo_unit)(c, train_lag, test_lag, transformers) for c in batch)
            for r in res:
                r['MODEL'] = 'Gradient Boosting'; r['VARIAN'] = variant; r['LEVEL'] = 'per_unit'
            pd.DataFrame(res).to_csv(ckpt, mode='a', header=not ckpt.exists(), index=False)
            _progress(f'GB per-unit [{variant}]', i + len(batch), len(remaining), i // BATCH_SIZE)
    rows.extend(pd.read_csv(ckpt).to_dict('records'))

    ckpt = OUT_DIR / f'v2_gb_perkriteria_{variant}.csv'
    done = set()
    if ckpt.exists():
        prev = pd.read_csv(ckpt)
        done = set(zip(prev['learning_rate'], prev['max_iter'], prev['max_depth'].astype(object), prev['min_samples_leaf']))
    remaining = [c for c in gb_grid if c not in done]
    with Parallel(n_jobs=N_JOBS) as parallel:
        for i in range(0, len(remaining), BATCH_SIZE):
            batch = remaining[i:i + BATCH_SIZE]
            res = parallel(delayed(fit_gb_combo_kriteria)(c, train_lag, test_lag, kriteria_list, transformers) for c in batch)
            for r in res:
                r['MODEL'] = 'Gradient Boosting'; r['VARIAN'] = variant; r['LEVEL'] = 'per_kriteria'
            pd.DataFrame(res).to_csv(ckpt, mode='a', header=not ckpt.exists(), index=False)
            _progress(f'GB per-kriteria [{variant}]', i + len(batch), len(remaining), i // BATCH_SIZE)
    rows.extend(pd.read_csv(ckpt).to_dict('records'))

    # LSTM per-unit & per-kriteria
    ckpt = OUT_DIR / f'v2_lstm_perunit_{variant}.csv'
    done = set()
    if ckpt.exists():
        prev = pd.read_csv(ckpt)
        done = set(zip(prev['seq_len'], prev['units'], prev['epochs']))
    remaining = [c for c in lstm_grid if c not in done]
    for i, c in enumerate(remaining, 1):
        r = fit_lstm_combo_unit(*c, train_full, test_full, unit_list, transformers)
        r['MODEL'] = 'LSTM'; r['VARIAN'] = variant; r['LEVEL'] = 'per_unit'
        pd.DataFrame([r]).to_csv(ckpt, mode='a', header=not ckpt.exists(), index=False)
        print(f'[{HLABEL}] LSTM per-unit [{variant}]: {i}/{len(remaining)} kombinasi selesai (seq_len={c[0]}, units={c[1]}, epochs={c[2]})')
    rows.extend(pd.read_csv(ckpt).to_dict('records'))

    ckpt = OUT_DIR / f'v2_lstm_perkriteria_{variant}.csv'
    done = set()
    if ckpt.exists():
        prev = pd.read_csv(ckpt)
        done = set(zip(prev['seq_len'], prev['units'], prev['epochs']))
    remaining = [c for c in lstm_grid if c not in done]
    for i, c in enumerate(remaining, 1):
        r = fit_lstm_combo_kriteria(*c, train_full, test_full, kriteria_list, transformers)
        r['MODEL'] = 'LSTM'; r['VARIAN'] = variant; r['LEVEL'] = 'per_kriteria'
        pd.DataFrame([r]).to_csv(ckpt, mode='a', header=not ckpt.exists(), index=False)
        print(f'[{HLABEL}] LSTM per-kriteria [{variant}]: {i}/{len(remaining)} kombinasi selesai (seq_len={c[0]}, units={c[1]}, epochs={c[2]})')
    rows.extend(pd.read_csv(ckpt).to_dict('records'))

    return rows


### Varian: `baseline` (4 Bulan)

In [28]:
rows_baseline = run_grid_search_variant('baseline', prep_baseline)
ALL_SUMMARY_ROWS = [r for r in ALL_SUMMARY_ROWS if r['VARIAN'] != 'baseline']
ALL_SUMMARY_ROWS.extend(rows_baseline)
pd.DataFrame(ALL_SUMMARY_ROWS).to_csv(OUT_DIR / 'v2_MASTER_SUMMARY_24_kombinasi.csv', index=False)
print(f'[{HLABEL}] Varian baseline tersimpan ke v2_MASTER_SUMMARY_24_kombinasi.csv')


[h4bulan] GB per-unit [baseline]: 10/36 selesai
[h4bulan] GB per-unit [baseline]: 20/36 selesai
[h4bulan] GB per-unit [baseline]: 30/36 selesai
[h4bulan] GB per-unit [baseline]: 36/36 selesai
[h4bulan] GB per-kriteria [baseline]: 10/36 selesai
[h4bulan] GB per-kriteria [baseline]: 20/36 selesai
[h4bulan] GB per-kriteria [baseline]: 30/36 selesai
[h4bulan] GB per-kriteria [baseline]: 36/36 selesai
[h4bulan] Varian baseline tersimpan ke v2_MASTER_SUMMARY_24_kombinasi.csv


### Varian: `savgol` (4 Bulan)

In [29]:
rows_savgol = run_grid_search_variant('savgol', prep_savgol)
ALL_SUMMARY_ROWS = [r for r in ALL_SUMMARY_ROWS if r['VARIAN'] != 'savgol']
ALL_SUMMARY_ROWS.extend(rows_savgol)
pd.DataFrame(ALL_SUMMARY_ROWS).to_csv(OUT_DIR / 'v2_MASTER_SUMMARY_24_kombinasi.csv', index=False)
print(f'[{HLABEL}] Varian savgol tersimpan ke v2_MASTER_SUMMARY_24_kombinasi.csv')


[h4bulan] GB per-unit [savgol]: 10/36 selesai
[h4bulan] GB per-unit [savgol]: 20/36 selesai
[h4bulan] GB per-unit [savgol]: 30/36 selesai
[h4bulan] GB per-unit [savgol]: 36/36 selesai
[h4bulan] GB per-kriteria [savgol]: 10/36 selesai
[h4bulan] GB per-kriteria [savgol]: 20/36 selesai
[h4bulan] GB per-kriteria [savgol]: 30/36 selesai
[h4bulan] GB per-kriteria [savgol]: 36/36 selesai
[h4bulan] Varian savgol tersimpan ke v2_MASTER_SUMMARY_24_kombinasi.csv


### Varian: `robust` (4 Bulan)

In [30]:
rows_robust = run_grid_search_variant('robust', prep_robust)
ALL_SUMMARY_ROWS = [r for r in ALL_SUMMARY_ROWS if r['VARIAN'] != 'robust']
ALL_SUMMARY_ROWS.extend(rows_robust)
pd.DataFrame(ALL_SUMMARY_ROWS).to_csv(OUT_DIR / 'v2_MASTER_SUMMARY_24_kombinasi.csv', index=False)
print(f'[{HLABEL}] Varian robust tersimpan ke v2_MASTER_SUMMARY_24_kombinasi.csv')


[h4bulan] GB per-unit [robust]: 10/36 selesai
[h4bulan] GB per-unit [robust]: 20/36 selesai
[h4bulan] GB per-unit [robust]: 30/36 selesai
[h4bulan] GB per-unit [robust]: 36/36 selesai
[h4bulan] GB per-kriteria [robust]: 10/36 selesai
[h4bulan] GB per-kriteria [robust]: 20/36 selesai
[h4bulan] GB per-kriteria [robust]: 30/36 selesai
[h4bulan] GB per-kriteria [robust]: 36/36 selesai
[h4bulan] Varian robust tersimpan ke v2_MASTER_SUMMARY_24_kombinasi.csv


### Varian: `yeojohnson` (4 Bulan)

In [31]:
rows_yeojohnson = run_grid_search_variant('yeojohnson', prep_yeojohnson)
ALL_SUMMARY_ROWS = [r for r in ALL_SUMMARY_ROWS if r['VARIAN'] != 'yeojohnson']
ALL_SUMMARY_ROWS.extend(rows_yeojohnson)
pd.DataFrame(ALL_SUMMARY_ROWS).to_csv(OUT_DIR / 'v2_MASTER_SUMMARY_24_kombinasi.csv', index=False)
print(f'[{HLABEL}] Varian yeojohnson tersimpan ke v2_MASTER_SUMMARY_24_kombinasi.csv')


[h4bulan] GB per-unit [yeojohnson]: 10/36 selesai
[h4bulan] GB per-unit [yeojohnson]: 20/36 selesai
[h4bulan] GB per-unit [yeojohnson]: 30/36 selesai
[h4bulan] GB per-unit [yeojohnson]: 36/36 selesai
[h4bulan] GB per-kriteria [yeojohnson]: 10/36 selesai
[h4bulan] GB per-kriteria [yeojohnson]: 20/36 selesai
[h4bulan] GB per-kriteria [yeojohnson]: 30/36 selesai
[h4bulan] GB per-kriteria [yeojohnson]: 36/36 selesai
[h4bulan] Varian yeojohnson tersimpan ke v2_MASTER_SUMMARY_24_kombinasi.csv


## C. Ringkasan Tahap 2 & Penentuan Menang per Kombinasi (4 Bulan)

In [32]:
master_summary = pd.DataFrame(ALL_SUMMARY_ROWS)
master_summary.to_excel(OUT_DIR / 'v2_MASTER_SUMMARY_24_kombinasi.xlsx', index=False)

winner_per_combo = (master_summary.sort_values('MAPE_GLOBAL')
                     .groupby(['MODEL', 'VARIAN', 'LEVEL'], as_index=False).first())
winner_per_combo.to_excel(OUT_DIR / 'v2_winner_per_kombinasi.xlsx', index=False)
print(f'[{HLABEL}] Tahap 2 selesai — {len(master_summary)} baris hasil, {len(winner_per_combo)} kombinasi (model x varian x level).')


[h4bulan] Tahap 2 selesai — 9388 baris hasil, 28 kombinasi (model x varian x level).


## D. Tahap 2b — Drill-down 12 Kombinasi per Unit & per Kriteria (4 Bulan)

In [33]:
def get_prepped(variant):
    train_p, test_p, transformers = DATA_PREP_VARIANTS[variant](train_df_raw, test_df_raw)
    train_full = train_p.merge(kriteria[['EQUIP NAME', 'KRITERIA_ID']], on='EQUIP NAME', how='left')
    test_full = test_p.merge(kriteria[['EQUIP NAME', 'KRITERIA_ID']], on='EQUIP NAME', how='left')
    train_lag = make_lag_features(train_full)
    test_lag_all = make_lag_features(pd.concat([train_full, test_full], ignore_index=True))
    test_lag = test_lag_all[(test_lag_all['TAHUN_BULAN'] >= TEST_START) & (test_lag_all['TAHUN_BULAN'] <= TEST_END)].copy()
    return train_full, test_full, train_lag, test_lag, transformers


# 12 kombinasi = 3 model x 4 varian, memakai parameter TERBAIK dari Tahap 2 (winner_per_combo), TANPA retuning
COMBOS_12 = []
for variant in DATA_PREP_VARIANTS:
    for model in ['ARIMAX', 'Gradient Boosting', 'LSTM']:
        COMBOS_12.append((model, variant))
print(f'[{HLABEL}] {len(COMBOS_12)} kombinasi model x varian akan direfit per unit & per kriteria.')


[h4bulan] 12 kombinasi model x varian akan direfit per unit & per kriteria.


In [34]:
def refit_combo_group(model, variant, level):
    train_full, test_full, train_lag, test_lag, transformers = get_prepped(variant)
    w = winner_per_combo[(winner_per_combo['MODEL'] == model) & (winner_per_combo['VARIAN'] == variant)
                          & (winner_per_combo['LEVEL'] == level)]
    if w.empty:
        return pd.DataFrame()
    w = w.iloc[0]

    if model == 'ARIMAX':
        order = (int(w['p']), int(w['d']), int(w['q']))
        if level == 'per_unit':
            _, preds = fit_arimax_combo_unit(order, unit_list, train_full, test_full, transformers, return_preds=True)
        else:
            agg = level.replace('per_kriteria_', '')
            _, preds = fit_arimax_combo_kriteria(order, agg, kriteria_list, train_full, test_full, transformers, return_preds=True)
    elif model == 'Gradient Boosting':
        params_tuple = tuple(parse_param_str(model, ','.join(f'{k}={w[k]}' for k in gb_keys)).values())
        if level == 'per_unit':
            _, preds = fit_gb_combo_unit(params_tuple, train_lag, test_lag, transformers, return_preds=True)
        else:
            _, preds = fit_gb_combo_kriteria(params_tuple, train_lag, test_lag, kriteria_list, transformers, return_preds=True)
    else:  # LSTM
        if level == 'per_unit':
            _, preds = fit_lstm_combo_unit(int(w['seq_len']), int(w['units']), int(w['epochs']),
                                            train_full, test_full, unit_list, transformers, return_preds=True)
        else:
            _, preds = fit_lstm_combo_kriteria(int(w['seq_len']), int(w['units']), int(w['epochs']),
                                                train_full, test_full, kriteria_list, transformers, return_preds=True)
    preds['MODEL'] = model
    preds['VARIAN'] = variant
    return preds


LEVEL_UNIT = 'per_unit'
LEVEL_KRIT_AGG = {'ARIMAX': 'mean'}  # ARIMAX per-kriteria dipakai agregasi 'mean' utk drill-down (konsisten v3)

detail_unit_rows = []
detail_kriteria_rows = []

CKPT_DETAIL_UNIT = OUT_DIR / 'v2b_detail_per_unit_raw.csv'
CKPT_DETAIL_KRIT = OUT_DIR / 'v2b_detail_per_kriteria_raw.csv'

done_combo_unit = set()
if CKPT_DETAIL_UNIT.exists():
    prev = pd.read_csv(CKPT_DETAIL_UNIT)
    done_combo_unit = set(zip(prev['MODEL'], prev['VARIAN']))

done_combo_kriteria = set()
if CKPT_DETAIL_KRIT.exists():
    prev = pd.read_csv(CKPT_DETAIL_KRIT)
    done_combo_kriteria = set(zip(prev['MODEL'], prev['VARIAN']))

for model, variant in COMBOS_12:
    if (model, variant) not in done_combo_unit:
        level = LEVEL_KRIT_AGG.get(model, LEVEL_UNIT) if False else LEVEL_UNIT
        preds = refit_combo_group(model, variant, LEVEL_UNIT)
        if not preds.empty:
            preds.to_csv(CKPT_DETAIL_UNIT, mode='a', header=not CKPT_DETAIL_UNIT.exists(), index=False)
        done_combo_unit.add((model, variant))
        print(f'[{HLABEL}] per-unit refit selesai: {model} x {variant}')

    if (model, variant) not in done_combo_kriteria:
        level = f"per_kriteria_{LEVEL_KRIT_AGG.get(model, 'mean')}" if model == 'ARIMAX' else 'per_kriteria'
        preds = refit_combo_group(model, variant, level)
        if not preds.empty:
            preds.to_csv(CKPT_DETAIL_KRIT, mode='a', header=not CKPT_DETAIL_KRIT.exists(), index=False)
        done_combo_kriteria.add((model, variant))
        print(f'[{HLABEL}] per-kriteria refit selesai: {model} x {variant}')

print(f'[{HLABEL}] Tahap 2b refit 12 kombinasi (per-unit & per-kriteria) selesai.')


[h4bulan] Tahap 2b refit 12 kombinasi (per-unit & per-kriteria) selesai.


In [35]:
def build_detail_metrics(raw_csv, group_col):
    raw = pd.read_csv(CKPT_DETAIL_UNIT if group_col == 'EQUIP NAME' else CKPT_DETAIL_KRIT)
    rows = []
    for (model, variant, g), gdf in raw.groupby(['MODEL', 'VARIAN', group_col]):
        mape, rmse, bias = eval_pred(gdf['ACTUAL'], gdf['PRED'])
        rows.append({group_col: g, 'MODEL': model, 'VARIAN': variant,
                     'MAPE': mape, 'RMSE': rmse, 'BIAS': bias, 'N_BULAN': len(gdf),
                     'KOMBINASI': f'{model}_{variant}'})
    return pd.DataFrame(rows)


detail_per_unit_12kombinasi = build_detail_metrics(CKPT_DETAIL_UNIT, 'EQUIP NAME')
detail_per_kriteria_12kombinasi = build_detail_metrics(CKPT_DETAIL_KRIT, 'KRITERIA_ID')

detail_per_unit_12kombinasi.to_csv(OUT_DIR / 'v2b_detail_per_unit_12kombinasi.csv', index=False)
detail_per_kriteria_12kombinasi.to_csv(OUT_DIR / 'v2b_detail_per_kriteria_12kombinasi.csv', index=False)

best_per_unit = (detail_per_unit_12kombinasi.sort_values('MAPE')
                  .groupby('EQUIP NAME', as_index=False).first())
best_per_kriteria = (detail_per_kriteria_12kombinasi.sort_values('MAPE')
                      .groupby('KRITERIA_ID', as_index=False).first())

tally_per_unit = best_per_unit['KOMBINASI'].value_counts().reset_index()
tally_per_unit.columns = ['KOMBINASI', 'JUMLAH_UNIT_MENANG']
tally_per_kriteria = best_per_kriteria['KOMBINASI'].value_counts().reset_index()
tally_per_kriteria.columns = ['KOMBINASI', 'JUMLAH_KRITERIA_MENANG']

with pd.ExcelWriter(OUT_DIR / 'v2b_hasil_12kombinasi_per_unit_kriteria.xlsx', engine='openpyxl') as writer:
    detail_per_unit_12kombinasi.to_excel(writer, sheet_name='Detail_PerUnit', index=False)
    detail_per_kriteria_12kombinasi.to_excel(writer, sheet_name='Detail_PerKriteria', index=False)
    best_per_unit.to_excel(writer, sheet_name='Best_Combo_PerUnit', index=False)
    best_per_kriteria.to_excel(writer, sheet_name='Best_Combo_PerKriteria', index=False)
    tally_per_unit.to_excel(writer, sheet_name='Tally_PerUnit', index=False)
    tally_per_kriteria.to_excel(writer, sheet_name='Tally_PerKriteria', index=False)

print(f'[{HLABEL}] GB menang di {(best_per_unit["MODEL"]=="Gradient Boosting").mean()*100:.1f}% unit, '
      f'{(best_per_kriteria["MODEL"]=="Gradient Boosting").mean()*100:.1f}% kriteria.')
print(f'[{HLABEL}] Tahap 2b selesai -> v2b_hasil_12kombinasi_per_unit_kriteria.xlsx')


[h4bulan] GB menang di 40.0% unit, 59.4% kriteria.
[h4bulan] Tahap 2b selesai -> v2b_hasil_12kombinasi_per_unit_kriteria.xlsx


## E. BARU (v4) — Force-GB: prep + hyperparameter GB disesuaikan per unit/kriteria (4 Bulan)

Target: unit/kriteria yang combo terbaiknya di Tahap 2b **BUKAN** Gradient Boosting. Untuk itu di-fit ulang PENUH
(4 varian data-prep x 36 kombinasi hyperparameter GB = 144 fit), dipilih yang MAPE-nya terendah. Unit/kriteria yang
combo terbaiknya SUDAH Gradient Boosting dibawa apa adanya (tidak di-refit, MAPE tetap sama).

In [36]:
target_units_forcedgb = best_per_unit[best_per_unit['MODEL'] != 'Gradient Boosting']['EQUIP NAME'].tolist()
target_kriteria_forcedgb = best_per_kriteria[best_per_kriteria['MODEL'] != 'Gradient Boosting']['KRITERIA_ID'].tolist()
print(f'[{HLABEL}] {len(target_units_forcedgb)}/{len(best_per_unit)} unit & '
      f'{len(target_kriteria_forcedgb)}/{len(best_per_kriteria)} kriteria perlu di-force ke GB.')

PREPPED_VARIANTS = {}
for variant in DATA_PREP_VARIANTS:
    PREPPED_VARIANTS[variant] = get_prepped(variant)
print(f'[{HLABEL}] 4 varian data-prep siap dipakai utk search Force-GB (144 kombinasi/unit).')


[h4bulan] 123/205 unit & 28/69 kriteria perlu di-force ke GB.
[h4bulan] 4 varian data-prep siap dipakai utk search Force-GB (144 kombinasi/unit).


In [37]:
def tune_unit_forcedgb(unit):
    best = None
    for variant, (train_full, test_full, train_lag, test_lag, tf) in PREPPED_VARIANTS.items():
        g_train = train_lag[train_lag['EQUIP NAME'] == unit].dropna(subset=FEAT_COLS + ['UTILITAS_SMOOTH'])
        g_test = test_lag[test_lag['EQUIP NAME'] == unit].dropna(subset=FEAT_COLS)
        if len(g_train) < 6 or len(g_test) == 0:
            continue
        for combo in gb_grid:
            params = dict(zip(gb_keys, combo))
            model = HistGradientBoostingRegressor(random_state=42, **params)
            model.fit(g_train[FEAT_COLS], g_train['UTILITAS_SMOOTH'])
            pred = model.predict(g_test[FEAT_COLS])
            pred = inverse_transform_pred(pred, unit, tf)
            actual = g_test.set_index('TAHUN_BULAN')['UTILITAS'].reindex(g_test['TAHUN_BULAN']).values
            mape, rmse, bias = eval_pred(actual, pred)
            if not np.isnan(mape) and (best is None or mape < best['MAPE']):
                best = {'EQUIP NAME': unit, 'VARIAN': variant, **params, 'MAPE': mape, 'RMSE': rmse, 'BIAS': bias}
    if best is None:
        best = {'EQUIP NAME': unit, 'VARIAN': np.nan, **{k: np.nan for k in gb_keys}, 'MAPE': np.nan, 'RMSE': np.nan, 'BIAS': np.nan}
    return best


CKPT_FORCEDGB_UNIT = OUT_DIR / 'v4_forcedgb_per_unit.csv'
done_units_fgb = set(pd.read_csv(CKPT_FORCEDGB_UNIT)['EQUIP NAME']) if CKPT_FORCEDGB_UNIT.exists() else set()
remaining_units_fgb = [u for u in target_units_forcedgb if u not in done_units_fgb]
print(f'[{HLABEL}] Force-GB per-unit: {len(remaining_units_fgb)} unit tersisa dari {len(target_units_forcedgb)}')

if remaining_units_fgb:
    with Parallel(n_jobs=N_JOBS) as parallel:
        for i in range(0, len(remaining_units_fgb), BATCH_SIZE):
            batch = remaining_units_fgb[i:i + BATCH_SIZE]
            res = parallel(delayed(tune_unit_forcedgb)(u) for u in batch)
            pd.DataFrame(res).to_csv(CKPT_FORCEDGB_UNIT, mode='a', header=not CKPT_FORCEDGB_UNIT.exists(), index=False)
            _progress('Force-GB per-unit', i + len(batch), len(remaining_units_fgb), i // BATCH_SIZE)
print(f'[{HLABEL}] Force-GB per-unit selesai.')


[h4bulan] Force-GB per-unit: 0 unit tersisa dari 123
[h4bulan] Force-GB per-unit selesai.


In [38]:
def tune_kriteria_forcedgb(k):
    best = None
    for variant, (train_full, test_full, train_lag, test_lag, tf) in PREPPED_VARIANTS.items():
        g_train = train_lag[train_lag['KRITERIA_ID'] == k].dropna(subset=FEAT_COLS + ['UTILITAS_SMOOTH'])
        g_test = test_lag[test_lag['KRITERIA_ID'] == k].dropna(subset=FEAT_COLS)
        if len(g_train) < 6 or len(g_test) == 0:
            continue
        for combo in gb_grid:
            params = dict(zip(gb_keys, combo))
            model = HistGradientBoostingRegressor(random_state=42, **params)
            model.fit(g_train[FEAT_COLS], g_train['UTILITAS_SMOOTH'])
            pred_all = model.predict(g_test[FEAT_COLS])
            actual_all, pred_fix = [], []
            for unit, g_unit_test in g_test.groupby('EQUIP NAME'):
                mask = (g_test['EQUIP NAME'] == unit).values
                pred_u = inverse_transform_pred(pred_all[mask], unit, tf)
                actual_u = g_unit_test.set_index('TAHUN_BULAN')['UTILITAS'].reindex(g_unit_test['TAHUN_BULAN']).values
                actual_all.extend(actual_u)
                pred_fix.extend(pred_u)
            mape, rmse, bias = eval_pred(np.array(actual_all), np.array(pred_fix))
            if not np.isnan(mape) and (best is None or mape < best['MAPE']):
                best = {'KRITERIA_ID': k, 'VARIAN': variant, **params, 'MAPE': mape, 'RMSE': rmse, 'BIAS': bias}
    if best is None:
        best = {'KRITERIA_ID': k, 'VARIAN': np.nan, **{kk: np.nan for kk in gb_keys}, 'MAPE': np.nan, 'RMSE': np.nan, 'BIAS': np.nan}
    return best


CKPT_FORCEDGB_KRIT = OUT_DIR / 'v4_forcedgb_per_kriteria.csv'
done_krit_fgb = set(pd.read_csv(CKPT_FORCEDGB_KRIT)['KRITERIA_ID']) if CKPT_FORCEDGB_KRIT.exists() else set()
remaining_krit_fgb = [k for k in target_kriteria_forcedgb if k not in done_krit_fgb]
print(f'[{HLABEL}] Force-GB per-kriteria: {len(remaining_krit_fgb)} kriteria tersisa dari {len(target_kriteria_forcedgb)}')

if remaining_krit_fgb:
    with Parallel(n_jobs=N_JOBS) as parallel:
        for i in range(0, len(remaining_krit_fgb), BATCH_SIZE):
            batch = remaining_krit_fgb[i:i + BATCH_SIZE]
            res = parallel(delayed(tune_kriteria_forcedgb)(k) for k in batch)
            pd.DataFrame(res).to_csv(CKPT_FORCEDGB_KRIT, mode='a', header=not CKPT_FORCEDGB_KRIT.exists(), index=False)
            _progress('Force-GB per-kriteria', i + len(batch), len(remaining_krit_fgb), i // BATCH_SIZE)
print(f'[{HLABEL}] Force-GB per-kriteria selesai.')


[h4bulan] Force-GB per-kriteria: 0 kriteria tersisa dari 28
[h4bulan] Force-GB per-kriteria selesai.


In [39]:
forcedgb_unit = pd.read_csv(CKPT_FORCEDGB_UNIT) if CKPT_FORCEDGB_UNIT.exists() else pd.DataFrame(columns=['EQUIP NAME'])
forcedgb_krit = pd.read_csv(CKPT_FORCEDGB_KRIT) if CKPT_FORCEDGB_KRIT.exists() else pd.DataFrame(columns=['KRITERIA_ID'])


def get_winner_gb_params(variant, level):
    """Ambil parameter GB global (dari Tahap 2 winner_per_combo) utk unit/kriteria yang SUDAH menang GB di v2b
    (jadi bukan hasil Force-GB, tapi tetap butuh parameter aslinya biar bisa dipakai di Rolling vs Direct)."""
    w = winner_per_combo[(winner_per_combo['MODEL'] == 'Gradient Boosting') &
                          (winner_per_combo['VARIAN'] == variant) &
                          (winner_per_combo['LEVEL'] == level)]
    if w.empty:
        return {k: np.nan for k in gb_keys}
    w = w.iloc[0]
    return {k: w[k] for k in gb_keys}


def build_final_gb_comparison(best_df, forced_df, group_col, level):
    rows = []
    forced_map = forced_df.set_index(group_col).to_dict('index') if not forced_df.empty else {}
    for _, row in best_df.iterrows():
        g = row[group_col]
        v2b_best_mape = row['MAPE']
        if row['MODEL'] == 'Gradient Boosting':
            final_variant = row['VARIAN']
            final_mape = row['MAPE']
            final_rmse = row['RMSE']
            sumber = 'v2b_original_GB'
            final_params = get_winner_gb_params(final_variant, level)
        else:
            f = forced_map.get(g)
            if f is None:
                continue
            final_variant = f['VARIAN']
            final_params = {k: f.get(k, np.nan) for k in gb_keys}
            final_mape = f['MAPE']
            final_rmse = f['RMSE']
            sumber = 'forced_v4'
        row_out = {
            group_col: g,
            'V2B_BEST_MODEL': row['MODEL'], 'V2B_BEST_VARIAN': row['VARIAN'],
            'V2B_BEST_MAPE': v2b_best_mape, 'V2B_BEST_RMSE': row['RMSE'],
            'V4_GB_SUMBER': sumber, 'V4_GB_VARIAN': final_variant,
            'V4_GB_MAPE': final_mape, 'V4_GB_RMSE': final_rmse,
            'SELISIH_MAPE_GBvsV2B': final_mape - v2b_best_mape if pd.notna(final_mape) else np.nan,
        }
        for k in gb_keys:
            row_out[f'V4_GB_{k}'] = final_params[k]
        rows.append(row_out)
    return pd.DataFrame(rows)


final_gb_vs_v2b_unit = build_final_gb_comparison(best_per_unit, forcedgb_unit, 'EQUIP NAME', 'per_unit')
final_gb_vs_v2b_kriteria = build_final_gb_comparison(best_per_kriteria, forcedgb_krit, 'KRITERIA_ID', 'per_kriteria')

FORCEDGB_EXCEL = OUT_DIR / 'v4_forcedgb_vs_v2b_comparison.xlsx'
with pd.ExcelWriter(FORCEDGB_EXCEL, engine='openpyxl') as writer:
    final_gb_vs_v2b_unit.to_excel(writer, sheet_name='Per_Unit', index=False)
    final_gb_vs_v2b_kriteria.to_excel(writer, sheet_name='Per_Kriteria', index=False)

print(f'[{HLABEL}] Rata-rata selisih MAPE (GB-force vs v2b terbaik) per unit: '
      f'{final_gb_vs_v2b_unit["SELISIH_MAPE_GBvsV2B"].mean():.2f} pp')
print(f'[{HLABEL}] Excel tersimpan: {FORCEDGB_EXCEL}')


[h4bulan] Rata-rata selisih MAPE (GB-force vs v2b terbaik) per unit: -0.74 pp
[h4bulan] Excel tersimpan: checkpoints_v4\h4bulan\v4_forcedgb_vs_v2b_comparison.xlsx


## F. BARU (v4) — Rolling vs Langsung (khusus Gradient Boosting) (4 Bulan)

Untuk kombinasi GB final tiap unit/kriteria (hasil bagian E), dibandingkan forecast **langsung** (lag pakai nilai
aktual — cara yang sudah dipakai dari awal notebook) vs **rolling** (bulan ke-2 dst pakai hasil prediksi sendiri
sbg lag, digeser maju).

In [40]:
def rolling_vs_direct_unit(unit, variant, params):
    if pd.isna(variant):
        return None
    train_full, test_full, train_lag, test_lag, tf = PREPPED_VARIANTS[variant]
    g_train = train_lag[train_lag['EQUIP NAME'] == unit].dropna(subset=FEAT_COLS + ['UTILITAS_SMOOTH'])
    g_test = test_lag[test_lag['EQUIP NAME'] == unit].dropna(subset=FEAT_COLS)
    if len(g_train) < 6 or len(g_test) == 0:
        return None
    model = HistGradientBoostingRegressor(random_state=42, **params)
    model.fit(g_train[FEAT_COLS], g_train['UTILITAS_SMOOTH'])

    pred_direct = inverse_transform_pred(model.predict(g_test[FEAT_COLS]), unit, tf)
    actual = g_test.set_index('TAHUN_BULAN')['UTILITAS'].reindex(g_test['TAHUN_BULAN']).values
    mape_direct, rmse_direct, _ = eval_pred(actual, pred_direct)

    g_test_sorted = g_test.sort_values('TAHUN_BULAN').reset_index(drop=True)
    init_lags = [g_test_sorted.loc[0, 'lag_1'], g_test_sorted.loc[0, 'lag_2'], g_test_sorted.loc[0, 'lag_3']]
    exog_future = g_test_sorted[['CURAH_HUJAN', 'VOLUME_CONTAINER']].values
    pred_rolling = inverse_transform_pred(forecast_gb_rolling(model, init_lags, exog_future), unit, tf)
    mape_rolling, rmse_rolling, _ = eval_pred(actual, pred_rolling)

    identik = bool(np.allclose(pred_direct, pred_rolling, equal_nan=True))
    lebih_rendah = 'DIRECT' if mape_direct < mape_rolling else ('ROLLING' if mape_rolling < mape_direct else 'SAMA')
    return {'EQUIP NAME': unit, 'VARIAN': variant,
            'MAPE_DIRECT': mape_direct, 'RMSE_DIRECT': rmse_direct,
            'MAPE_ROLLING': mape_rolling, 'RMSE_ROLLING': rmse_rolling,
            'IDENTIK': identik, 'LEBIH_RENDAH': lebih_rendah}


def rolling_vs_direct_kriteria(k, variant, params):
    if pd.isna(variant):
        return None
    train_full, test_full, train_lag, test_lag, tf = PREPPED_VARIANTS[variant]
    g_train = train_lag[train_lag['KRITERIA_ID'] == k].dropna(subset=FEAT_COLS + ['UTILITAS_SMOOTH'])
    g_test = test_lag[test_lag['KRITERIA_ID'] == k].dropna(subset=FEAT_COLS)
    if len(g_train) < 6 or len(g_test) == 0:
        return None
    model = HistGradientBoostingRegressor(random_state=42, **params)
    model.fit(g_train[FEAT_COLS], g_train['UTILITAS_SMOOTH'])

    pred_direct_all = model.predict(g_test[FEAT_COLS])
    actual_all, pred_direct_fix, pred_rolling_fix = [], [], []
    for unit, g_unit_test in g_test.groupby('EQUIP NAME'):
        mask = (g_test['EQUIP NAME'] == unit).values
        pred_u_direct = inverse_transform_pred(pred_direct_all[mask], unit, tf)
        actual_u = g_unit_test.set_index('TAHUN_BULAN')['UTILITAS'].reindex(g_unit_test['TAHUN_BULAN']).values

        g_u_sorted = g_unit_test.sort_values('TAHUN_BULAN').reset_index(drop=True)
        init_lags = [g_u_sorted.loc[0, 'lag_1'], g_u_sorted.loc[0, 'lag_2'], g_u_sorted.loc[0, 'lag_3']]
        exog_future = g_u_sorted[['CURAH_HUJAN', 'VOLUME_CONTAINER']].values
        pred_u_rolling = inverse_transform_pred(forecast_gb_rolling(model, init_lags, exog_future), unit, tf)

        actual_all.extend(actual_u)
        pred_direct_fix.extend(pred_u_direct)
        pred_rolling_fix.extend(pred_u_rolling)

    actual_all, pred_direct_fix, pred_rolling_fix = (np.array(actual_all), np.array(pred_direct_fix), np.array(pred_rolling_fix))
    mape_direct, rmse_direct, _ = eval_pred(actual_all, pred_direct_fix)
    mape_rolling, rmse_rolling, _ = eval_pred(actual_all, pred_rolling_fix)
    identik = bool(np.allclose(pred_direct_fix, pred_rolling_fix, equal_nan=True))
    lebih_rendah = 'DIRECT' if mape_direct < mape_rolling else ('ROLLING' if mape_rolling < mape_direct else 'SAMA')
    return {'KRITERIA_ID': k, 'VARIAN': variant,
            'MAPE_DIRECT': mape_direct, 'RMSE_DIRECT': rmse_direct,
            'MAPE_ROLLING': mape_rolling, 'RMSE_ROLLING': rmse_rolling,
            'IDENTIK': identik, 'LEBIH_RENDAH': lebih_rendah}


In [41]:
CKPT_ROLLING_UNIT = OUT_DIR / 'v4_rolling_vs_direct_per_unit.csv'
done_units_roll = set(pd.read_csv(CKPT_ROLLING_UNIT)['EQUIP NAME']) if CKPT_ROLLING_UNIT.exists() else set()
rows_roll_unit = []
for _, row in final_gb_vs_v2b_unit.iterrows():
    unit = row['EQUIP NAME']
    if unit in done_units_roll or pd.isna(row['V4_GB_VARIAN']):
        continue
    params = clean_gb_params(row)
    res = rolling_vs_direct_unit(unit, row['V4_GB_VARIAN'], params)
    if res:
        rows_roll_unit.append(res)
    if len(rows_roll_unit) >= BATCH_SIZE:
        pd.DataFrame(rows_roll_unit).to_csv(CKPT_ROLLING_UNIT, mode='a', header=not CKPT_ROLLING_UNIT.exists(), index=False)
        rows_roll_unit = []
if rows_roll_unit:
    pd.DataFrame(rows_roll_unit).to_csv(CKPT_ROLLING_UNIT, mode='a', header=not CKPT_ROLLING_UNIT.exists(), index=False)
print(f'[{HLABEL}] Rolling vs Direct per-unit selesai.')


[h4bulan] Rolling vs Direct per-unit selesai.


In [42]:
CKPT_ROLLING_KRIT = OUT_DIR / 'v4_rolling_vs_direct_per_kriteria.csv'
done_krit_roll = set(pd.read_csv(CKPT_ROLLING_KRIT)['KRITERIA_ID']) if CKPT_ROLLING_KRIT.exists() else set()
rows_roll_krit = []
for _, row in final_gb_vs_v2b_kriteria.iterrows():
    k = row['KRITERIA_ID']
    if k in done_krit_roll or pd.isna(row['V4_GB_VARIAN']):
        continue
    params = clean_gb_params(row)
    res = rolling_vs_direct_kriteria(k, row['V4_GB_VARIAN'], params)
    if res:
        rows_roll_krit.append(res)
    if len(rows_roll_krit) >= BATCH_SIZE:
        pd.DataFrame(rows_roll_krit).to_csv(CKPT_ROLLING_KRIT, mode='a', header=not CKPT_ROLLING_KRIT.exists(), index=False)
        rows_roll_krit = []
if rows_roll_krit:
    pd.DataFrame(rows_roll_krit).to_csv(CKPT_ROLLING_KRIT, mode='a', header=not CKPT_ROLLING_KRIT.exists(), index=False)
print(f'[{HLABEL}] Rolling vs Direct per-kriteria selesai.')


[h4bulan] Rolling vs Direct per-kriteria selesai.


In [43]:
rolling_unit_df = pd.read_csv(CKPT_ROLLING_UNIT) if CKPT_ROLLING_UNIT.exists() else pd.DataFrame()
rolling_krit_df = pd.read_csv(CKPT_ROLLING_KRIT) if CKPT_ROLLING_KRIT.exists() else pd.DataFrame()

ROLLING_EXCEL = OUT_DIR / 'v4_rolling_vs_direct.xlsx'
with pd.ExcelWriter(ROLLING_EXCEL, engine='openpyxl') as writer:
    rolling_unit_df.to_excel(writer, sheet_name='Per_Unit', index=False)
    rolling_krit_df.to_excel(writer, sheet_name='Per_Kriteria', index=False)
    ringkasan_rows = []
    for label, df_r in [('Per Unit', rolling_unit_df), ('Per Kriteria', rolling_krit_df)]:
        if df_r.empty:
            continue
        ringkasan_rows.append({
            'LEVEL': label,
            'JUMLAH_TOTAL': len(df_r),
            'JUMLAH_IDENTIK': int(df_r['IDENTIK'].sum()),
            'DIRECT_LEBIH_RENDAH': int((df_r['LEBIH_RENDAH'] == 'DIRECT').sum()),
            'ROLLING_LEBIH_RENDAH': int((df_r['LEBIH_RENDAH'] == 'ROLLING').sum()),
            'SAMA': int((df_r['LEBIH_RENDAH'] == 'SAMA').sum()),
            'RATA2_MAPE_DIRECT': df_r['MAPE_DIRECT'].mean(),
            'RATA2_MAPE_ROLLING': df_r['MAPE_ROLLING'].mean(),
        })
    pd.DataFrame(ringkasan_rows).to_excel(writer, sheet_name='Ringkasan', index=False)

print(f'[{HLABEL}] Excel Rolling vs Direct tersimpan: {ROLLING_EXCEL}')
if not rolling_unit_df.empty:
    print(f'[{HLABEL}] Per-unit: {rolling_unit_df["IDENTIK"].sum()}/{len(rolling_unit_df)} identik, '
          f'lebih rendah -> {rolling_unit_df["LEBIH_RENDAH"].value_counts().to_dict()}')


[h4bulan] Excel Rolling vs Direct tersimpan: checkpoints_v4\h4bulan\v4_rolling_vs_direct.xlsx
[h4bulan] Per-unit: 57/205 identik, lebih rendah -> {'DIRECT': 110, 'SAMA': 58, 'ROLLING': 37}


# HORIZON: FORECAST 3 BULAN

## A. Setup Horizon 3 Bulan & Split Train/Test

Train: Jan 2023 - Sep 2025. Test: Okt - Des 2025 (3 bulan).

In [44]:
HLABEL = 'h3bulan'
OUT_DIR = Path(f'./checkpoints_v4/h3bulan')
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_END = pd.Period('2025-09', freq='M')
TEST_START = pd.Period('2025-10', freq='M')
TEST_END = pd.Period('2025-12', freq='M')
MIN_TEST_MONTHS = 3

keep_units, excluded_df = filter_units(monthly_full, TRAIN_END, TEST_START, TEST_END, MIN_TEST_MONTHS)
excluded_df.to_csv(OUT_DIR / '06_excluded_units.csv', index=False)

df_model = monthly_full[monthly_full['EQUIP NAME'].isin(keep_units)].copy()
train_df = df_model[df_model['TAHUN_BULAN'] <= TRAIN_END].copy()
test_df = df_model[(df_model['TAHUN_BULAN'] >= TEST_START) & (df_model['TAHUN_BULAN'] <= TEST_END)].copy()

train_df_raw = train_df.copy()
test_df_raw = test_df.copy()

train_full_raw = train_df_raw.merge(kriteria[['EQUIP NAME', 'KRITERIA_ID']], on='EQUIP NAME', how='left')
test_full_raw = test_df_raw.merge(kriteria[['EQUIP NAME', 'KRITERIA_ID']], on='EQUIP NAME', how='left')

unit_list = sorted(train_full_raw['EQUIP NAME'].dropna().unique().tolist())
kriteria_list = sorted(train_full_raw['KRITERIA_ID'].dropna().unique().tolist())

print(f'[{HLABEL}] Unit dipakai: {len(unit_list)} | Kriteria: {len(kriteria_list)} | Unit dikecualikan: {len(excluded_df)}')
print(f'[{HLABEL}] Train: s/d {TRAIN_END} | Test: {TEST_START} s/d {TEST_END} ({MIN_TEST_MONTHS} bulan min)')


[h3bulan] Unit dipakai: 205 | Kriteria: 69 | Unit dikecualikan: 25
[h3bulan] Train: s/d 2025-09 | Test: 2025-10 s/d 2025-12 (3 bulan min)


## B. Tahap 2 — Grid Search 24 Kombinasi (3 Bulan)

In [45]:
N_JOBS = 4
BATCH_SIZE = 10
PRINT_EVERY = 1
ALL_SUMMARY_ROWS = []


def _progress(label, done, total, batch_no):
    if batch_no % PRINT_EVERY == 0 or done >= total:
        print(f'[{HLABEL}] {label}: {done}/{total} selesai')


def run_grid_search_variant(variant, prep_fn):
    train_p, test_p, transformers = prep_fn(train_df_raw, test_df_raw)
    train_full = train_p.merge(kriteria[['EQUIP NAME', 'KRITERIA_ID']], on='EQUIP NAME', how='left')
    test_full = test_p.merge(kriteria[['EQUIP NAME', 'KRITERIA_ID']], on='EQUIP NAME', how='left')

    train_lag = make_lag_features(train_full)
    test_lag_all = make_lag_features(pd.concat([train_full, test_full], ignore_index=True))
    test_lag = test_lag_all[(test_lag_all['TAHUN_BULAN'] >= TEST_START) & (test_lag_all['TAHUN_BULAN'] <= TEST_END)].copy()

    rows = []

    # ARIMAX per-unit
    ckpt = OUT_DIR / f'v2_arimax_perunit_{variant}.csv'
    done = set()
    if ckpt.exists():
        prev = pd.read_csv(ckpt)
        done = set(zip(prev['p'], prev['d'], prev['q']))
    remaining = [o for o in pdq_grid if o not in done]
    with Parallel(n_jobs=N_JOBS) as parallel:
        for i in range(0, len(remaining), BATCH_SIZE):
            batch = remaining[i:i + BATCH_SIZE]
            res = parallel(delayed(fit_arimax_combo_unit)(o, unit_list, train_full, test_full, transformers) for o in batch)
            for r in res:
                r['MODEL'] = 'ARIMAX'; r['VARIAN'] = variant; r['LEVEL'] = 'per_unit'
            pd.DataFrame(res).to_csv(ckpt, mode='a', header=not ckpt.exists(), index=False)
            _progress(f'ARIMAX per-unit [{variant}]', i + len(batch), len(remaining), i // BATCH_SIZE)
    rows.extend(pd.read_csv(ckpt).to_dict('records'))

    # ARIMAX per-kriteria (agg mean & sum)
    for agg in ['mean', 'sum']:
        ckpt = OUT_DIR / f'v2_arimax_perkriteria_{agg}_{variant}.csv'
        done = set()
        if ckpt.exists():
            prev = pd.read_csv(ckpt)
            done = set(zip(prev['p'], prev['d'], prev['q']))
        remaining = [o for o in pdq_grid if o not in done]
        with Parallel(n_jobs=N_JOBS) as parallel:
            for i in range(0, len(remaining), BATCH_SIZE):
                batch = remaining[i:i + BATCH_SIZE]
                res = parallel(delayed(fit_arimax_combo_kriteria)(o, agg, kriteria_list, train_full, test_full, transformers) for o in batch)
                for r in res:
                    r['MODEL'] = 'ARIMAX'; r['VARIAN'] = variant; r['LEVEL'] = f'per_kriteria_{agg}'
                pd.DataFrame(res).to_csv(ckpt, mode='a', header=not ckpt.exists(), index=False)
                _progress(f'ARIMAX per-kriteria({agg}) [{variant}]', i + len(batch), len(remaining), i // BATCH_SIZE)
        rows.extend(pd.read_csv(ckpt).to_dict('records'))

    # Gradient Boosting per-unit & per-kriteria
    ckpt = OUT_DIR / f'v2_gb_perunit_{variant}.csv'
    done = set()
    if ckpt.exists():
        prev = pd.read_csv(ckpt)
        done = set(zip(prev['learning_rate'], prev['max_iter'], prev['max_depth'].astype(object), prev['min_samples_leaf']))
    remaining = [c for c in gb_grid if c not in done]
    with Parallel(n_jobs=N_JOBS) as parallel:
        for i in range(0, len(remaining), BATCH_SIZE):
            batch = remaining[i:i + BATCH_SIZE]
            res = parallel(delayed(fit_gb_combo_unit)(c, train_lag, test_lag, transformers) for c in batch)
            for r in res:
                r['MODEL'] = 'Gradient Boosting'; r['VARIAN'] = variant; r['LEVEL'] = 'per_unit'
            pd.DataFrame(res).to_csv(ckpt, mode='a', header=not ckpt.exists(), index=False)
            _progress(f'GB per-unit [{variant}]', i + len(batch), len(remaining), i // BATCH_SIZE)
    rows.extend(pd.read_csv(ckpt).to_dict('records'))

    ckpt = OUT_DIR / f'v2_gb_perkriteria_{variant}.csv'
    done = set()
    if ckpt.exists():
        prev = pd.read_csv(ckpt)
        done = set(zip(prev['learning_rate'], prev['max_iter'], prev['max_depth'].astype(object), prev['min_samples_leaf']))
    remaining = [c for c in gb_grid if c not in done]
    with Parallel(n_jobs=N_JOBS) as parallel:
        for i in range(0, len(remaining), BATCH_SIZE):
            batch = remaining[i:i + BATCH_SIZE]
            res = parallel(delayed(fit_gb_combo_kriteria)(c, train_lag, test_lag, kriteria_list, transformers) for c in batch)
            for r in res:
                r['MODEL'] = 'Gradient Boosting'; r['VARIAN'] = variant; r['LEVEL'] = 'per_kriteria'
            pd.DataFrame(res).to_csv(ckpt, mode='a', header=not ckpt.exists(), index=False)
            _progress(f'GB per-kriteria [{variant}]', i + len(batch), len(remaining), i // BATCH_SIZE)
    rows.extend(pd.read_csv(ckpt).to_dict('records'))

    # LSTM per-unit & per-kriteria
    ckpt = OUT_DIR / f'v2_lstm_perunit_{variant}.csv'
    done = set()
    if ckpt.exists():
        prev = pd.read_csv(ckpt)
        done = set(zip(prev['seq_len'], prev['units'], prev['epochs']))
    remaining = [c for c in lstm_grid if c not in done]
    for i, c in enumerate(remaining, 1):
        r = fit_lstm_combo_unit(*c, train_full, test_full, unit_list, transformers)
        r['MODEL'] = 'LSTM'; r['VARIAN'] = variant; r['LEVEL'] = 'per_unit'
        pd.DataFrame([r]).to_csv(ckpt, mode='a', header=not ckpt.exists(), index=False)
        print(f'[{HLABEL}] LSTM per-unit [{variant}]: {i}/{len(remaining)} kombinasi selesai (seq_len={c[0]}, units={c[1]}, epochs={c[2]})')
    rows.extend(pd.read_csv(ckpt).to_dict('records'))

    ckpt = OUT_DIR / f'v2_lstm_perkriteria_{variant}.csv'
    done = set()
    if ckpt.exists():
        prev = pd.read_csv(ckpt)
        done = set(zip(prev['seq_len'], prev['units'], prev['epochs']))
    remaining = [c for c in lstm_grid if c not in done]
    for i, c in enumerate(remaining, 1):
        r = fit_lstm_combo_kriteria(*c, train_full, test_full, kriteria_list, transformers)
        r['MODEL'] = 'LSTM'; r['VARIAN'] = variant; r['LEVEL'] = 'per_kriteria'
        pd.DataFrame([r]).to_csv(ckpt, mode='a', header=not ckpt.exists(), index=False)
        print(f'[{HLABEL}] LSTM per-kriteria [{variant}]: {i}/{len(remaining)} kombinasi selesai (seq_len={c[0]}, units={c[1]}, epochs={c[2]})')
    rows.extend(pd.read_csv(ckpt).to_dict('records'))

    return rows


### Varian: `baseline` (3 Bulan)

In [46]:
rows_baseline = run_grid_search_variant('baseline', prep_baseline)
ALL_SUMMARY_ROWS = [r for r in ALL_SUMMARY_ROWS if r['VARIAN'] != 'baseline']
ALL_SUMMARY_ROWS.extend(rows_baseline)
pd.DataFrame(ALL_SUMMARY_ROWS).to_csv(OUT_DIR / 'v2_MASTER_SUMMARY_24_kombinasi.csv', index=False)
print(f'[{HLABEL}] Varian baseline tersimpan ke v2_MASTER_SUMMARY_24_kombinasi.csv')


[h3bulan] GB per-unit [baseline]: 10/36 selesai
[h3bulan] GB per-unit [baseline]: 20/36 selesai
[h3bulan] GB per-unit [baseline]: 30/36 selesai
[h3bulan] GB per-unit [baseline]: 36/36 selesai
[h3bulan] GB per-kriteria [baseline]: 10/36 selesai
[h3bulan] GB per-kriteria [baseline]: 20/36 selesai
[h3bulan] GB per-kriteria [baseline]: 30/36 selesai
[h3bulan] GB per-kriteria [baseline]: 36/36 selesai
[h3bulan] Varian baseline tersimpan ke v2_MASTER_SUMMARY_24_kombinasi.csv


### Varian: `savgol` (3 Bulan)

In [47]:
rows_savgol = run_grid_search_variant('savgol', prep_savgol)
ALL_SUMMARY_ROWS = [r for r in ALL_SUMMARY_ROWS if r['VARIAN'] != 'savgol']
ALL_SUMMARY_ROWS.extend(rows_savgol)
pd.DataFrame(ALL_SUMMARY_ROWS).to_csv(OUT_DIR / 'v2_MASTER_SUMMARY_24_kombinasi.csv', index=False)
print(f'[{HLABEL}] Varian savgol tersimpan ke v2_MASTER_SUMMARY_24_kombinasi.csv')


KeyboardInterrupt: 

### Varian: `robust` (3 Bulan)

In [ ]:
rows_robust = run_grid_search_variant('robust', prep_robust)
ALL_SUMMARY_ROWS = [r for r in ALL_SUMMARY_ROWS if r['VARIAN'] != 'robust']
ALL_SUMMARY_ROWS.extend(rows_robust)
pd.DataFrame(ALL_SUMMARY_ROWS).to_csv(OUT_DIR / 'v2_MASTER_SUMMARY_24_kombinasi.csv', index=False)
print(f'[{HLABEL}] Varian robust tersimpan ke v2_MASTER_SUMMARY_24_kombinasi.csv')


[h3bulan] GB per-unit [robust]: 10/36 selesai
[h3bulan] GB per-unit [robust]: 20/36 selesai
[h3bulan] GB per-unit [robust]: 30/36 selesai
[h3bulan] GB per-unit [robust]: 36/36 selesai
[h3bulan] GB per-kriteria [robust]: 10/36 selesai
[h3bulan] GB per-kriteria [robust]: 20/36 selesai
[h3bulan] GB per-kriteria [robust]: 30/36 selesai
[h3bulan] GB per-kriteria [robust]: 36/36 selesai
[h3bulan] Varian robust tersimpan ke v2_MASTER_SUMMARY_24_kombinasi.csv


### Varian: `yeojohnson` (3 Bulan)

In [ ]:
rows_yeojohnson = run_grid_search_variant('yeojohnson', prep_yeojohnson)
ALL_SUMMARY_ROWS = [r for r in ALL_SUMMARY_ROWS if r['VARIAN'] != 'yeojohnson']
ALL_SUMMARY_ROWS.extend(rows_yeojohnson)
pd.DataFrame(ALL_SUMMARY_ROWS).to_csv(OUT_DIR / 'v2_MASTER_SUMMARY_24_kombinasi.csv', index=False)
print(f'[{HLABEL}] Varian yeojohnson tersimpan ke v2_MASTER_SUMMARY_24_kombinasi.csv')


[h3bulan] GB per-unit [yeojohnson]: 10/36 selesai
[h3bulan] GB per-unit [yeojohnson]: 20/36 selesai
[h3bulan] GB per-unit [yeojohnson]: 30/36 selesai
[h3bulan] GB per-unit [yeojohnson]: 36/36 selesai
[h3bulan] GB per-kriteria [yeojohnson]: 10/36 selesai
[h3bulan] GB per-kriteria [yeojohnson]: 20/36 selesai
[h3bulan] GB per-kriteria [yeojohnson]: 30/36 selesai
[h3bulan] GB per-kriteria [yeojohnson]: 36/36 selesai
[h3bulan] Varian yeojohnson tersimpan ke v2_MASTER_SUMMARY_24_kombinasi.csv


## C. Ringkasan Tahap 2 & Penentuan Menang per Kombinasi (3 Bulan)

In [ ]:
master_summary = pd.DataFrame(ALL_SUMMARY_ROWS)
master_summary.to_excel(OUT_DIR / 'v2_MASTER_SUMMARY_24_kombinasi.xlsx', index=False)

winner_per_combo = (master_summary.sort_values('MAPE_GLOBAL')
                     .groupby(['MODEL', 'VARIAN', 'LEVEL'], as_index=False).first())
winner_per_combo.to_excel(OUT_DIR / 'v2_winner_per_kombinasi.xlsx', index=False)
print(f'[{HLABEL}] Tahap 2 selesai — {len(master_summary)} baris hasil, {len(winner_per_combo)} kombinasi (model x varian x level).')


[h3bulan] Tahap 2 selesai — 7660 baris hasil, 28 kombinasi (model x varian x level).


## D. Tahap 2b — Drill-down 12 Kombinasi per Unit & per Kriteria (3 Bulan)

In [ ]:
def get_prepped(variant):
    train_p, test_p, transformers = DATA_PREP_VARIANTS[variant](train_df_raw, test_df_raw)
    train_full = train_p.merge(kriteria[['EQUIP NAME', 'KRITERIA_ID']], on='EQUIP NAME', how='left')
    test_full = test_p.merge(kriteria[['EQUIP NAME', 'KRITERIA_ID']], on='EQUIP NAME', how='left')
    train_lag = make_lag_features(train_full)
    test_lag_all = make_lag_features(pd.concat([train_full, test_full], ignore_index=True))
    test_lag = test_lag_all[(test_lag_all['TAHUN_BULAN'] >= TEST_START) & (test_lag_all['TAHUN_BULAN'] <= TEST_END)].copy()
    return train_full, test_full, train_lag, test_lag, transformers


# 12 kombinasi = 3 model x 4 varian, memakai parameter TERBAIK dari Tahap 2 (winner_per_combo), TANPA retuning
COMBOS_12 = []
for variant in DATA_PREP_VARIANTS:
    for model in ['ARIMAX', 'Gradient Boosting', 'LSTM']:
        COMBOS_12.append((model, variant))
print(f'[{HLABEL}] {len(COMBOS_12)} kombinasi model x varian akan direfit per unit & per kriteria.')


[h3bulan] 12 kombinasi model x varian akan direfit per unit & per kriteria.


In [ ]:
def refit_combo_group(model, variant, level):
    train_full, test_full, train_lag, test_lag, transformers = get_prepped(variant)
    w = winner_per_combo[(winner_per_combo['MODEL'] == model) & (winner_per_combo['VARIAN'] == variant)
                          & (winner_per_combo['LEVEL'] == level)]
    if w.empty:
        return pd.DataFrame()
    w = w.iloc[0]

    if model == 'ARIMAX':
        order = (int(w['p']), int(w['d']), int(w['q']))
        if level == 'per_unit':
            _, preds = fit_arimax_combo_unit(order, unit_list, train_full, test_full, transformers, return_preds=True)
        else:
            agg = level.replace('per_kriteria_', '')
            _, preds = fit_arimax_combo_kriteria(order, agg, kriteria_list, train_full, test_full, transformers, return_preds=True)
    elif model == 'Gradient Boosting':
        params_tuple = tuple(parse_param_str(model, ','.join(f'{k}={w[k]}' for k in gb_keys)).values())
        if level == 'per_unit':
            _, preds = fit_gb_combo_unit(params_tuple, train_lag, test_lag, transformers, return_preds=True)
        else:
            _, preds = fit_gb_combo_kriteria(params_tuple, train_lag, test_lag, kriteria_list, transformers, return_preds=True)
    else:  # LSTM
        if level == 'per_unit':
            _, preds = fit_lstm_combo_unit(int(w['seq_len']), int(w['units']), int(w['epochs']),
                                            train_full, test_full, unit_list, transformers, return_preds=True)
        else:
            _, preds = fit_lstm_combo_kriteria(int(w['seq_len']), int(w['units']), int(w['epochs']),
                                                train_full, test_full, kriteria_list, transformers, return_preds=True)
    preds['MODEL'] = model
    preds['VARIAN'] = variant
    return preds


LEVEL_UNIT = 'per_unit'
LEVEL_KRIT_AGG = {'ARIMAX': 'mean'}  # ARIMAX per-kriteria dipakai agregasi 'mean' utk drill-down (konsisten v3)

detail_unit_rows = []
detail_kriteria_rows = []

CKPT_DETAIL_UNIT = OUT_DIR / 'v2b_detail_per_unit_raw.csv'
CKPT_DETAIL_KRIT = OUT_DIR / 'v2b_detail_per_kriteria_raw.csv'

done_combo_unit = set()
if CKPT_DETAIL_UNIT.exists():
    prev = pd.read_csv(CKPT_DETAIL_UNIT)
    done_combo_unit = set(zip(prev['MODEL'], prev['VARIAN']))

done_combo_kriteria = set()
if CKPT_DETAIL_KRIT.exists():
    prev = pd.read_csv(CKPT_DETAIL_KRIT)
    done_combo_kriteria = set(zip(prev['MODEL'], prev['VARIAN']))

for model, variant in COMBOS_12:
    if (model, variant) not in done_combo_unit:
        level = LEVEL_KRIT_AGG.get(model, LEVEL_UNIT) if False else LEVEL_UNIT
        preds = refit_combo_group(model, variant, LEVEL_UNIT)
        if not preds.empty:
            preds.to_csv(CKPT_DETAIL_UNIT, mode='a', header=not CKPT_DETAIL_UNIT.exists(), index=False)
        done_combo_unit.add((model, variant))
        print(f'[{HLABEL}] per-unit refit selesai: {model} x {variant}')

    if (model, variant) not in done_combo_kriteria:
        level = f"per_kriteria_{LEVEL_KRIT_AGG.get(model, 'mean')}" if model == 'ARIMAX' else 'per_kriteria'
        preds = refit_combo_group(model, variant, level)
        if not preds.empty:
            preds.to_csv(CKPT_DETAIL_KRIT, mode='a', header=not CKPT_DETAIL_KRIT.exists(), index=False)
        done_combo_kriteria.add((model, variant))
        print(f'[{HLABEL}] per-kriteria refit selesai: {model} x {variant}')

print(f'[{HLABEL}] Tahap 2b refit 12 kombinasi (per-unit & per-kriteria) selesai.')


[h3bulan] Tahap 2b refit 12 kombinasi (per-unit & per-kriteria) selesai.


In [ ]:
def build_detail_metrics(raw_csv, group_col):
    raw = pd.read_csv(CKPT_DETAIL_UNIT if group_col == 'EQUIP NAME' else CKPT_DETAIL_KRIT)
    rows = []
    for (model, variant, g), gdf in raw.groupby(['MODEL', 'VARIAN', group_col]):
        mape, rmse, bias = eval_pred(gdf['ACTUAL'], gdf['PRED'])
        rows.append({group_col: g, 'MODEL': model, 'VARIAN': variant,
                     'MAPE': mape, 'RMSE': rmse, 'BIAS': bias, 'N_BULAN': len(gdf),
                     'KOMBINASI': f'{model}_{variant}'})
    return pd.DataFrame(rows)


detail_per_unit_12kombinasi = build_detail_metrics(CKPT_DETAIL_UNIT, 'EQUIP NAME')
detail_per_kriteria_12kombinasi = build_detail_metrics(CKPT_DETAIL_KRIT, 'KRITERIA_ID')

detail_per_unit_12kombinasi.to_csv(OUT_DIR / 'v2b_detail_per_unit_12kombinasi.csv', index=False)
detail_per_kriteria_12kombinasi.to_csv(OUT_DIR / 'v2b_detail_per_kriteria_12kombinasi.csv', index=False)

best_per_unit = (detail_per_unit_12kombinasi.sort_values('MAPE')
                  .groupby('EQUIP NAME', as_index=False).first())
best_per_kriteria = (detail_per_kriteria_12kombinasi.sort_values('MAPE')
                      .groupby('KRITERIA_ID', as_index=False).first())

tally_per_unit = best_per_unit['KOMBINASI'].value_counts().reset_index()
tally_per_unit.columns = ['KOMBINASI', 'JUMLAH_UNIT_MENANG']
tally_per_kriteria = best_per_kriteria['KOMBINASI'].value_counts().reset_index()
tally_per_kriteria.columns = ['KOMBINASI', 'JUMLAH_KRITERIA_MENANG']

with pd.ExcelWriter(OUT_DIR / 'v2b_hasil_12kombinasi_per_unit_kriteria.xlsx', engine='openpyxl') as writer:
    detail_per_unit_12kombinasi.to_excel(writer, sheet_name='Detail_PerUnit', index=False)
    detail_per_kriteria_12kombinasi.to_excel(writer, sheet_name='Detail_PerKriteria', index=False)
    best_per_unit.to_excel(writer, sheet_name='Best_Combo_PerUnit', index=False)
    best_per_kriteria.to_excel(writer, sheet_name='Best_Combo_PerKriteria', index=False)
    tally_per_unit.to_excel(writer, sheet_name='Tally_PerUnit', index=False)
    tally_per_kriteria.to_excel(writer, sheet_name='Tally_PerKriteria', index=False)

print(f'[{HLABEL}] GB menang di {(best_per_unit["MODEL"]=="Gradient Boosting").mean()*100:.1f}% unit, '
      f'{(best_per_kriteria["MODEL"]=="Gradient Boosting").mean()*100:.1f}% kriteria.')
print(f'[{HLABEL}] Tahap 2b selesai -> v2b_hasil_12kombinasi_per_unit_kriteria.xlsx')


[h3bulan] GB menang di 39.0% unit, 69.6% kriteria.
[h3bulan] Tahap 2b selesai -> v2b_hasil_12kombinasi_per_unit_kriteria.xlsx


## E. BARU (v4) — Force-GB: prep + hyperparameter GB disesuaikan per unit/kriteria (3 Bulan)

Target: unit/kriteria yang combo terbaiknya di Tahap 2b **BUKAN** Gradient Boosting. Untuk itu di-fit ulang PENUH
(4 varian data-prep x 36 kombinasi hyperparameter GB = 144 fit), dipilih yang MAPE-nya terendah. Unit/kriteria yang
combo terbaiknya SUDAH Gradient Boosting dibawa apa adanya (tidak di-refit, MAPE tetap sama).

In [ ]:
target_units_forcedgb = best_per_unit[best_per_unit['MODEL'] != 'Gradient Boosting']['EQUIP NAME'].tolist()
target_kriteria_forcedgb = best_per_kriteria[best_per_kriteria['MODEL'] != 'Gradient Boosting']['KRITERIA_ID'].tolist()
print(f'[{HLABEL}] {len(target_units_forcedgb)}/{len(best_per_unit)} unit & '
      f'{len(target_kriteria_forcedgb)}/{len(best_per_kriteria)} kriteria perlu di-force ke GB.')

PREPPED_VARIANTS = {}
for variant in DATA_PREP_VARIANTS:
    PREPPED_VARIANTS[variant] = get_prepped(variant)
print(f'[{HLABEL}] 4 varian data-prep siap dipakai utk search Force-GB (144 kombinasi/unit).')


[h3bulan] 125/205 unit & 21/69 kriteria perlu di-force ke GB.
[h3bulan] 4 varian data-prep siap dipakai utk search Force-GB (144 kombinasi/unit).


In [ ]:
def tune_unit_forcedgb(unit):
    best = None
    for variant, (train_full, test_full, train_lag, test_lag, tf) in PREPPED_VARIANTS.items():
        g_train = train_lag[train_lag['EQUIP NAME'] == unit].dropna(subset=FEAT_COLS + ['UTILITAS_SMOOTH'])
        g_test = test_lag[test_lag['EQUIP NAME'] == unit].dropna(subset=FEAT_COLS)
        if len(g_train) < 6 or len(g_test) == 0:
            continue
        for combo in gb_grid:
            params = dict(zip(gb_keys, combo))
            model = HistGradientBoostingRegressor(random_state=42, **params)
            model.fit(g_train[FEAT_COLS], g_train['UTILITAS_SMOOTH'])
            pred = model.predict(g_test[FEAT_COLS])
            pred = inverse_transform_pred(pred, unit, tf)
            actual = g_test.set_index('TAHUN_BULAN')['UTILITAS'].reindex(g_test['TAHUN_BULAN']).values
            mape, rmse, bias = eval_pred(actual, pred)
            if not np.isnan(mape) and (best is None or mape < best['MAPE']):
                best = {'EQUIP NAME': unit, 'VARIAN': variant, **params, 'MAPE': mape, 'RMSE': rmse, 'BIAS': bias}
    if best is None:
        best = {'EQUIP NAME': unit, 'VARIAN': np.nan, **{k: np.nan for k in gb_keys}, 'MAPE': np.nan, 'RMSE': np.nan, 'BIAS': np.nan}
    return best


CKPT_FORCEDGB_UNIT = OUT_DIR / 'v4_forcedgb_per_unit.csv'
done_units_fgb = set(pd.read_csv(CKPT_FORCEDGB_UNIT)['EQUIP NAME']) if CKPT_FORCEDGB_UNIT.exists() else set()
remaining_units_fgb = [u for u in target_units_forcedgb if u not in done_units_fgb]
print(f'[{HLABEL}] Force-GB per-unit: {len(remaining_units_fgb)} unit tersisa dari {len(target_units_forcedgb)}')

if remaining_units_fgb:
    with Parallel(n_jobs=N_JOBS) as parallel:
        for i in range(0, len(remaining_units_fgb), BATCH_SIZE):
            batch = remaining_units_fgb[i:i + BATCH_SIZE]
            res = parallel(delayed(tune_unit_forcedgb)(u) for u in batch)
            pd.DataFrame(res).to_csv(CKPT_FORCEDGB_UNIT, mode='a', header=not CKPT_FORCEDGB_UNIT.exists(), index=False)
            _progress('Force-GB per-unit', i + len(batch), len(remaining_units_fgb), i // BATCH_SIZE)
print(f'[{HLABEL}] Force-GB per-unit selesai.')


[h3bulan] Force-GB per-unit: 0 unit tersisa dari 125
[h3bulan] Force-GB per-unit selesai.


In [ ]:
def tune_kriteria_forcedgb(k):
    best = None
    for variant, (train_full, test_full, train_lag, test_lag, tf) in PREPPED_VARIANTS.items():
        g_train = train_lag[train_lag['KRITERIA_ID'] == k].dropna(subset=FEAT_COLS + ['UTILITAS_SMOOTH'])
        g_test = test_lag[test_lag['KRITERIA_ID'] == k].dropna(subset=FEAT_COLS)
        if len(g_train) < 6 or len(g_test) == 0:
            continue
        for combo in gb_grid:
            params = dict(zip(gb_keys, combo))
            model = HistGradientBoostingRegressor(random_state=42, **params)
            model.fit(g_train[FEAT_COLS], g_train['UTILITAS_SMOOTH'])
            pred_all = model.predict(g_test[FEAT_COLS])
            actual_all, pred_fix = [], []
            for unit, g_unit_test in g_test.groupby('EQUIP NAME'):
                mask = (g_test['EQUIP NAME'] == unit).values
                pred_u = inverse_transform_pred(pred_all[mask], unit, tf)
                actual_u = g_unit_test.set_index('TAHUN_BULAN')['UTILITAS'].reindex(g_unit_test['TAHUN_BULAN']).values
                actual_all.extend(actual_u)
                pred_fix.extend(pred_u)
            mape, rmse, bias = eval_pred(np.array(actual_all), np.array(pred_fix))
            if not np.isnan(mape) and (best is None or mape < best['MAPE']):
                best = {'KRITERIA_ID': k, 'VARIAN': variant, **params, 'MAPE': mape, 'RMSE': rmse, 'BIAS': bias}
    if best is None:
        best = {'KRITERIA_ID': k, 'VARIAN': np.nan, **{kk: np.nan for kk in gb_keys}, 'MAPE': np.nan, 'RMSE': np.nan, 'BIAS': np.nan}
    return best


CKPT_FORCEDGB_KRIT = OUT_DIR / 'v4_forcedgb_per_kriteria.csv'
done_krit_fgb = set(pd.read_csv(CKPT_FORCEDGB_KRIT)['KRITERIA_ID']) if CKPT_FORCEDGB_KRIT.exists() else set()
remaining_krit_fgb = [k for k in target_kriteria_forcedgb if k not in done_krit_fgb]
print(f'[{HLABEL}] Force-GB per-kriteria: {len(remaining_krit_fgb)} kriteria tersisa dari {len(target_kriteria_forcedgb)}')

if remaining_krit_fgb:
    with Parallel(n_jobs=N_JOBS) as parallel:
        for i in range(0, len(remaining_krit_fgb), BATCH_SIZE):
            batch = remaining_krit_fgb[i:i + BATCH_SIZE]
            res = parallel(delayed(tune_kriteria_forcedgb)(k) for k in batch)
            pd.DataFrame(res).to_csv(CKPT_FORCEDGB_KRIT, mode='a', header=not CKPT_FORCEDGB_KRIT.exists(), index=False)
            _progress('Force-GB per-kriteria', i + len(batch), len(remaining_krit_fgb), i // BATCH_SIZE)
print(f'[{HLABEL}] Force-GB per-kriteria selesai.')


[h3bulan] Force-GB per-kriteria: 0 kriteria tersisa dari 21
[h3bulan] Force-GB per-kriteria selesai.


In [ ]:
forcedgb_unit = pd.read_csv(CKPT_FORCEDGB_UNIT) if CKPT_FORCEDGB_UNIT.exists() else pd.DataFrame(columns=['EQUIP NAME'])
forcedgb_krit = pd.read_csv(CKPT_FORCEDGB_KRIT) if CKPT_FORCEDGB_KRIT.exists() else pd.DataFrame(columns=['KRITERIA_ID'])


def get_winner_gb_params(variant, level):
    """Ambil parameter GB global (dari Tahap 2 winner_per_combo) utk unit/kriteria yang SUDAH menang GB di v2b
    (jadi bukan hasil Force-GB, tapi tetap butuh parameter aslinya biar bisa dipakai di Rolling vs Direct)."""
    w = winner_per_combo[(winner_per_combo['MODEL'] == 'Gradient Boosting') &
                          (winner_per_combo['VARIAN'] == variant) &
                          (winner_per_combo['LEVEL'] == level)]
    if w.empty:
        return {k: np.nan for k in gb_keys}
    w = w.iloc[0]
    return {k: w[k] for k in gb_keys}


def build_final_gb_comparison(best_df, forced_df, group_col, level):
    rows = []
    forced_map = forced_df.set_index(group_col).to_dict('index') if not forced_df.empty else {}
    for _, row in best_df.iterrows():
        g = row[group_col]
        v2b_best_mape = row['MAPE']
        if row['MODEL'] == 'Gradient Boosting':
            final_variant = row['VARIAN']
            final_mape = row['MAPE']
            final_rmse = row['RMSE']
            sumber = 'v2b_original_GB'
            final_params = get_winner_gb_params(final_variant, level)
        else:
            f = forced_map.get(g)
            if f is None:
                continue
            final_variant = f['VARIAN']
            final_params = {k: f.get(k, np.nan) for k in gb_keys}
            final_mape = f['MAPE']
            final_rmse = f['RMSE']
            sumber = 'forced_v4'
        row_out = {
            group_col: g,
            'V2B_BEST_MODEL': row['MODEL'], 'V2B_BEST_VARIAN': row['VARIAN'],
            'V2B_BEST_MAPE': v2b_best_mape, 'V2B_BEST_RMSE': row['RMSE'],
            'V4_GB_SUMBER': sumber, 'V4_GB_VARIAN': final_variant,
            'V4_GB_MAPE': final_mape, 'V4_GB_RMSE': final_rmse,
            'SELISIH_MAPE_GBvsV2B': final_mape - v2b_best_mape if pd.notna(final_mape) else np.nan,
        }
        for k in gb_keys:
            row_out[f'V4_GB_{k}'] = final_params[k]
        rows.append(row_out)
    return pd.DataFrame(rows)


final_gb_vs_v2b_unit = build_final_gb_comparison(best_per_unit, forcedgb_unit, 'EQUIP NAME', 'per_unit')
final_gb_vs_v2b_kriteria = build_final_gb_comparison(best_per_kriteria, forcedgb_krit, 'KRITERIA_ID', 'per_kriteria')

FORCEDGB_EXCEL = OUT_DIR / 'v4_forcedgb_vs_v2b_comparison.xlsx'
with pd.ExcelWriter(FORCEDGB_EXCEL, engine='openpyxl') as writer:
    final_gb_vs_v2b_unit.to_excel(writer, sheet_name='Per_Unit', index=False)
    final_gb_vs_v2b_kriteria.to_excel(writer, sheet_name='Per_Kriteria', index=False)

print(f'[{HLABEL}] Rata-rata selisih MAPE (GB-force vs v2b terbaik) per unit: '
      f'{final_gb_vs_v2b_unit["SELISIH_MAPE_GBvsV2B"].mean():.2f} pp')
print(f'[{HLABEL}] Excel tersimpan: {FORCEDGB_EXCEL}')


[h3bulan] Rata-rata selisih MAPE (GB-force vs v2b terbaik) per unit: 0.49 pp
[h3bulan] Excel tersimpan: checkpoints_v4\h3bulan\v4_forcedgb_vs_v2b_comparison.xlsx


## F. BARU (v4) — Rolling vs Langsung (khusus Gradient Boosting) (3 Bulan)

Untuk kombinasi GB final tiap unit/kriteria (hasil bagian E), dibandingkan forecast **langsung** (lag pakai nilai
aktual — cara yang sudah dipakai dari awal notebook) vs **rolling** (bulan ke-2 dst pakai hasil prediksi sendiri
sbg lag, digeser maju).

In [ ]:
def rolling_vs_direct_unit(unit, variant, params):
    if pd.isna(variant):
        return None
    train_full, test_full, train_lag, test_lag, tf = PREPPED_VARIANTS[variant]
    g_train = train_lag[train_lag['EQUIP NAME'] == unit].dropna(subset=FEAT_COLS + ['UTILITAS_SMOOTH'])
    g_test = test_lag[test_lag['EQUIP NAME'] == unit].dropna(subset=FEAT_COLS)
    if len(g_train) < 6 or len(g_test) == 0:
        return None
    model = HistGradientBoostingRegressor(random_state=42, **params)
    model.fit(g_train[FEAT_COLS], g_train['UTILITAS_SMOOTH'])

    pred_direct = inverse_transform_pred(model.predict(g_test[FEAT_COLS]), unit, tf)
    actual = g_test.set_index('TAHUN_BULAN')['UTILITAS'].reindex(g_test['TAHUN_BULAN']).values
    mape_direct, rmse_direct, _ = eval_pred(actual, pred_direct)

    g_test_sorted = g_test.sort_values('TAHUN_BULAN').reset_index(drop=True)
    init_lags = [g_test_sorted.loc[0, 'lag_1'], g_test_sorted.loc[0, 'lag_2'], g_test_sorted.loc[0, 'lag_3']]
    exog_future = g_test_sorted[['CURAH_HUJAN', 'VOLUME_CONTAINER']].values
    pred_rolling = inverse_transform_pred(forecast_gb_rolling(model, init_lags, exog_future), unit, tf)
    mape_rolling, rmse_rolling, _ = eval_pred(actual, pred_rolling)

    identik = bool(np.allclose(pred_direct, pred_rolling, equal_nan=True))
    lebih_rendah = 'DIRECT' if mape_direct < mape_rolling else ('ROLLING' if mape_rolling < mape_direct else 'SAMA')
    return {'EQUIP NAME': unit, 'VARIAN': variant,
            'MAPE_DIRECT': mape_direct, 'RMSE_DIRECT': rmse_direct,
            'MAPE_ROLLING': mape_rolling, 'RMSE_ROLLING': rmse_rolling,
            'IDENTIK': identik, 'LEBIH_RENDAH': lebih_rendah}


def rolling_vs_direct_kriteria(k, variant, params):
    if pd.isna(variant):
        return None
    train_full, test_full, train_lag, test_lag, tf = PREPPED_VARIANTS[variant]
    g_train = train_lag[train_lag['KRITERIA_ID'] == k].dropna(subset=FEAT_COLS + ['UTILITAS_SMOOTH'])
    g_test = test_lag[test_lag['KRITERIA_ID'] == k].dropna(subset=FEAT_COLS)
    if len(g_train) < 6 or len(g_test) == 0:
        return None
    model = HistGradientBoostingRegressor(random_state=42, **params)
    model.fit(g_train[FEAT_COLS], g_train['UTILITAS_SMOOTH'])

    pred_direct_all = model.predict(g_test[FEAT_COLS])
    actual_all, pred_direct_fix, pred_rolling_fix = [], [], []
    for unit, g_unit_test in g_test.groupby('EQUIP NAME'):
        mask = (g_test['EQUIP NAME'] == unit).values
        pred_u_direct = inverse_transform_pred(pred_direct_all[mask], unit, tf)
        actual_u = g_unit_test.set_index('TAHUN_BULAN')['UTILITAS'].reindex(g_unit_test['TAHUN_BULAN']).values

        g_u_sorted = g_unit_test.sort_values('TAHUN_BULAN').reset_index(drop=True)
        init_lags = [g_u_sorted.loc[0, 'lag_1'], g_u_sorted.loc[0, 'lag_2'], g_u_sorted.loc[0, 'lag_3']]
        exog_future = g_u_sorted[['CURAH_HUJAN', 'VOLUME_CONTAINER']].values
        pred_u_rolling = inverse_transform_pred(forecast_gb_rolling(model, init_lags, exog_future), unit, tf)

        actual_all.extend(actual_u)
        pred_direct_fix.extend(pred_u_direct)
        pred_rolling_fix.extend(pred_u_rolling)

    actual_all, pred_direct_fix, pred_rolling_fix = (np.array(actual_all), np.array(pred_direct_fix), np.array(pred_rolling_fix))
    mape_direct, rmse_direct, _ = eval_pred(actual_all, pred_direct_fix)
    mape_rolling, rmse_rolling, _ = eval_pred(actual_all, pred_rolling_fix)
    identik = bool(np.allclose(pred_direct_fix, pred_rolling_fix, equal_nan=True))
    lebih_rendah = 'DIRECT' if mape_direct < mape_rolling else ('ROLLING' if mape_rolling < mape_direct else 'SAMA')
    return {'KRITERIA_ID': k, 'VARIAN': variant,
            'MAPE_DIRECT': mape_direct, 'RMSE_DIRECT': rmse_direct,
            'MAPE_ROLLING': mape_rolling, 'RMSE_ROLLING': rmse_rolling,
            'IDENTIK': identik, 'LEBIH_RENDAH': lebih_rendah}


In [ ]:
CKPT_ROLLING_UNIT = OUT_DIR / 'v4_rolling_vs_direct_per_unit.csv'
done_units_roll = set(pd.read_csv(CKPT_ROLLING_UNIT)['EQUIP NAME']) if CKPT_ROLLING_UNIT.exists() else set()
rows_roll_unit = []
for _, row in final_gb_vs_v2b_unit.iterrows():
    unit = row['EQUIP NAME']
    if unit in done_units_roll or pd.isna(row['V4_GB_VARIAN']):
        continue
    params = clean_gb_params(row)
    res = rolling_vs_direct_unit(unit, row['V4_GB_VARIAN'], params)
    if res:
        rows_roll_unit.append(res)
    if len(rows_roll_unit) >= BATCH_SIZE:
        pd.DataFrame(rows_roll_unit).to_csv(CKPT_ROLLING_UNIT, mode='a', header=not CKPT_ROLLING_UNIT.exists(), index=False)
        rows_roll_unit = []
if rows_roll_unit:
    pd.DataFrame(rows_roll_unit).to_csv(CKPT_ROLLING_UNIT, mode='a', header=not CKPT_ROLLING_UNIT.exists(), index=False)
print(f'[{HLABEL}] Rolling vs Direct per-unit selesai.')


[h3bulan] Rolling vs Direct per-unit selesai.


In [ ]:
CKPT_ROLLING_KRIT = OUT_DIR / 'v4_rolling_vs_direct_per_kriteria.csv'
done_krit_roll = set(pd.read_csv(CKPT_ROLLING_KRIT)['KRITERIA_ID']) if CKPT_ROLLING_KRIT.exists() else set()
rows_roll_krit = []
for _, row in final_gb_vs_v2b_kriteria.iterrows():
    k = row['KRITERIA_ID']
    if k in done_krit_roll or pd.isna(row['V4_GB_VARIAN']):
        continue
    params = clean_gb_params(row)
    res = rolling_vs_direct_kriteria(k, row['V4_GB_VARIAN'], params)
    if res:
        rows_roll_krit.append(res)
    if len(rows_roll_krit) >= BATCH_SIZE:
        pd.DataFrame(rows_roll_krit).to_csv(CKPT_ROLLING_KRIT, mode='a', header=not CKPT_ROLLING_KRIT.exists(), index=False)
        rows_roll_krit = []
if rows_roll_krit:
    pd.DataFrame(rows_roll_krit).to_csv(CKPT_ROLLING_KRIT, mode='a', header=not CKPT_ROLLING_KRIT.exists(), index=False)
print(f'[{HLABEL}] Rolling vs Direct per-kriteria selesai.')


[h3bulan] Rolling vs Direct per-kriteria selesai.


In [ ]:
rolling_unit_df = pd.read_csv(CKPT_ROLLING_UNIT) if CKPT_ROLLING_UNIT.exists() else pd.DataFrame()
rolling_krit_df = pd.read_csv(CKPT_ROLLING_KRIT) if CKPT_ROLLING_KRIT.exists() else pd.DataFrame()

ROLLING_EXCEL = OUT_DIR / 'v4_rolling_vs_direct.xlsx'
with pd.ExcelWriter(ROLLING_EXCEL, engine='openpyxl') as writer:
    rolling_unit_df.to_excel(writer, sheet_name='Per_Unit', index=False)
    rolling_krit_df.to_excel(writer, sheet_name='Per_Kriteria', index=False)
    ringkasan_rows = []
    for label, df_r in [('Per Unit', rolling_unit_df), ('Per Kriteria', rolling_krit_df)]:
        if df_r.empty:
            continue
        ringkasan_rows.append({
            'LEVEL': label,
            'JUMLAH_TOTAL': len(df_r),
            'JUMLAH_IDENTIK': int(df_r['IDENTIK'].sum()),
            'DIRECT_LEBIH_RENDAH': int((df_r['LEBIH_RENDAH'] == 'DIRECT').sum()),
            'ROLLING_LEBIH_RENDAH': int((df_r['LEBIH_RENDAH'] == 'ROLLING').sum()),
            'SAMA': int((df_r['LEBIH_RENDAH'] == 'SAMA').sum()),
            'RATA2_MAPE_DIRECT': df_r['MAPE_DIRECT'].mean(),
            'RATA2_MAPE_ROLLING': df_r['MAPE_ROLLING'].mean(),
        })
    pd.DataFrame(ringkasan_rows).to_excel(writer, sheet_name='Ringkasan', index=False)

print(f'[{HLABEL}] Excel Rolling vs Direct tersimpan: {ROLLING_EXCEL}')
if not rolling_unit_df.empty:
    print(f'[{HLABEL}] Per-unit: {rolling_unit_df["IDENTIK"].sum()}/{len(rolling_unit_df)} identik, '
          f'lebih rendah -> {rolling_unit_df["LEBIH_RENDAH"].value_counts().to_dict()}')


[h3bulan] Excel Rolling vs Direct tersimpan: checkpoints_v4\h3bulan\v4_rolling_vs_direct.xlsx
[h3bulan] Per-unit: 83/205 identik, lebih rendah -> {'DIRECT': 89, 'SAMA': 83, 'ROLLING': 33}


# RINGKASAN LINTAS-HORIZON (4 Bulan vs 3 Bulan)

In [ ]:
def load_h(hlabel):
    d = Path(f'./checkpoints_v4/{hlabel}')
    out = {}
    out['master_summary'] = pd.read_csv(d / 'v2_MASTER_SUMMARY_24_kombinasi.csv') if (d / 'v2_MASTER_SUMMARY_24_kombinasi.csv').exists() else pd.DataFrame()
    out['best_per_unit'] = pd.read_csv(d / 'v2b_detail_per_unit_12kombinasi.csv') if (d / 'v2b_detail_per_unit_12kombinasi.csv').exists() else pd.DataFrame()
    out['best_per_kriteria'] = pd.read_csv(d / 'v2b_detail_per_kriteria_12kombinasi.csv') if (d / 'v2b_detail_per_kriteria_12kombinasi.csv').exists() else pd.DataFrame()
    out['forcedgb_unit'] = pd.read_csv(d / 'v4_forcedgb_vs_v2b_comparison.xlsx') if False else None
    out['forcedgb_unit_xlsx'] = d / 'v4_forcedgb_vs_v2b_comparison.xlsx'
    out['rolling_xlsx'] = d / 'v4_rolling_vs_direct.xlsx'
    return out

h4 = load_h('h4bulan')
h3 = load_h('h3bulan')

rows = []
for hlabel, h in [('4 Bulan', h4), ('3 Bulan', h3)]:
    if h['forcedgb_unit_xlsx'].exists():
        fu = pd.read_excel(h['forcedgb_unit_xlsx'], sheet_name='Per_Unit')
        fk = pd.read_excel(h['forcedgb_unit_xlsx'], sheet_name='Per_Kriteria')
        rows.append({
            'HORIZON': hlabel, 'LEVEL': 'Per Unit',
            'RATA2_MAPE_V2B_TERBAIK': fu['V2B_BEST_MAPE'].mean(),
            'RATA2_MAPE_GB_FINAL': fu['V4_GB_MAPE'].mean(),
            'RATA2_SELISIH_MAPE': fu['SELISIH_MAPE_GBvsV2B'].mean(),
            'JUMLAH_DI_FORCE_KE_GB': (fu['V4_GB_SUMBER'] == 'forced_v4').sum(),
        })
        rows.append({
            'HORIZON': hlabel, 'LEVEL': 'Per Kriteria',
            'RATA2_MAPE_V2B_TERBAIK': fk['V2B_BEST_MAPE'].mean(),
            'RATA2_MAPE_GB_FINAL': fk['V4_GB_MAPE'].mean(),
            'RATA2_SELISIH_MAPE': fk['SELISIH_MAPE_GBvsV2B'].mean(),
            'JUMLAH_DI_FORCE_KE_GB': (fk['V4_GB_SUMBER'] == 'forced_v4').sum(),
        })

ringkasan_forcedgb_lintas_horizon = pd.DataFrame(rows)

rows2 = []
for hlabel, h in [('4 Bulan', h4), ('3 Bulan', h3)]:
    if h['rolling_xlsx'].exists():
        ru = pd.read_excel(h['rolling_xlsx'], sheet_name='Per_Unit')
        rk = pd.read_excel(h['rolling_xlsx'], sheet_name='Per_Kriteria')
        for label, df_r in [('Per Unit', ru), ('Per Kriteria', rk)]:
            if df_r.empty:
                continue
            rows2.append({
                'HORIZON': hlabel, 'LEVEL': label,
                'JUMLAH_TOTAL': len(df_r),
                'JUMLAH_IDENTIK': int(df_r['IDENTIK'].sum()),
                'DIRECT_LEBIH_RENDAH': int((df_r['LEBIH_RENDAH'] == 'DIRECT').sum()),
                'ROLLING_LEBIH_RENDAH': int((df_r['LEBIH_RENDAH'] == 'ROLLING').sum()),
                'RATA2_MAPE_DIRECT': df_r['MAPE_DIRECT'].mean(),
                'RATA2_MAPE_ROLLING': df_r['MAPE_ROLLING'].mean(),
            })

ringkasan_rolling_lintas_horizon = pd.DataFrame(rows2)

FINAL_EXCEL = Path('./checkpoints_v4') / 'v4_RINGKASAN_LINTAS_HORIZON.xlsx'
with pd.ExcelWriter(FINAL_EXCEL, engine='openpyxl') as writer:
    ringkasan_forcedgb_lintas_horizon.to_excel(writer, sheet_name='ForceGB_vs_v2b', index=False)
    ringkasan_rolling_lintas_horizon.to_excel(writer, sheet_name='Rolling_vs_Direct', index=False)

print('Ringkasan lintas-horizon tersimpan:', FINAL_EXCEL)
ringkasan_forcedgb_lintas_horizon


Ringkasan lintas-horizon tersimpan: checkpoints_v4\v4_RINGKASAN_LINTAS_HORIZON.xlsx


,HORIZON,LEVEL,RATA2_MAPE_V2B_TERBAIK,RATA2_MAPE_GB_FINAL,RATA2_SELISIH_MAPE,JUMLAH_DI_FORCE_KE_GB
0,4 Bulan,Per Unit,16.934454,16.190109,-0.744346,123
1,4 Bulan,Per Kriteria,23.975023,23.509741,-0.465282,28
2,3 Bulan,Per Unit,14.990601,15.479149,0.488548,125
3,3 Bulan,Per Kriteria,22.625829,22.292542,-0.333287,21


In [ ]:
# ============================================================
# HEAD-TO-HEAD 4 BULAN vs 3 BULAN  ->  v4_PERBANDINGAN_4BULAN_vs_3BULAN.xlsx
# Membandingkan hasil GB final (kolom V4_GB_*) per unit & per kriteria antar horizon.
# Prasyarat: cell di atas (load_h) sudah jalan dan file v4_forcedgb_vs_v2b_comparison.xlsx
# sudah ada di h4bulan/ dan h3bulan/.
# Catatan: MAPE 4 bulan vs 3 bulan dihitung pada jendela test berbeda (3 bulan = 1 bulan lebih pendek),
# jadi perbandingan ini indikatif, bukan apple-to-apple murni.
# ============================================================
def _head_to_head(key_col, sheet_name):
    f4, f3 = h4['forcedgb_unit_xlsx'], h3['forcedgb_unit_xlsx']
    if not (f4.exists() and f3.exists()):
        print(f'[skip] file forcedgb belum lengkap untuk {sheet_name}: h4={f4.exists()}, h3={f3.exists()}')
        return pd.DataFrame()
    a = pd.read_excel(f4, sheet_name=sheet_name)[[key_col, 'V4_GB_VARIAN', 'V4_GB_MAPE', 'V4_GB_RMSE']]
    b = pd.read_excel(f3, sheet_name=sheet_name)[[key_col, 'V4_GB_VARIAN', 'V4_GB_MAPE', 'V4_GB_RMSE']]
    a = a.rename(columns={'V4_GB_VARIAN': 'VARIAN_4BLN', 'V4_GB_MAPE': 'MAPE_4BLN', 'V4_GB_RMSE': 'RMSE_4BLN'})
    b = b.rename(columns={'V4_GB_VARIAN': 'VARIAN_3BLN', 'V4_GB_MAPE': 'MAPE_3BLN', 'V4_GB_RMSE': 'RMSE_3BLN'})
    m = a.merge(b, on=key_col, how='inner')
    m = m.dropna(subset=['MAPE_4BLN', 'MAPE_3BLN']).reset_index(drop=True)
    m['SELISIH_MAPE_3m4'] = m['MAPE_3BLN'] - m['MAPE_4BLN']   # negatif = 3 bulan lebih baik
    m['SELISIH_RMSE_3m4'] = m['RMSE_3BLN'] - m['RMSE_4BLN']

    def _pick(x4, x3):
        if np.isclose(x4, x3, rtol=0, atol=1e-9):
            return 'SAMA'
        return '3 BULAN' if x3 < x4 else '4 BULAN'

    m['LEBIH_RENDAH'] = [_pick(x, y) for x, y in zip(m['MAPE_4BLN'], m['MAPE_3BLN'])]
    m['LEBIH_RENDAH_RMSE'] = [_pick(x, y) for x, y in zip(m['RMSE_4BLN'], m['RMSE_3BLN'])]
    return m


def _ringkas(df, level):
    if df.empty:
        return None
    mape4, mape3 = df['MAPE_4BLN'].mean(), df['MAPE_3BLN'].mean()
    rmse4, rmse3 = df['RMSE_4BLN'].mean(), df['RMSE_3BLN'].mean()
    try:
        from scipy.stats import wilcoxon
        d = df['MAPE_3BLN'] - df['MAPE_4BLN']
        p = wilcoxon(d).pvalue if (d != 0).any() else np.nan
    except Exception:
        p = np.nan
    return {
        'LEVEL': level,
        'JUMLAH_DIBANDINGKAN': len(df),
        'LEBIH_RENDAH_3BLN': int((df['LEBIH_RENDAH'] == '3 BULAN').sum()),
        'LEBIH_RENDAH_4BLN': int((df['LEBIH_RENDAH'] == '4 BULAN').sum()),
        'SAMA': int((df['LEBIH_RENDAH'] == 'SAMA').sum()),
        'RATA2_MAPE_4BLN': mape4, 'RATA2_MAPE_3BLN': mape3,
        'SELISIH_MAPE_3m4': mape3 - mape4,
        'MEDIAN_MAPE_4BLN': df['MAPE_4BLN'].median(), 'MEDIAN_MAPE_3BLN': df['MAPE_3BLN'].median(),
        'RATA2_RMSE_4BLN': rmse4, 'RATA2_RMSE_3BLN': rmse3,
        'P_VALUE_WILCOXON_MAPE': p,
        'HORIZON_MENANG_RATA2_MAPE': '3 BULAN' if mape3 < mape4 else ('4 BULAN' if mape4 < mape3 else 'SAMA'),
    }


hh_unit = _head_to_head('EQUIP NAME', 'Per_Unit')
hh_krit = _head_to_head('KRITERIA_ID', 'Per_Kriteria')

ringkasan_hh_rows = [r for r in (_ringkas(hh_unit, 'Per Unit'), _ringkas(hh_krit, 'Per Kriteria')) if r]
ringkasan_hh = pd.DataFrame(ringkasan_hh_rows)

PERBANDINGAN_EXCEL = Path('./checkpoints_v4') / 'v4_PERBANDINGAN_4BULAN_vs_3BULAN.xlsx'
if not ringkasan_hh.empty:
    with pd.ExcelWriter(PERBANDINGAN_EXCEL, engine='openpyxl') as writer:
        hh_unit.to_excel(writer, sheet_name='Per_Unit', index=False)
        hh_krit.to_excel(writer, sheet_name='Per_Kriteria', index=False)
        ringkasan_hh.to_excel(writer, sheet_name='Ringkasan', index=False)
    print('Perbandingan 4 vs 3 bulan tersimpan:', PERBANDINGAN_EXCEL)

    for _, r in ringkasan_hh.iterrows():
        print(f"\n[{r['LEVEL']}] n={r['JUMLAH_DIBANDINGKAN']} | "
              f"3 bln lebih rendah: {r['LEBIH_RENDAH_3BLN']}, 4 bln lebih rendah: {r['LEBIH_RENDAH_4BLN']}, sama: {r['SAMA']}")
        print(f"  Rata2 MAPE  4 bln = {r['RATA2_MAPE_4BLN']:.2f} | 3 bln = {r['RATA2_MAPE_3BLN']:.2f} "
              f"(selisih 3-4 = {r['SELISIH_MAPE_3m4']:+.2f} pp)")
        print(f"  Median MAPE 4 bln = {r['MEDIAN_MAPE_4BLN']:.2f} | 3 bln = {r['MEDIAN_MAPE_3BLN']:.2f}")
        print(f"  Wilcoxon p-value (MAPE, berpasangan) = {r['P_VALUE_WILCOXON_MAPE']:.4f}")
        print(f"  => Horizon dengan rata-rata MAPE lebih rendah: {r['HORIZON_MENANG_RATA2_MAPE']}")
else:
    print('Belum ada data untuk dibandingkan (cek h4bulan/h3bulan).')

ringkasan_hh


Perbandingan 4 vs 3 bulan tersimpan: checkpoints_v4\v4_PERBANDINGAN_4BULAN_vs_3BULAN.xlsx

[Per Unit] n=205 | 3 bln lebih rendah: 113, 4 bln lebih rendah: 92, sama: 0
  Rata2 MAPE  4 bln = 16.19 | 3 bln = 15.48 (selisih 3-4 = -0.71 pp)
  Median MAPE 4 bln = 11.19 | 3 bln = 10.16
  Wilcoxon p-value (MAPE, berpasangan) = 0.2785
  => Horizon dengan rata-rata MAPE lebih rendah: 3 BULAN

[Per Kriteria] n=69 | 3 bln lebih rendah: 33, 4 bln lebih rendah: 36, sama: 0
  Rata2 MAPE  4 bln = 23.51 | 3 bln = 22.29 (selisih 3-4 = -1.22 pp)
  Median MAPE 4 bln = 16.61 | 3 bln = 16.64
  Wilcoxon p-value (MAPE, berpasangan) = 0.7131
  => Horizon dengan rata-rata MAPE lebih rendah: 3 BULAN


,LEVEL,JUMLAH_DIBANDINGKAN,LEBIH_RENDAH_3BLN,LEBIH_RENDAH_4BLN,SAMA,RATA2_MAPE_4BLN,RATA2_MAPE_3BLN,SELISIH_MAPE_3m4,MEDIAN_MAPE_4BLN,MEDIAN_MAPE_3BLN,RATA2_RMSE_4BLN,RATA2_RMSE_3BLN,P_VALUE_WILCOXON_MAPE,HORIZON_MENANG_RATA2_MAPE
0,Per Unit,205,113,92,0,16.190109,15.479149,-0.710960,11.193800,10.164290,40.967555,38.793261,0.278542,3 BULAN
1,Per Kriteria,69,33,36,0,23.509741,22.292542,-1.217199,16.609675,16.642082,39.747758,38.371631,0.713093,3 BULAN


## Rangkuman file output v4

Per horizon (`./checkpoints_v4/h4bulan/` dan `./checkpoints_v4/h3bulan/`):
- `v2_MASTER_SUMMARY_24_kombinasi.xlsx` — hasil Tahap 2 (24 kombinasi)
- `v2b_hasil_12kombinasi_per_unit_kriteria.xlsx` — hasil Tahap 2b (drill-down per unit/kriteria)
- `v4_forcedgb_vs_v2b_comparison.xlsx` — **BARU**: GB (prep+param disesuaikan) vs MAPE terbaik v2b, per unit & kriteria
- `v4_rolling_vs_direct.xlsx` — **BARU**: perbandingan cara rolling vs langsung, per unit & kriteria

Lintas horizon:
- `./checkpoints_v4/v4_RINGKASAN_LINTAS_HORIZON.xlsx`
- `./checkpoints_v4/v4_PERBANDINGAN_4BULAN_vs_3BULAN.xlsx` — **BARU**: head-to-head MAPE/RMSE final 4 vs 3 bulan per unit & kriteria (kolom `LEBIH_RENDAH`), sheet `Ringkasan`

**Kalau macet di tengah jalan (restart kernel dsb.)**: cukup rerun dari cell import di paling atas s/d cell
setup horizon yang sedang dikerjakan, lalu lanjut — semua tahap sudah checkpoint per unit/kombinasi.


# TAHAP 3 -- MODEL FINAL & FUTURE FORECAST (horizon 3 bulan, per unit, GB final)

Horizon yang dipakai di sini SUDAH ditetapkan sebagai horizon final (3 bulan, pendekatan per unit,
GB final per unit hasil Tahap 2b/Force-GB -- lihat `v4_forcedgb_vs_v2b_comparison.xlsx` h3bulan).
Tahap ini TIDAK melakukan pencarian kombinasi ulang -- semua konfigurasi (varian data-prep +
hyperparameter GB) per unit diambil apa adanya dari hasil Tahap 2b/Force-GB yang sudah selesai.

Dua hal penting:
1. **Retrain pakai SEMUA histori** (Jan 2023 s/d bulan terakhir yang ada di `monthly_full`), bukan cuma
   train Jan 2023-Sep 2025 seperti saat evaluasi. Tujuannya model final memanfaatkan data test (Okt-Des 2025)
   juga sebagai bahan belajar, bukan cuma buat diuji.
2. **Forecast ke depan WAJIB rolling/recursive** (lag_1 = prediksi bulan sebelumnya), BUKAN direct seperti
   saat evaluasi. Ini karena direct butuh nilai lag AKTUAL, dan untuk bulan di masa depan nilai aktualnya
   belum ada sama sekali. Rolling tahu dari hasil Tahap F (rolling vs direct) errornya lebih besar dan makin
   menumpuk makin jauh horizon-nya -- itulah dasar kenapa dashboard nanti perlu kasih catatan "makin jauh
   horizon forecast, makin besar potensi error", bukan klaim akurasi yang sama sepanjang horizon.

Variabel eksogen (CURAH_HUJAN, VOLUME_CONTAINER) utk bulan-bulan masa depan juga belum ada nilai aktualnya,
jadi dipakai ESTIMASI rata-rata musiman (rata-rata historis per bulan kalender, Jan/Feb/.../Des, dari
2023-2025). ASUMSI ini adalah limitasi metodologi yang perlu disebutkan di buku/laporan -- bukan nilai
curah hujan/volume container yang sesungguhnya akan terjadi.


In [ ]:
# ======== TAHAP 3 - SETUP ========
HLABEL = 'h3bulan'  # horizon final yang dipilih (3 bulan, per-unit, GB final)
OUT_DIR = Path(f'./checkpoints_v4/{HLABEL}')
FUTURE_DIR = OUT_DIR / 'future_forecast'
FUTURE_DIR.mkdir(parents=True, exist_ok=True)

final_gb_unit = pd.read_excel(OUT_DIR / 'v4_forcedgb_vs_v2b_comparison.xlsx', sheet_name='Per_Unit')
print(f'[{HLABEL}] Konfigurasi final GB per unit dimuat: {len(final_gb_unit)} baris')

# -- Pisahkan unit yang konfigurasinya valid vs tidak (VARIAN/param NaN -> gagal tuning Force-GB) --
invalid_cfg = final_gb_unit[final_gb_unit['V4_GB_VARIAN'].isna()].copy()
valid_cfg = final_gb_unit[final_gb_unit['V4_GB_VARIAN'].notna()].copy()
print(f'[{HLABEL}] Unit dgn konfigurasi valid: {len(valid_cfg)} | tidak valid (NaN, dikeluarkan): {len(invalid_cfg)}')


[h3bulan] Konfigurasi final GB per unit dimuat: 205 baris
[h3bulan] Unit dgn konfigurasi valid: 205 | tidak valid (NaN, dikeluarkan): 0


In [ ]:
# ======== Estimasi variabel eksogen masa depan (curah hujan & volume container) ========
# Nilai aktual masa depan belum ada (belum terjadi), jadi dipakai PROKSI rata-rata musiman:
# rata-rata nilai historis pada BULAN KALENDER yang sama (Jan/Feb/.../Des, digabung dari 2023-2025).
# Ini asumsi, bukan data riil -- dicatat sbg limitasi metodologi pada bagian pembahasan.
MAX_HORIZON = 12  # bulan maksimum yang disiapkan (dashboard default pakai 3, bisa ditarik s/d 12)
N_JOBS = 4
BATCH_SIZE_FUTURE = 20

exog_monthly = monthly_full.groupby('TAHUN_BULAN')[['CURAH_HUJAN', 'VOLUME_CONTAINER']].first()
exog_monthly_bulan = exog_monthly.copy()
exog_monthly_bulan['BULAN_KALENDER'] = exog_monthly_bulan.index.map(lambda p: p.month)
exog_seasonal_avg = exog_monthly_bulan.groupby('BULAN_KALENDER')[['CURAH_HUJAN', 'VOLUME_CONTAINER']].mean()

LAST_PERIOD = monthly_full['TAHUN_BULAN'].max()
future_periods = pd.period_range(LAST_PERIOD + 1, periods=MAX_HORIZON, freq='M')
exog_future_df = pd.DataFrame({
    'TAHUN_BULAN': future_periods,
    'BULAN_KALENDER': [p.month for p in future_periods],
}).merge(exog_seasonal_avg.reset_index(), on='BULAN_KALENDER', how='left')
exog_future_df = exog_future_df[['TAHUN_BULAN', 'CURAH_HUJAN', 'VOLUME_CONTAINER']]
exog_future_df.to_csv(FUTURE_DIR / 'exog_future_estimasi_musiman.csv', index=False)
print(f'[{HLABEL}] Estimasi eksogen {MAX_HORIZON} bulan ke depan '
      f'({future_periods[0]} s/d {future_periods[-1]}) tersimpan.')
exog_future_df.head(12)


[h3bulan] Estimasi eksogen 12 bulan ke depan (2026-01 s/d 2026-12) tersimpan.


,TAHUN_BULAN,CURAH_HUJAN,VOLUME_CONTAINER
0,2026-01,324.133333,357333.666667
1,2026-02,343.466667,347183.333333
2,2026-03,258.700000,340911.000000
3,2026-04,197.366667,289576.666667
4,2026-05,118.666667,357386.333333
5,2026-06,37.066667,338612.333333
6,2026-07,15.666667,360732.666667
7,2026-08,13.000000,360640.000000
8,2026-09,12.300000,361504.666667
9,2026-10,47.666667,387665.333333


In [ ]:
def retrain_and_forecast_unit(row, exog_future_df, horizon=MAX_HORIZON):
    '''Retrain GB pakai SELURUH histori unit (data-prep & hyperparameter = hasil final Tahap 2b/Force-GB),
    lalu forecast ke depan secara rolling/recursive (lag_1 = prediksi bulan sebelumnya utk bulan ke-2 dst).
    Return: (DataFrame hasil forecast, None) jika berhasil, (None, None) jika gagal (data tidak cukup).'''
    unit = row['EQUIP NAME']
    variant = row['V4_GB_VARIAN']
    params = clean_gb_params(row)

    g_full = monthly_full[monthly_full['EQUIP NAME'] == unit].sort_values('TAHUN_BULAN').copy()
    if len(g_full) < 6:
        return None, None

    prep_fn = DATA_PREP_VARIANTS[variant]
    # prep_fn butuh (train_df, test_df); test_df diisi g_full lagi (bukan slice kosong) krn
    # groupby().apply() pandas pada dataframe 0-baris bisa menjatuhkan kolom group_col (bug
    # pandas, bukan punya kita) -- output test di sini tetap dibuang, tidak dipakai.
    g_prepped, _, transformers = prep_fn(g_full, g_full)
    g_lag = make_lag_features(g_prepped)
    g_lag_valid = g_lag.dropna(subset=FEAT_COLS + ['UTILITAS_SMOOTH'])
    if len(g_lag_valid) < 6:
        return None, None

    model = HistGradientBoostingRegressor(random_state=42, **params)
    model.fit(g_lag_valid[FEAT_COLS], g_lag_valid['UTILITAS_SMOOTH'])

    last_row = g_lag.iloc[-1]
    last_lags = [last_row['lag_1'], last_row['lag_2'], last_row['lag_3']]
    if any(pd.isna(last_lags)):
        return None, None

    exog_h = exog_future_df.head(horizon)[['CURAH_HUJAN', 'VOLUME_CONTAINER']].values
    pred_smooth = forecast_gb_rolling(model, last_lags, exog_h)
    pred_real = inverse_transform_pred(pred_smooth, unit, transformers)

    out = pd.DataFrame({
        'EQUIP NAME': unit,
        'TAHUN_BULAN': exog_future_df.head(horizon)['TAHUN_BULAN'].values,
        'HORIZON_BULAN_KE': range(1, horizon + 1),
        'FORECAST_UTILITAS': pred_real,
        'VARIAN': variant,
    })
    return out, None


def _retrain_forecast_safe(row):
    try:
        out, _ = retrain_and_forecast_unit(row, exog_future_df, horizon=MAX_HORIZON)
        return out
    except Exception:
        return None


In [ ]:
CKPT_FUTURE = FUTURE_DIR / 'v4_future_forecast_per_unit.csv'
done_units_future = set(pd.read_csv(CKPT_FUTURE)['EQUIP NAME']) if CKPT_FUTURE.exists() else set()
remaining_rows = [r for _, r in valid_cfg.iterrows() if r['EQUIP NAME'] not in done_units_future]
print(f'[{HLABEL}] Future forecast: {len(remaining_rows)} unit tersisa dari {len(valid_cfg)}')

failed_units_future = []
with Parallel(n_jobs=N_JOBS) as parallel:
    for i in range(0, len(remaining_rows), BATCH_SIZE_FUTURE):
        batch = remaining_rows[i:i + BATCH_SIZE_FUTURE]
        results = parallel(delayed(_retrain_forecast_safe)(r) for r in batch)
        ok_results = [r for r in results if r is not None]
        failed_units_future.extend([batch[j]['EQUIP NAME'] for j, r in enumerate(results) if r is None])
        if ok_results:
            pd.concat(ok_results, ignore_index=True).to_csv(
                CKPT_FUTURE, mode='a', header=not CKPT_FUTURE.exists(), index=False)
        print(f'[{HLABEL}] Future forecast: {min(i + len(batch), len(remaining_rows))}/{len(remaining_rows)} unit diproses')

print(f'[{HLABEL}] Selesai. Unit gagal retrain/forecast: {len(failed_units_future)}')
if failed_units_future:
    print(failed_units_future[:10], '...' if len(failed_units_future) > 10 else '')

future_forecast_df = pd.read_csv(CKPT_FUTURE)
print(future_forecast_df.shape)
future_forecast_df.head(12)

[h3bulan] Future forecast: 205 unit tersisa dari 205
[h3bulan] Future forecast: 20/205 unit diproses
[h3bulan] Future forecast: 40/205 unit diproses
[h3bulan] Future forecast: 60/205 unit diproses
[h3bulan] Future forecast: 80/205 unit diproses
[h3bulan] Future forecast: 100/205 unit diproses
[h3bulan] Future forecast: 120/205 unit diproses
[h3bulan] Future forecast: 140/205 unit diproses
[h3bulan] Future forecast: 160/205 unit diproses
[h3bulan] Future forecast: 180/205 unit diproses
[h3bulan] Future forecast: 200/205 unit diproses
[h3bulan] Future forecast: 205/205 unit diproses
[h3bulan] Selesai. Unit gagal retrain/forecast: 0
(2460, 5)


,EQUIP NAME,TAHUN_BULAN,HORIZON_BULAN_KE,FORECAST_UTILITAS,VARIAN
0,UNIT_0002,2026-01,1,183.329534,savgol
1,UNIT_0002,2026-02,2,199.185399,savgol
2,UNIT_0002,2026-03,3,191.803580,savgol
3,UNIT_0002,2026-04,4,203.736134,savgol
4,UNIT_0002,2026-05,5,212.632119,savgol
5,UNIT_0002,2026-06,6,193.730666,savgol
6,UNIT_0002,2026-07,7,216.656043,savgol
7,UNIT_0002,2026-08,8,208.132551,savgol
8,UNIT_0002,2026-09,9,212.308203,savgol
9,UNIT_0002,2026-10,10,200.517008,savgol


In [ ]:
# ======== Rangkuman unit yang DI-EXCLUDE dari sistem, lengkap dengan alasan & tahapnya ========
# Dipakai nanti utk tab "Unit Di-exclude" pada dashboard perusahaan.
alasan_rows = []

for old_code in sorted(EXCLUDED_OLD_CODES, key=lambda c: int(c.split('_')[1])):
    alasan_rows.append({
        'EQUIP NAME': old_code, 'TAHAP': 'Populasi awal',
        'ALASAN': 'Unit non-operasional / alat pendukung',
    })

excluded_units_h3 = pd.read_csv(OUT_DIR / '06_excluded_units.csv')
for _, r in excluded_units_h3.iterrows():
    alasan_rows.append({
        'EQUIP NAME': r['EQUIP NAME'], 'TAHAP': 'Filter kelayakan horizon 3 bulan',
        'ALASAN': r['ALASAN_EXCLUDE'],
    })

for u in invalid_cfg['EQUIP NAME']:
    alasan_rows.append({
        'EQUIP NAME': u, 'TAHAP': 'Force-GB tuning',
        'ALASAN': 'Tidak ditemukan kombinasi data-prep + hyperparameter GB yang valid (data train/test terlalu sedikit)',
    })

for u in failed_units_future:
    alasan_rows.append({
        'EQUIP NAME': u, 'TAHAP': 'Retrain & future forecast',
        'ALASAN': 'Gagal retrain pada seluruh histori data (data tidak cukup setelah transformasi lag)',
    })

excluded_summary_df = pd.DataFrame(alasan_rows)
EXCLUDED_SUMMARY_EXCEL = FUTURE_DIR / 'v4_unit_excluded_summary.xlsx'
excluded_summary_df.to_excel(EXCLUDED_SUMMARY_EXCEL, index=False)
print(f'[{HLABEL}] Rangkuman {len(excluded_summary_df)} unit di-exclude (semua tahap) tersimpan: {EXCLUDED_SUMMARY_EXCEL}')
excluded_summary_df['TAHAP'].value_counts()


[h3bulan] Rangkuman 35 unit di-exclude (semua tahap) tersimpan: checkpoints_v4\h3bulan\future_forecast\v4_unit_excluded_summary.xlsx


TAHAP
Filter kelayakan horizon 3 bulan    25
Populasi awal                       10
Name: count, dtype: int64

In [ ]:
FUTURE_FORECAST_EXCEL = FUTURE_DIR / 'v4_future_forecast_final.xlsx'
with pd.ExcelWriter(FUTURE_FORECAST_EXCEL, engine='openpyxl') as writer:
    future_forecast_df.to_excel(writer, sheet_name='Forecast_Panjang', index=False)
    future_forecast_df.pivot(index='EQUIP NAME', columns='HORIZON_BULAN_KE',
                              values='FORECAST_UTILITAS').to_excel(writer, sheet_name='Forecast_Pivot')
    excluded_summary_df.to_excel(writer, sheet_name='Unit_Diexclude', index=False)

print(f'[{HLABEL}] Hasil future forecast final tersimpan: {FUTURE_FORECAST_EXCEL}')
print(f'[{HLABEL}] Unit berhasil forecast: {future_forecast_df["EQUIP NAME"].nunique()} | '
      f'Unit di-exclude (semua tahap): {len(excluded_summary_df)}')


[h3bulan] Hasil future forecast final tersimpan: checkpoints_v4\h3bulan\future_forecast\v4_future_forecast_final.xlsx
[h3bulan] Unit berhasil forecast: 205 | Unit di-exclude (semua tahap): 35


## Rangkuman file output Tahap 3

- `./checkpoints_v4/h3bulan/future_forecast/exog_future_estimasi_musiman.csv` -- estimasi CURAH_HUJAN &
  VOLUME_CONTAINER 12 bulan ke depan (rata-rata musiman historis per bulan kalender).
- `./checkpoints_v4/h3bulan/future_forecast/v4_future_forecast_per_unit.csv` -- checkpoint mentah (long
  format), auto-skip kalau dijalankan ulang.
- `./checkpoints_v4/h3bulan/future_forecast/v4_future_forecast_final.xlsx` -- hasil akhir, 3 sheet:
  `Forecast_Panjang` (long format, 1 baris = 1 unit x 1 bulan ke depan), `Forecast_Pivot` (1 baris = 1 unit,
  kolom = horizon bulan ke-1 s/d 12), `Unit_Diexclude` (daftar unit + tahap + alasan exclude).
- `./checkpoints_v4/h3bulan/future_forecast/v4_unit_excluded_summary.xlsx` -- sama dgn sheet `Unit_Diexclude`
  di atas, file terpisah biar gampang dipakai langsung di dashboard.

File-file ini yang nanti dibaca oleh `appDemo.py` dan `appPerusahaan.py` (lewat `pipeline_lib.py`), bukan
notebook ini secara langsung.


# BAGIAN BARU: ANALISIS EKSOGEN LAG-1 & EVALUASI MODEL FINAL (190 UNIT)
## Evaluasi Eksogen Bebas Leakage (Lag-1), ARIMA vs ARIMAX, dan Model Final 190 Unit
Bagian ini memuat analisis lanjutan perbandingan model deret waktu dan variabel eksogen:
1. **Bebas Leakage**: Eksogen dimundurkan 1 bulan ($t-1$) agar saat memprediksi bulan $t$, data eksogen sudah pasti diketahui tanpa asumsi masa depan.
2. **ARIMA vs ARIMAX**: Membandingkan akurasi ARIMA murni vs ARIMAX ($t$) vs ARIMAX ($t-1$).
3. **Gradient Boosting Eksogen Lag-1**: Menguji performa GB dengan eksogen lag-1 vs eksogen saat ini vs tanpa eksogen.
4. **190 Unit Analisis**: Pemodelan terfokus pada 190 unit armada alat berat yang memenuhi kriteria kelayakan analisis.
Output bagian ini disimpan di `./checkpoints_v4/rev_h3/`.


In [ ]:
# SETUP ANALISIS 190 UNIT & EKSOGEN LAG-1 (DENGAN PENAMAAN UNIT & KODE KRITERIA BARU)
REVISI_DIR = Path('./checkpoints_v4/rev_h3')
REVISI_DIR.mkdir(parents=True, exist_ok=True)

# Muat mapping kriteria dan penamaan unit baru (Format: K<NoKriteria>_UNIT_<Nomor3Digit>)
ref_mapping_path = Path('./checkpoints_v4/mapping_unit_dan_kriteria.xlsx')
if ref_mapping_path.exists():
    df_ref = pd.read_excel(ref_mapping_path, sheet_name='Mapping_Unit')
    unit_map_new = dict(zip(df_ref['EQUIP_NAME_LAMA'], df_ref['EQUIP_NAME_BARU']))
    kriteria_code_map = dict(zip(df_ref['KRITERIA_ID'], df_ref['KODE_KRITERIA']))
else:
    unit_map_new = {}
    kriteria_code_map = {}

# 1. Muat dataset dan siapkan fitur eksogen lag-1 (mundur 1 bulan)
monthly_full_revisi = pd.read_csv('./checkpoints_v4/04_monthly_with_exog.csv')
monthly_full_revisi['TAHUN_BULAN'] = pd.PeriodIndex(monthly_full_revisi['TAHUN_BULAN'], freq='M')
monthly_full_revisi['EQUIP NAME'] = monthly_full_revisi['EQUIP NAME'].map(lambda u: unit_map_new.get(u, u))

exog_m = monthly_full_revisi.groupby('TAHUN_BULAN')[['CURAH_HUJAN', 'VOLUME_CONTAINER']].first().sort_index()
exog_m['CH_LAG1'] = exog_m['CURAH_HUJAN'].shift(1).bfill()
exog_m['VC_LAG1'] = exog_m['VOLUME_CONTAINER'].shift(1).bfill()

monthly_full_revisi = monthly_full_revisi.merge(exog_m[['CH_LAG1', 'VC_LAG1']], on='TAHUN_BULAN', how='left')
monthly_full_revisi.to_csv(REVISI_DIR / '04_monthly_with_exog_lag1.csv', index=False)

# 2. Pemfilteran unit analisis (190 unit armada valid)
NEW_EXCLUDED_15_OLD = [
    'UNIT_0027', 'UNIT_0301', 'UNIT_0187', 'UNIT_0006', 'UNIT_0339',
    'UNIT_0080', 'UNIT_0091', 'UNIT_0220', 'UNIT_0344', 'UNIT_0021',
    'UNIT_0081', 'UNIT_0142', 'UNIT_0024', 'UNIT_0197', 'UNIT_0185'
]
NEW_EXCLUDED_15 = [unit_map_new.get(u, u) for u in NEW_EXCLUDED_15_OLD]

df_exc_lama = pl.load_excluded_summary(Path('./checkpoints_v4/h3bulan'))
df_exc_lama['EQUIP NAME'] = df_exc_lama['EQUIP NAME'].map(lambda u: unit_map_new.get(u, u))

# Rangkuman unit yang diexclude operasional (25 unit resmi)
df_exc_revisi_total = df_exc_lama.copy()
df_exc_revisi_total = df_exc_revisi_total[~df_exc_revisi_total['ALASAN'].astype(str).str.contains('review MAPE|data baru|threshold|toleransi', case=False, na=False)].reset_index(drop=True)
df_exc_revisi_total['ALASAN'] = df_exc_revisi_total['ALASAN'].astype(str).str.replace('utilitas', 'operating hours', case=False)
df_exc_revisi_total.to_excel(REVISI_DIR / 'unit_excluded_revisi_summary.xlsx', index=False)
df_exc_revisi_total.to_csv(REVISI_DIR / '06_excluded_units_revisi.csv', index=False)

# Unit valid menjadi 190 unit
final_unit_lama = pl.load_final_config(Path('./checkpoints_v4/h3bulan'), level='unit')
final_unit_lama['EQUIP NAME'] = final_unit_lama['EQUIP NAME'].map(lambda u: unit_map_new.get(u, u))
final_unit_190 = final_unit_lama[~final_unit_lama['EQUIP NAME'].isin(NEW_EXCLUDED_15)].copy().reset_index(drop=True)

final_krit = pl.load_final_config(Path('./checkpoints_v4/h3bulan'), level='kriteria')
if 'KODE_KRITERIA' not in final_krit.columns:
    final_krit['KODE_KRITERIA'] = final_krit['KRITERIA_ID'].map(kriteria_code_map)
    cols = ['KODE_KRITERIA'] + [c for c in final_krit.columns if c != 'KODE_KRITERIA']
    final_krit = final_krit[cols]

with pd.ExcelWriter(REVISI_DIR / 'forcedgb_vs_v2b_comparison_190unit.xlsx', engine='openpyxl') as writer:
    final_unit_190.to_excel(writer, sheet_name='Per_Unit', index=False)
    final_krit.to_excel(writer, sheet_name='Per_Kriteria', index=False)

# Unit valid: 190 unit
# Populasi 190 unit armada valid
print(f'[Analisis 190 Unit] Unit valid tersisa: {len(final_unit_190)} unit')
print(f'[Analisis 190 Unit] Rata-rata MAPE Model Final 190 unit: {final_unit_190["V4_GB_MAPE"].mean():.2f}%')
print(f'[Analisis 190 Unit] Median MAPE Model Final 190 unit: {final_unit_190["V4_GB_MAPE"].median():.2f}%')


In [ ]:
# ======== KOMPARASI: ARIMA vs ARIMAX vs GRADIENT BOOSTING (CHECKPOINT) ========
EXCEL_COMP_REVISI = REVISI_DIR / 'perbandingan_eksogen_dan_model.xlsx'

if EXCEL_COMP_REVISI.exists():
    print(f'[Analisis] Memuat hasil komparasi dari: {EXCEL_COMP_REVISI}')
    tabel_ringkas_190 = pd.read_excel(EXCEL_COMP_REVISI, sheet_name='Ringkasan')
else:
    print('[Analisis] Menghitung perbandingan 6 skenario model...')
    import sys
    sys.path.insert(0, '.')
    from kode_eksperimen_6_skenario import run_experiment_6_skenario
    tabel_ringkas_190 = run_experiment_6_skenario(output_dir=REVISI_DIR)

print('\n=== HASIL PERBANDINGAN MODEL (190 UNIT ANALISIS) ===')
print(tabel_ringkas_190.round(2).to_string(index=False))


In [ ]:
# ======== PROYEKSI MASA DEPAN (FUTURE FORECAST) 190 UNIT ========
FUTURE_REVISI = REVISI_DIR / 'future_forecast'
FUTURE_REVISI.mkdir(parents=True, exist_ok=True)
EXCEL_FC_REVISI = FUTURE_REVISI / 'future_forecast_final_190unit.xlsx'

if EXCEL_FC_REVISI.exists():
    print(f'[Revisi] Memuat forecast masa depan 190 unit dari: {EXCEL_FC_REVISI}')
    df_fc_190 = pd.read_excel(EXCEL_FC_REVISI, sheet_name='Forecast_Per_Unit')
else:
    exog_fut = pl.estimate_future_exog(monthly_full_revisi, horizon=12)
    frames = [pl.retrain_and_forecast_unit(monthly_full_revisi, r, exog_fut, 12) for _, r in final_unit_190.iterrows()]
    df_fc_190 = pd.concat([f for f in frames if f is not None], ignore_index=True)
    df_fc_190.to_csv(FUTURE_REVISI / 'future_forecast_per_unit_190unit.csv', index=False)
    with pd.ExcelWriter(EXCEL_FC_REVISI, engine='openpyxl') as writer:
        df_fc_190.to_excel(writer, sheet_name='Forecast_Per_Unit', index=False)

print(f'[Revisi] Proyeksi jam kerja 12 bulan ke depan untuk {df_fc_190["EQUIP NAME"].nunique()} unit selesai.')
print('Contoh 6 baris pertama hasil proyeksi:')
print(df_fc_190.head(6)[['EQUIP NAME', 'TAHUN_BULAN', 'HORIZON_BULAN_KE', 'FORECAST_OPERATING_HOURS', 'VARIAN']])
